In [1]:
!pip install selectolax
!pip install playwright

In [2]:
!playwright install chromium

In [66]:
import time
import random
import asyncio
from urllib.parse import urlparse
from playwright.async_api import async_playwright

class HeadlessBrowserManager:
    def __init__(self):
        self.playwright: Playwright | None = None
        self.browser: Browser | None = None

    async def initialize(self):
        """launches a localize headless chromium browser instance"""
        if not self.browser:
            self.playwright = await async_playwright().start()
            self.browser = await self.playwright.chromium.launch(
                headless=False,
                args=["--disable-gpu", "--no-sandbox", "--disable-dev-shm-usage"],
            )

    async def scrape_dynamic_page(self, url: str, wait_for_selector: str = "p", use_page = False):
        """Loads a page in a full browser environment, waits for elements to render, and pulls HTML."""

        await self.initialize()

        if self.browser:
            # Open an isolated browser tab context with desktop screen dimension mimicry
            context = await self.browser.new_context(
                user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
                viewport={"width": 1280, "height": 800},
            )

            page = await context.new_page()
            
            try:
                # Navigate to the target web novel page URL
                await page.goto(url, wait_until="domcontentloaded", timeout=30000)

                # Explicitly wait for javascript text frameworks to finish rendering elements into DOM
                await page.wait_for_selector(wait_for_selector, timeout=10000)

                # Extract the raw rendered HTML string code format
                html_content = await page.content()
                return (html_content, page if use_page else None)

            finally:
                if not use_page:
                    await page.close()
                    await context.close()

                
    async def fetch_with_browser_interaction(
        self, url: str, *, wait_for_selector="body", interaction_callback=None
    ) -> str | None:
        """
        Launches Playwright, navigates to the page, executes an optional
        custom callback (like clicking tabs), and harvests the final DOM.
        """
        await self.initialize()

        if self.browser:
            context = await self.browser.new_context(
                user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
                viewport={"width": 1280, "height": 800},
            )
            page = await context.new_page()

            try:
                await page.goto(url, wait_until="domcontentloaded", timeout=60000)
                # Explicitly wait for javascript text frameworks to finish rendering elements into DOM
                await page.wait_for_selector(wait_for_selector, timeout=10000)

                # If the specific scraper passed a tab-toggling instruction, execute it now
                if interaction_callback:
                    await interaction_callback(page)
                    # Give JavaScript a brief window to animate or hydrate the new layout
                    await page.wait_for_timeout(800)

                return await page.content()
            finally:
                await page.close()
                await context.close()

    async def shutdown(self):
        """Closes browser background processes cleany on app exit."""
        if self.browser:
            await self.browser.close()
        if self.playwright:
            await self.playwright.stop()


test_web_novel = "https://wtr-lab.com/en/novel/53992/lord-god-tier-attribute-recruits-fallen-angels-of-original-sin"

async def fetch_page_content():
    browser = HeadlessBrowserManager()
    content = await browser.scrape_dynamic_page(test_web_novel, wait_for_selector=".chapter-details")
    return content

# async def fetch_chapter_tab_content():
#     # Define a custom page-interaction script for this specific site structure
#     async def toggle_chapters_tab(page):
#         # Target the tab button via its text, ID, or CSS class parameters
#         # tab_selector = "button:has-text('Table of Contents')"

#         # tab = page.locator(tab_selector)
#         # tab_count = await tab.count()

#         # print(f"[+] Is Table of Content found {'[Yes]' if tab_count > 0 else '[No]'}")

#         tab = page.get_by_role('tab', name='Table of Contents')
#         print(f"ToC {tab}")
#         await asyncio.sleep(1)
#         await tab.click()
        
#         # await asyncio.sleep(10)
#         await page.wait_for_selector(".chapter-list", timeout=10000)

#         triggers = page.locator("button[data-slot='accordion-trigger']")
#         count = await triggers.count()
#         print(f"[+] Found {count} accordion panels to expand.")

#         if count > 0:
#             for trigger in await triggers.all():
#                 await trigger.click()
#             await page.wait_for_selector("div[data-slot='accordion-content'] a", timeout=10000)

#             links: list[str] = []
#             anchor_elements = page.locator("div[data-slot='accordion-content'] a")
#             for a in await anchor_elements.all():
#                 link = await a.get_attribute("href")
#                 if link:
#                     links.append(link)
#             print(f"Links: {links}")

                
        
        
        
            
#         # Verify if the tab target element actually exists on the active webview surface
#         # if tab_count > 0:
#         #     print("🖱️ Intercepted dynamic tab menu. Simulating click event...")

#         #     # tab_selector_btn = page.locator(tab_selector)
#         #     # btn_count = await tab_selector_btn.count()

#         #     # print(f"Is Tab selector found {btn_count > 0}")
            
#         #     # await page.click(tab_selector)
#         #     await tab.click(force=True)

#         #     # print(await page.content())
#         #     # Wait for the specific container to populate with anchor elements
#         #     await page.wait_for_selector(".chapter-list", timeout=10000)

#         #     print("[+] Page loaded. Finding accordion trigger buttons...")

#         #     # Direct target using the specific data-slot attribute from your HTML
#         #     triggers = page.locator(".accordion-trigger")
#         #     count = await triggers.count()
#         #     print(f"[+] Found {count} accordion panels to expand.")
            
#     browser = HeadlessBrowserManager()
#     content = await browser.fetch_with_browser_interaction(test_web_novel, wait_for_selector=".chapter-details",interaction_callback=toggle_chapters_tab)
#     return content

async def fetch_main_page_html_contents(url: str):
    instance = HeadlessBrowserManager()

    await instance.initialize()
    browser = instance.browser

    print(browser)

    initial_html_content: str = ''
    chapters: list[str] = []

    if browser:
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
            viewport={"width": 1280, "height": 800},
        )

        page = await context.new_page()

        try:
            # Navigate to the target web novel page URL
            await page.goto(url, wait_until="domcontentloaded", timeout=60000)

            # Explicitly wait for javascript text frameworks to finish rendering elements into DOM
            await page.wait_for_selector(".chapter-details", timeout=60000)

            initial_html_content = await page.content()

            tab = page.get_by_role("tab", name="Table of Contents")
            print(f"ToC {tab}")
            await asyncio.sleep(1)
            await tab.click()
            await page.wait_for_selector(".chapter-list", timeout=10000)


            accordion_items = page.locator("div[data-slot='accordion-item']")
            count = await accordion_items.count()
            print(f"[+] Found {count} accordion panels to expand.")
            
            accordions = await accordion_items.all()
            
            if count > 0:
                for idx, item in enumerate(accordions):
                    # 1. Fixed the quote nesting issue
                    data_index = await item.get_attribute("data-index")
                    print(f"[+] Processing accordion index: {data_index} (Loop index: {idx})")
                    
                    # Click the trigger to open the accordion
                    trigger = item.locator("button[data-slot='accordion-trigger']")
                    await trigger.click()
            
                    # Target the specific anchor tags inside this item
                    anchor_elements = item.locator("div[data-slot='accordion-content'] a")
                    
                    # FIX: Explicitly wait for the first link in THIS accordion to become attached/visible.
                    # This pauses execution until the network/animation finishes rendering the links.
                    try:
                        await anchor_elements.first.wait_for(state="attached", timeout=5000)
                    except Exception:
                        print(f"[!] Warning: No links appeared in accordion {idx} after 5 seconds.")
                    
                    for a in await anchor_elements.all():
                        link = await a.get_attribute("href")
                        if link:
                            chapters.append(link)
                            
                    await asyncio.sleep(1)
            
            return initial_html_content, chapters

        finally:
            await page.close()
            await context.close()
            await instance.shutdown()

# result = await fetch_page_content()
# table_of_contents_result = await fetch_chapter_tab_content()

# table_of_contents_result

results = await fetch_main_page_html_contents(test_web_novel)

info_html = results[0]
raw_chapter_links = results[1]

print(f"\n\n[+] Found {len(raw_chapter_links)} Chapters")



<Browser type=<BrowserType name=chromium executable_path=/Users/dayodanielfamuyiwa/Library/Caches/ms-playwright/chromium-1228/chrome-mac-arm64/Google Chrome for Testing.app/Contents/MacOS/Google Chrome for Testing> version=149.0.7827.55>
ToC <Locator frame=<Frame name= url='https://wtr-lab.com/en/novel/53992/lord-god-tier-attribute-recruits-fallen-angels-of-original-sin'> selector='internal:role=tab[name="Table of Contents"i]'>
[+] Found 3 accordion panels to expand.
[<Locator frame=<Frame name= url='https://wtr-lab.com/en/novel/53992/lord-god-tier-attribute-recruits-fallen-angels-of-original-sin?tab=toc'> selector="div[data-slot='accordion-item'] >> nth=0">, <Locator frame=<Frame name= url='https://wtr-lab.com/en/novel/53992/lord-god-tier-attribute-recruits-fallen-angels-of-original-sin?tab=toc'> selector="div[data-slot='accordion-item'] >> nth=1">, <Locator frame=<Frame name= url='https://wtr-lab.com/en/novel/53992/lord-god-tier-attribute-recruits-fallen-angels-of-original-sin?tab=to

In [32]:
from urllib.parse import urlsplit
from selectolax.lexbor import LexborHTMLParser

html = """
<html lang="en" class="light bprogress-busy" data-color-scheme="light" style="color-scheme: light;"><head><meta charset="utf-8" data-next-head=""><meta name="viewport" content="width=device-width" data-next-head=""><title data-next-head="">Read Mortal Bones RAW English Translation - WTR-LAB</title><link rel="preconnect" href="https://fonts.googleapis.com"><link rel="preconnect" href="https://fonts.gstatic.com" crossorigin="anonymous"><link rel="icon" type="image/png" href="/assets/favicon/favicon-96x96.png" sizes="96x96"><link rel="icon" type="image/svg+xml" href="/assets/favicon/favicon-light.svg"><link rel="shortcut icon" href="/assets/favicon/favicon.ico"><link rel="apple-touch-icon" sizes="180x180" href="/assets/favicon/apple-touch-icon.png"><meta name="apple-mobile-web-app-title" content="wtr-lab"><link rel="manifest" href="/assets/favicon/site.webmanifest"><link rel="preload" href="/_next/static/chunks/ad638c3c01f639f8.css" as="style"><meta name="sentry-trace" content="972186b2db5b9e27acfb49c1022aaeeb-918a3e415ffce71e"><meta name="baggage" content="sentry-environment=production,sentry-public_key=c06b7ce45264c8ecc508ce6bf913dd34,sentry-trace_id=972186b2db5b9e27acfb49c1022aaeeb,sentry-org_id=4508930697068544"><link href="https://fonts.googleapis.com/css2?family=JetBrains+Mono:ital,wght@0,100..800;1,100..800&amp;family=Nunito+Sans:opsz,wght@6..12,200..1000&amp;display=swap" rel="stylesheet"><link rel="stylesheet" href="/_next/static/chunks/ad638c3c01f639f8.css" data-n-g=""><noscript data-n-css=""></noscript><script defer="" nomodule="" src="/_next/static/chunks/a6dad97d9634a72d.js"></script><script src="/_next/static/chunks/ba6e81906175a8d5.js" defer=""></script><script src="/_next/static/chunks/d604e29fa84975be.js" defer=""></script><script src="/_next/static/chunks/5b20248b7c652da1.js" defer=""></script><script src="/_next/static/chunks/943107ce85400c20.js" defer=""></script><script src="/_next/static/chunks/fd3557e6ebbaa8c6.js" defer=""></script><script src="/_next/static/chunks/b1d44c827f38772c.js" defer=""></script><script src="/_next/static/chunks/f125e46c6e2cd5c1.js" defer=""></script><script src="/_next/static/chunks/fade602ba06fdac4.js" defer=""></script><script src="/_next/static/chunks/c4c1e4545cd8584c.js" defer=""></script><script src="/_next/static/chunks/1e19b7795cec3f66.js" defer=""></script><script src="/_next/static/chunks/9b79bbaba4591794.js" defer=""></script><script src="/_next/static/chunks/515505cf083caca0.js" defer=""></script><script src="/_next/static/chunks/8ce9f54e00d2f555.js" defer=""></script><script src="/_next/static/chunks/041f022d83a647b0.js" defer=""></script><script src="/_next/static/chunks/64a200a3fa70750f.js" defer=""></script><script src="/_next/static/chunks/bc4d372e3eec8f27.js" defer=""></script><script src="/_next/static/chunks/b994afa3e361e021.js" defer=""></script><script src="/_next/static/chunks/be6cb443a41c4a8e.js" defer=""></script><script src="/_next/static/chunks/6a7a63a31aa9827f.js" defer=""></script><script src="/_next/static/chunks/27138e60f7fd2372.js" defer=""></script><script src="/_next/static/chunks/2cbc5d41b5f3b7dc.js" defer=""></script><script src="/_next/static/chunks/388a274b2ef0d4e1.js" defer=""></script><script src="/_next/static/chunks/0dd18f0999b419dc.js" defer=""></script><script src="/_next/static/chunks/turbopack-f2503c656c8fe1cb.js" defer=""></script><script src="/_next/static/chunks/ef7aea2f0d7630fd.js" defer=""></script><script src="/_next/static/chunks/3ac4f810e1ae72c7.js" defer=""></script><script src="/_next/static/chunks/d3933f32018b15f6.js" defer=""></script><script src="/_next/static/chunks/6722971c681c563d.js" defer=""></script><script src="/_next/static/chunks/e41dc375691a79be.js" defer=""></script><script src="/_next/static/chunks/7c6102ea26fcb2dc.js" defer=""></script><script src="/_next/static/chunks/88d7c1ef3f7a06e6.js" defer=""></script><script src="/_next/static/chunks/2216678994bf8838.js" defer=""></script><script src="/_next/static/chunks/e43a1e81e7ad6b8e.js" defer=""></script><script src="/_next/static/chunks/4ed7a3b5b0a958c6.js" defer=""></script><script src="/_next/static/chunks/f1284758969025b7.js" defer=""></script><script src="/_next/static/chunks/5ebd33993ef047a2.js" defer=""></script><script src="/_next/static/chunks/a721a51bdead4c46.js" defer=""></script><script src="/_next/static/chunks/ea8392bbb66918bd.js" defer=""></script><script src="/_next/static/chunks/4040f79c2183593c.js" defer=""></script><script src="/_next/static/chunks/5be2de8511ef511d.js" defer=""></script><script src="/_next/static/chunks/turbopack-350c814dde5750e4.js" defer=""></script><script src="/_next/static/0rJUaTB6TLmnU7nTm-twr/_ssgManifest.js" defer=""></script><script src="/_next/static/0rJUaTB6TLmnU7nTm-twr/_buildManifest.js" defer=""></script><style id="__jsx-2deb0be93c618d80">body{-webkit-font-smoothing:antialiased;-moz-osx-font-smoothing:grayscale}</style><style type="text/css">[data-sonner-toaster][dir=ltr],html[dir=ltr]{--toast-icon-margin-start:-3px;--toast-icon-margin-end:4px;--toast-svg-margin-start:-1px;--toast-svg-margin-end:0px;--toast-button-margin-start:auto;--toast-button-margin-end:0;--toast-close-button-start:0;--toast-close-button-end:unset;--toast-close-button-transform:translate(-35%, -35%)}[data-sonner-toaster][dir=rtl],html[dir=rtl]{--toast-icon-margin-start:4px;--toast-icon-margin-end:-3px;--toast-svg-margin-start:0px;--toast-svg-margin-end:-1px;--toast-button-margin-start:0;--toast-button-margin-end:auto;--toast-close-button-start:unset;--toast-close-button-end:0;--toast-close-button-transform:translate(35%, -35%)}[data-sonner-toaster]{position:fixed;width:var(--width);font-family:ui-sans-serif,system-ui,-apple-system,BlinkMacSystemFont,Segoe UI,Roboto,Helvetica Neue,Arial,Noto Sans,sans-serif,Apple Color Emoji,Segoe UI Emoji,Segoe UI Symbol,Noto Color Emoji;--gray1:hsl(0, 0%, 99%);--gray2:hsl(0, 0%, 97.3%);--gray3:hsl(0, 0%, 95.1%);--gray4:hsl(0, 0%, 93%);--gray5:hsl(0, 0%, 90.9%);--gray6:hsl(0, 0%, 88.7%);--gray7:hsl(0, 0%, 85.8%);--gray8:hsl(0, 0%, 78%);--gray9:hsl(0, 0%, 56.1%);--gray10:hsl(0, 0%, 52.3%);--gray11:hsl(0, 0%, 43.5%);--gray12:hsl(0, 0%, 9%);--border-radius:8px;box-sizing:border-box;padding:0;margin:0;list-style:none;outline:0;z-index:999999999;transition:transform .4s ease}@media (hover:none) and (pointer:coarse){[data-sonner-toaster][data-lifted=true]{transform:none}}[data-sonner-toaster][data-x-position=right]{right:var(--offset-right)}[data-sonner-toaster][data-x-position=left]{left:var(--offset-left)}[data-sonner-toaster][data-x-position=center]{left:50%;transform:translateX(-50%)}[data-sonner-toaster][data-y-position=top]{top:var(--offset-top)}[data-sonner-toaster][data-y-position=bottom]{bottom:var(--offset-bottom)}[data-sonner-toast]{--y:translateY(100%);--lift-amount:calc(var(--lift) * var(--gap));z-index:var(--z-index);position:absolute;opacity:0;transform:var(--y);touch-action:none;transition:transform .4s,opacity .4s,height .4s,box-shadow .2s;box-sizing:border-box;outline:0;overflow-wrap:anywhere}[data-sonner-toast][data-styled=true]{padding:16px;background:var(--normal-bg);border:1px solid var(--normal-border);color:var(--normal-text);border-radius:var(--border-radius);box-shadow:0 4px 12px rgba(0,0,0,.1);width:var(--width);font-size:13px;display:flex;align-items:center;gap:6px}[data-sonner-toast]:focus-visible{box-shadow:0 4px 12px rgba(0,0,0,.1),0 0 0 2px rgba(0,0,0,.2)}[data-sonner-toast][data-y-position=top]{top:0;--y:translateY(-100%);--lift:1;--lift-amount:calc(1 * var(--gap))}[data-sonner-toast][data-y-position=bottom]{bottom:0;--y:translateY(100%);--lift:-1;--lift-amount:calc(var(--lift) * var(--gap))}[data-sonner-toast][data-styled=true] [data-description]{font-weight:400;line-height:1.4;color:#3f3f3f}[data-rich-colors=true][data-sonner-toast][data-styled=true] [data-description]{color:inherit}[data-sonner-toaster][data-sonner-theme=dark] [data-description]{color:#e8e8e8}[data-sonner-toast][data-styled=true] [data-title]{font-weight:500;line-height:1.5;color:inherit}[data-sonner-toast][data-styled=true] [data-icon]{display:flex;height:16px;width:16px;position:relative;justify-content:flex-start;align-items:center;flex-shrink:0;margin-left:var(--toast-icon-margin-start);margin-right:var(--toast-icon-margin-end)}[data-sonner-toast][data-promise=true] [data-icon]>svg{opacity:0;transform:scale(.8);transform-origin:center;animation:sonner-fade-in .3s ease forwards}[data-sonner-toast][data-styled=true] [data-icon]>*{flex-shrink:0}[data-sonner-toast][data-styled=true] [data-icon] svg{margin-left:var(--toast-svg-margin-start);margin-right:var(--toast-svg-margin-end)}[data-sonner-toast][data-styled=true] [data-content]{display:flex;flex-direction:column;gap:2px}[data-sonner-toast][data-styled=true] [data-button]{border-radius:4px;padding-left:8px;padding-right:8px;height:24px;font-size:12px;color:var(--normal-bg);background:var(--normal-text);margin-left:var(--toast-button-margin-start);margin-right:var(--toast-button-margin-end);border:none;font-weight:500;cursor:pointer;outline:0;display:flex;align-items:center;flex-shrink:0;transition:opacity .4s,box-shadow .2s}[data-sonner-toast][data-styled=true] [data-button]:focus-visible{box-shadow:0 0 0 2px rgba(0,0,0,.4)}[data-sonner-toast][data-styled=true] [data-button]:first-of-type{margin-left:var(--toast-button-margin-start);margin-right:var(--toast-button-margin-end)}[data-sonner-toast][data-styled=true] [data-cancel]{color:var(--normal-text);background:rgba(0,0,0,.08)}[data-sonner-toaster][data-sonner-theme=dark] [data-sonner-toast][data-styled=true] [data-cancel]{background:rgba(255,255,255,.3)}[data-sonner-toast][data-styled=true] [data-close-button]{position:absolute;left:var(--toast-close-button-start);right:var(--toast-close-button-end);top:0;height:20px;width:20px;display:flex;justify-content:center;align-items:center;padding:0;color:var(--gray12);background:var(--normal-bg);border:1px solid var(--gray4);transform:var(--toast-close-button-transform);border-radius:50%;cursor:pointer;z-index:1;transition:opacity .1s,background .2s,border-color .2s}[data-sonner-toast][data-styled=true] [data-close-button]:focus-visible{box-shadow:0 4px 12px rgba(0,0,0,.1),0 0 0 2px rgba(0,0,0,.2)}[data-sonner-toast][data-styled=true] [data-disabled=true]{cursor:not-allowed}[data-sonner-toast][data-styled=true]:hover [data-close-button]:hover{background:var(--gray2);border-color:var(--gray5)}[data-sonner-toast][data-swiping=true]::before{content:'';position:absolute;left:-100%;right:-100%;height:100%;z-index:-1}[data-sonner-toast][data-y-position=top][data-swiping=true]::before{bottom:50%;transform:scaleY(3) translateY(50%)}[data-sonner-toast][data-y-position=bottom][data-swiping=true]::before{top:50%;transform:scaleY(3) translateY(-50%)}[data-sonner-toast][data-swiping=false][data-removed=true]::before{content:'';position:absolute;inset:0;transform:scaleY(2)}[data-sonner-toast][data-expanded=true]::after{content:'';position:absolute;left:0;height:calc(var(--gap) + 1px);bottom:100%;width:100%}[data-sonner-toast][data-mounted=true]{--y:translateY(0);opacity:1}[data-sonner-toast][data-expanded=false][data-front=false]{--scale:var(--toasts-before) * 0.05 + 1;--y:translateY(calc(var(--lift-amount) * var(--toasts-before))) scale(calc(-1 * var(--scale)));height:var(--front-toast-height)}[data-sonner-toast]>*{transition:opacity .4s}[data-sonner-toast][data-x-position=right]{right:0}[data-sonner-toast][data-x-position=left]{left:0}[data-sonner-toast][data-expanded=false][data-front=false][data-styled=true]>*{opacity:0}[data-sonner-toast][data-visible=false]{opacity:0;pointer-events:none}[data-sonner-toast][data-mounted=true][data-expanded=true]{--y:translateY(calc(var(--lift) * var(--offset)));height:var(--initial-height)}[data-sonner-toast][data-removed=true][data-front=true][data-swipe-out=false]{--y:translateY(calc(var(--lift) * -100%));opacity:0}[data-sonner-toast][data-removed=true][data-front=false][data-swipe-out=false][data-expanded=true]{--y:translateY(calc(var(--lift) * var(--offset) + var(--lift) * -100%));opacity:0}[data-sonner-toast][data-removed=true][data-front=false][data-swipe-out=false][data-expanded=false]{--y:translateY(40%);opacity:0;transition:transform .5s,opacity .2s}[data-sonner-toast][data-removed=true][data-front=false]::before{height:calc(var(--initial-height) + 20%)}[data-sonner-toast][data-swiping=true]{transform:var(--y) translateY(var(--swipe-amount-y,0)) translateX(var(--swipe-amount-x,0));transition:none}[data-sonner-toast][data-swiped=true]{user-select:none}[data-sonner-toast][data-swipe-out=true][data-y-position=bottom],[data-sonner-toast][data-swipe-out=true][data-y-position=top]{animation-duration:.2s;animation-timing-function:ease-out;animation-fill-mode:forwards}[data-sonner-toast][data-swipe-out=true][data-swipe-direction=left]{animation-name:swipe-out-left}[data-sonner-toast][data-swipe-out=true][data-swipe-direction=right]{animation-name:swipe-out-right}[data-sonner-toast][data-swipe-out=true][data-swipe-direction=up]{animation-name:swipe-out-up}[data-sonner-toast][data-swipe-out=true][data-swipe-direction=down]{animation-name:swipe-out-down}@keyframes swipe-out-left{from{transform:var(--y) translateX(var(--swipe-amount-x));opacity:1}to{transform:var(--y) translateX(calc(var(--swipe-amount-x) - 100%));opacity:0}}@keyframes swipe-out-right{from{transform:var(--y) translateX(var(--swipe-amount-x));opacity:1}to{transform:var(--y) translateX(calc(var(--swipe-amount-x) + 100%));opacity:0}}@keyframes swipe-out-up{from{transform:var(--y) translateY(var(--swipe-amount-y));opacity:1}to{transform:var(--y) translateY(calc(var(--swipe-amount-y) - 100%));opacity:0}}@keyframes swipe-out-down{from{transform:var(--y) translateY(var(--swipe-amount-y));opacity:1}to{transform:var(--y) translateY(calc(var(--swipe-amount-y) + 100%));opacity:0}}@media (max-width:600px){[data-sonner-toaster]{position:fixed;right:var(--mobile-offset-right);left:var(--mobile-offset-left);width:100%}[data-sonner-toaster][dir=rtl]{left:calc(var(--mobile-offset-left) * -1)}[data-sonner-toaster] [data-sonner-toast]{left:0;right:0;width:calc(100% - var(--mobile-offset-left) * 2)}[data-sonner-toaster][data-x-position=left]{left:var(--mobile-offset-left)}[data-sonner-toaster][data-y-position=bottom]{bottom:var(--mobile-offset-bottom)}[data-sonner-toaster][data-y-position=top]{top:var(--mobile-offset-top)}[data-sonner-toaster][data-x-position=center]{left:var(--mobile-offset-left);right:var(--mobile-offset-right);transform:none}}[data-sonner-toaster][data-sonner-theme=light]{--normal-bg:#fff;--normal-border:var(--gray4);--normal-text:var(--gray12);--success-bg:hsl(143, 85%, 96%);--success-border:hsl(145, 92%, 87%);--success-text:hsl(140, 100%, 27%);--info-bg:hsl(208, 100%, 97%);--info-border:hsl(221, 91%, 93%);--info-text:hsl(210, 92%, 45%);--warning-bg:hsl(49, 100%, 97%);--warning-border:hsl(49, 91%, 84%);--warning-text:hsl(31, 92%, 45%);--error-bg:hsl(359, 100%, 97%);--error-border:hsl(359, 100%, 94%);--error-text:hsl(360, 100%, 45%)}[data-sonner-toaster][data-sonner-theme=light] [data-sonner-toast][data-invert=true]{--normal-bg:#000;--normal-border:hsl(0, 0%, 20%);--normal-text:var(--gray1)}[data-sonner-toaster][data-sonner-theme=dark] [data-sonner-toast][data-invert=true]{--normal-bg:#fff;--normal-border:var(--gray3);--normal-text:var(--gray12)}[data-sonner-toaster][data-sonner-theme=dark]{--normal-bg:#000;--normal-bg-hover:hsl(0, 0%, 12%);--normal-border:hsl(0, 0%, 20%);--normal-border-hover:hsl(0, 0%, 25%);--normal-text:var(--gray1);--success-bg:hsl(150, 100%, 6%);--success-border:hsl(147, 100%, 12%);--success-text:hsl(150, 86%, 65%);--info-bg:hsl(215, 100%, 6%);--info-border:hsl(223, 43%, 17%);--info-text:hsl(216, 87%, 65%);--warning-bg:hsl(64, 100%, 6%);--warning-border:hsl(60, 100%, 9%);--warning-text:hsl(46, 87%, 65%);--error-bg:hsl(358, 76%, 10%);--error-border:hsl(357, 89%, 16%);--error-text:hsl(358, 100%, 81%)}[data-sonner-toaster][data-sonner-theme=dark] [data-sonner-toast] [data-close-button]{background:var(--normal-bg);border-color:var(--normal-border);color:var(--normal-text)}[data-sonner-toaster][data-sonner-theme=dark] [data-sonner-toast] [data-close-button]:hover{background:var(--normal-bg-hover);border-color:var(--normal-border-hover)}[data-rich-colors=true][data-sonner-toast][data-type=success]{background:var(--success-bg);border-color:var(--success-border);color:var(--success-text)}[data-rich-colors=true][data-sonner-toast][data-type=success] [data-close-button]{background:var(--success-bg);border-color:var(--success-border);color:var(--success-text)}[data-rich-colors=true][data-sonner-toast][data-type=info]{background:var(--info-bg);border-color:var(--info-border);color:var(--info-text)}[data-rich-colors=true][data-sonner-toast][data-type=info] [data-close-button]{background:var(--info-bg);border-color:var(--info-border);color:var(--info-text)}[data-rich-colors=true][data-sonner-toast][data-type=warning]{background:var(--warning-bg);border-color:var(--warning-border);color:var(--warning-text)}[data-rich-colors=true][data-sonner-toast][data-type=warning] [data-close-button]{background:var(--warning-bg);border-color:var(--warning-border);color:var(--warning-text)}[data-rich-colors=true][data-sonner-toast][data-type=error]{background:var(--error-bg);border-color:var(--error-border);color:var(--error-text)}[data-rich-colors=true][data-sonner-toast][data-type=error] [data-close-button]{background:var(--error-bg);border-color:var(--error-border);color:var(--error-text)}.sonner-loading-wrapper{--size:16px;height:var(--size);width:var(--size);position:absolute;inset:0;z-index:10}.sonner-loading-wrapper[data-visible=false]{transform-origin:center;animation:sonner-fade-out .2s ease forwards}.sonner-spinner{position:relative;top:50%;left:50%;height:var(--size);width:var(--size)}.sonner-loading-bar{animation:sonner-spin 1.2s linear infinite;background:var(--gray11);border-radius:6px;height:8%;left:-10%;position:absolute;top:-3.9%;width:24%}.sonner-loading-bar:first-child{animation-delay:-1.2s;transform:rotate(.0001deg) translate(146%)}.sonner-loading-bar:nth-child(2){animation-delay:-1.1s;transform:rotate(30deg) translate(146%)}.sonner-loading-bar:nth-child(3){animation-delay:-1s;transform:rotate(60deg) translate(146%)}.sonner-loading-bar:nth-child(4){animation-delay:-.9s;transform:rotate(90deg) translate(146%)}.sonner-loading-bar:nth-child(5){animation-delay:-.8s;transform:rotate(120deg) translate(146%)}.sonner-loading-bar:nth-child(6){animation-delay:-.7s;transform:rotate(150deg) translate(146%)}.sonner-loading-bar:nth-child(7){animation-delay:-.6s;transform:rotate(180deg) translate(146%)}.sonner-loading-bar:nth-child(8){animation-delay:-.5s;transform:rotate(210deg) translate(146%)}.sonner-loading-bar:nth-child(9){animation-delay:-.4s;transform:rotate(240deg) translate(146%)}.sonner-loading-bar:nth-child(10){animation-delay:-.3s;transform:rotate(270deg) translate(146%)}.sonner-loading-bar:nth-child(11){animation-delay:-.2s;transform:rotate(300deg) translate(146%)}.sonner-loading-bar:nth-child(12){animation-delay:-.1s;transform:rotate(330deg) translate(146%)}@keyframes sonner-fade-in{0%{opacity:0;transform:scale(.8)}100%{opacity:1;transform:scale(1)}}@keyframes sonner-fade-out{0%{opacity:1;transform:scale(1)}100%{opacity:0;transform:scale(.8)}}@keyframes sonner-spin{0%{opacity:1}100%{opacity:.15}}@media (prefers-reduced-motion){.sonner-loading-bar,[data-sonner-toast],[data-sonner-toast]>*{transition:none!important;animation:none!important}}.sonner-loader{position:absolute;top:50%;left:50%;transform:translate(-50%,-50%);transform-origin:center;transition:opacity .2s,transform .2s}.sonner-loader[data-visible=false]{opacity:0;transform:scale(.8) translate(-50%,-50%)}</style><style id="theme-dynamic-styles">
    .chapter-body, .chapter-theme {
      background-color: white !important;
      color: inherit !important;
    }
    .chapter-bg {
      background-color: #f2f3f4 !important;
    }
  </style><script src="//rk.prinkitonama.com/rXlGlP467POdguhu/elXqr" async="" id="ads-galaksion" type="text/javascript" data-cfasync="false"></script><link as="script" rel="prefetch" href="/_next/./static/chunks/abb6af767f9755d4.js"><script src="/_next/static/chunks/3d0fc44809f2e86f.js"></script><script src="/_next/static/chunks/458287d2ee812c6b.js"></script><script src="/_next/static/chunks/97d22cb7d80f03e7.js"></script><script src="/_next/static/chunks/fe457db9532d8220.js"></script><script src="/_next/static/chunks/b85d39dfc70366da.js"></script><script src="/_next/static/chunks/1ac53cd0548b2be9.js"></script><script src="/_next/static/chunks/653d544d9a508ae2.js"></script><script src="/_next/static/chunks/b48dd96a39559fa6.js"></script><script src="/_next/static/chunks/ae55af41973d7cf6.js"></script><script src="/_next/static/chunks/c976cfcc6d7fe009.js"></script><script src="/_next/static/chunks/ece8b1ba65b4d778.js"></script><script src="/_next/static/chunks/4dfd76f4be6821ce.js"></script><script src="/_next/static/chunks/3a1311c8b0016535.js"></script><script src="/_next/static/chunks/d272a2b91440bb91.js"></script><script src="/_next/static/chunks/1490cfb8a7633ff9.js"></script><script src="/_next/static/chunks/f47632dd32f72e8d.js"></script><script src="/_next/static/chunks/9af7f5c172156026.js"></script><script src="/_next/static/chunks/turbopack-872e5ae5a4f7b89f.js"></script><script src="/_next/static/chunks/026ae353426345f1.js"></script><script src="/_next/static/chunks/f342da1c4e498233.js"></script><script src="/_next/static/chunks/8454306f65cd122f.js"></script><meta name="description" content="Read Mortal Bones / 凡骨 raw light novel translation in English - WTR-LAB" data-next-head=""><meta property="og:title" content="Read Mortal Bones RAW English Translation - WTR-LAB" data-next-head=""><meta property="og:description" content="Read Mortal Bones / 凡骨 raw light novel translation in English - WTR-LAB" data-next-head=""><meta property="og:image" content="https://img.wtr-lab.com/cdn/series/P0DFglC6FSy273hoZ3e48A.jpg" data-next-head=""><meta property="og:type" content="book" data-next-head=""><meta property="og:url" content="https://wtr-lab.com/en/novel/7336/mortal-bones" data-next-head=""><meta property="og:site_name" content="WTR-LAB.com" data-next-head=""><meta property="og:locale" content="en_US" data-next-head=""><link rel="canonical" href="https://wtr-lab.com/en/novel/7336/mortal-bones" data-next-head=""><style data-tiptap-style="">.ProseMirror {
  position: relative;
}

.ProseMirror {
  word-wrap: break-word;
  white-space: pre-wrap;
  white-space: break-spaces;
  -webkit-font-variant-ligatures: none;
  font-variant-ligatures: none;
  font-feature-settings: "liga" 0; /* the above doesn't seem to work in Edge */
}

.ProseMirror [contenteditable="false"] {
  white-space: normal;
}

.ProseMirror [contenteditable="false"] [contenteditable="true"] {
  white-space: pre-wrap;
}

.ProseMirror pre {
  white-space: pre-wrap;
}

img.ProseMirror-separator {
  display: inline !important;
  border: none !important;
  margin: 0 !important;
  width: 0 !important;
  height: 0 !important;
}

.ProseMirror-gapcursor {
  display: none;
  pointer-events: none;
  position: absolute;
  margin: 0;
}

.ProseMirror-gapcursor:after {
  content: "";
  display: block;
  position: absolute;
  top: -2px;
  width: 20px;
  border-top: 1px solid black;
  animation: ProseMirror-cursor-blink 1.1s steps(2, start) infinite;
}

@keyframes ProseMirror-cursor-blink {
  to {
    visibility: hidden;
  }
}

.ProseMirror-hideselection *::selection {
  background: transparent;
}

.ProseMirror-hideselection *::-moz-selection {
  background: transparent;
}

.ProseMirror-hideselection * {
  caret-color: transparent;
}

.ProseMirror-focused .ProseMirror-gapcursor {
  display: block;
}</style><script src="/_next/static/chunks/be89a86b40e10122.js"></script><script src="/_next/static/chunks/200a2839e59a4303.js"></script><script src="/_next/static/chunks/f3351f2c2e404c30.js"></script><script src="/_next/static/chunks/83d1a90616558e76.js"></script><script src="/_next/static/chunks/4b5274ffaadbf130.js"></script><script src="/_next/static/chunks/3709839ef776fad5.js"></script><script src="/_next/static/chunks/7c475112082b35a5.js"></script><script src="/_next/static/chunks/turbopack-1b3c438b8e42f8ce.js"></script><script src="/_next/static/chunks/9625cabfd23c3d09.js"></script><script src="/_next/static/chunks/turbopack-d1e688b32ed01bae.js"></script><script src="/_next/static/chunks/98193a1b843eafdd.js"></script><script src="/_next/static/chunks/83be8c9eb3201dc1.js"></script><script src="/_next/static/chunks/b416568bd67efdb5.js"></script><script src="/_next/static/chunks/b7f7f8f88c286c51.js"></script><script src="/_next/static/chunks/1b1dec90101dead9.js"></script><script src="/_next/static/chunks/2294c6a98f386486.js"></script><script src="/_next/static/chunks/turbopack-459166fe863d3607.js"></script></head><body class="wtr-adblock-off hide-navbar" style=""><div id="svg-sprite-container" aria-hidden="true" style="display: none;"><svg xmlns="http://www.w3.org/2000/svg" xmlns:xlink="http://www.w3.org/1999/xlink"><symbol viewBox="0 0 24 24" id="account-box-outline" xmlns="http://www.w3.org/2000/svg"><path d="M19 19H5V5h14m0-2H5a2 2 0 0 0-2 2v14a2 2 0 0 0 2 2h14a2 2 0 0 0 2-2V5a2 2 0 0 0-2-2m-2.5 13.25c0-1.5-3-2.25-4.5-2.25s-4.5.75-4.5 2.25V17h9M12 12.25A2.25 2.25 0 0 0 14.25 10 2.25 2.25 0 0 0 12 7.75 2.25 2.25 0 0 0 9.75 10 2.25 2.25 0 0 0 12 12.25"></path></symbol><symbol viewBox="0 -960 960 960" id="add" xmlns="http://www.w3.org/2000/svg"><path d="M450-450H250q-12.75 0-21.37-8.63-8.63-8.63-8.63-21.38 0-12.76 8.63-21.37Q237.25-510 250-510h200v-200q0-12.75 8.63-21.37 8.63-8.63 21.38-8.63 12.76 0 21.37 8.63Q510-722.75 510-710v200h200q12.75 0 21.37 8.63 8.63 8.63 8.63 21.38 0 12.76-8.63 21.37Q722.75-450 710-450H510v200q0 12.75-8.63 21.37-8.63 8.63-21.38 8.63-12.76 0-21.37-8.63Q450-237.25 450-250z"></path></symbol><symbol viewBox="0 -960 960 960" id="add-circle-outline" xmlns="http://www.w3.org/2000/svg"><path d="M450-450v130q0 12.75 8.63 21.37 8.63 8.63 21.38 8.63 12.76 0 21.37-8.63Q510-307.25 510-320v-130h130q12.75 0 21.37-8.63 8.63-8.63 8.63-21.38 0-12.76-8.63-21.37Q652.75-510 640-510H510v-130q0-12.75-8.63-21.37-8.63-8.63-21.38-8.63-12.76 0-21.37 8.63Q450-652.75 450-640v130H320q-12.75 0-21.37 8.63-8.63 8.63-8.63 21.38 0 12.76 8.63 21.37Q307.25-450 320-450zm30.07 350q-78.84 0-148.21-29.92t-120.68-81.21-81.25-120.63Q100-401.1 100-479.93q0-78.84 29.92-148.21t81.21-120.68 120.63-81.25Q401.1-860 479.93-860q78.84 0 148.21 29.92t120.68 81.21 81.25 120.63Q860-558.9 860-480.07q0 78.84-29.92 148.21t-81.21 120.68-120.63 81.25Q558.9-100 480.07-100m-.07-60q134 0 227-93t93-227-93-227-227-93-227 93-93 227 93 227 227 93m0-320"></path></symbol><symbol fill-rule="evenodd" clip-rule="evenodd" image-rendering="optimizeQuality" shape-rendering="geometricPrecision" text-rendering="geometricPrecision" viewBox="0 0 512 407.96" id="ads" xmlns="http://www.w3.org/2000/svg"><path fill-rule="nonzero" d="M74.36 0h295.21c40.84 0 74.37 33.52 74.37 74.36v146.3c-9.49-5.1-19.01-10.13-28.35-14.93V74.36c0-25.32-20.69-46.02-46.02-46.02H74.36c-25.34 0-46.02 20.68-46.02 46.02v169.07c0 25.27 20.76 46.01 46.02 46.01h240.72l4.55 28.35H74.36C33.46 317.79 0 284.33 0 243.43V74.36C0 33.51 33.51 0 74.36 0M460.8 406.23c-5.6 3.19-12.89 1.91-16.8-3.38l-28.14-38.81-19.56 27.06c-1.61 2.22-3.37 4.22-5.2 5.88-11.66 10.7-26.11 7.98-29.12-9.25l-25.01-166.24c-1.59-7.56 6.08-13.54 13.1-10.47 41.84 16.73 107.81 52.79 151 76.3 20.73 11.36 8.43 28.53-7.57 33.59-9.71 3.71-21.78 6.88-31.9 10.07l28.07 39.08c3.84 5.52 2.52 13.21-2.7 17.36-7.55 5.36-18.61 14.72-26.17 18.81m-6.17-13.11L477 376.97c-7.25-9.92-31.76-39.89-35.15-48.82-1.19-3.75.94-7.79 4.69-8.96 13.6-4.19 27.8-7.94 41.53-11.83 3.16-1.01 5.95-2.36 8.11-4.94-1.09-1.1-1.74-1.62-3.14-2.38l-138.81-70.13 22.94 153.4c.08.44.91 3.8 1.1 4.03 3.36-2 5.02-3.25 7.41-6.55 4.87-7.26 19.14-31.77 24.72-35.97 3.19-2.19 7.6-1.46 9.88 1.68zM232.67 215.38V102.41h50.61c20.36 0 34.34 4.34 41.93 13.02 7.59 8.67 11.39 23.17 11.39 43.47s-3.8 34.79-11.39 43.46c-7.59 8.68-21.57 13.02-41.93 13.02zm51.15-84.04h-15v55.12h15c4.94 0 8.53-.58 10.75-1.72 2.23-1.14 3.35-3.77 3.35-7.86v-35.97c0-4.09-1.12-6.71-3.35-7.86-2.22-1.14-5.81-1.71-10.75-1.71m-138.35 84.04h-38.14l29.28-112.97h55.85l29.28 112.97H183.6l-4.15-17.9h-29.83zm18.07-78.26-7.41 31.63h16.63l-7.23-31.63z"></path></symbol><symbol viewBox="0 0 24 24" id="ai" xmlns="http://www.w3.org/2000/svg"><path fill="#fff" d="M4.83 4.621h14.4v14.668H4.83z"></path><path d="M5 3c-1.102 0-2 .898-2 2v14c0 1.102.898 2 2 2h14c1.102 0 2-.898 2-2V5c0-1.102-.898-2-2-2zm4.688 4.906h1.625l2.687 8.5h-1.906l-.5-1.718H9.313l-.5 1.718H7zM14.7 7.9h1.7l.006 8.506h-1.718zm-4.2 2.288-.812 3h1.625z"></path></symbol><symbol viewBox="0 -960 960 960" id="alert-outline" xmlns="http://www.w3.org/2000/svg"><path d="M137.02-140q-10.17 0-18.27-4.97t-12.59-13.11q-4.68-8.08-5.15-17.5t5.08-18.66l342.43-591.52q5.56-9.24 13.9-13.66 8.35-4.42 17.58-4.42t17.58 4.42q8.34 4.42 13.9 13.66l342.43 591.52q5.55 9.24 5.05 18.58-.5 9.35-5.12 17.58-4.61 8.23-12.65 13.16-8.04 4.92-18.21 4.92zM178-200h604L480-720zm324.92-57.08q9.39-9.38 9.39-22.92t-9.39-22.92q-9.38-9.39-22.92-9.39t-22.92 9.39q-9.39 9.38-9.39 22.92t9.39 22.92q9.38 9.39 22.92 9.39t22.92-9.39m-1.54-103.85q8.62-8.63 8.62-21.38v-140q0-12.75-8.63-21.37-8.63-8.63-21.38-8.63-12.76 0-21.37 8.63-8.62 8.62-8.62 21.37v140q0 12.75 8.63 21.38 8.63 8.62 21.38 8.62 12.76 0 21.37-8.62M480-460"></path></symbol><symbol viewBox="0 -960 960 960" id="alt_route" xmlns="http://www.w3.org/2000/svg"><path d="M440-80v-200q0-56-17-83t-45-53l57-57q12 11 23 23.5t22 26.5q14-19 28.5-33.5T538-485q38-35 69-81t33-161l-63 63-57-56 160-160 160 160-56 56-64-63q-2 143-44 203.5T592-425q-32 29-52 56.5T520-280v200zM248-633q-4-20-5.5-44t-2.5-50l-64 63-56-56 160-160 160 160-57 56-63-62q0 21 2 39.5t4 34.5zm86 176q-20-21-38.5-49T263-575l77-19q10 27 23 46t28 34z"></path></symbol><symbol viewBox="0 -960 960 960" id="arrow_forward" xmlns="http://www.w3.org/2000/svg"><path d="M647-440H200q-17 0-28.5-11.5T160-480t11.5-28.5T200-520h447L451-716q-12-12-11.5-28t12.5-28q12-11 28-11.5t28 11.5l264 264q6 6 8.5 13t2.5 15-2.5 15-8.5 13L508-188q-11 11-27.5 11T452-188q-12-12-12-28.5t12-28.5z"></path></symbol><symbol viewBox="0 0 24 24" id="assistant" xmlns="http://www.w3.org/2000/svg"><path fill="none" d="M0 0h24v24H0z"></path><path d="M19 2H5c-1.1 0-2 .9-2 2v14c0 1.1.9 2 2 2h4l3 3 3-3h4c1.1 0 2-.9 2-2V4c0-1.1-.9-2-2-2m0 16h-4.83l-.59.59L12 20.17l-1.59-1.59-.58-.58H5V4h14zm-7-1 1.88-4.12L18 11l-4.12-1.88L12 5l-1.88 4.12L6 11l4.12 1.88z"></path></symbol><symbol fill="none" stroke="currentColor" stroke-linecap="round" stroke-linejoin="round" stroke-width="2" class="lucide lucide-badge-info-icon lucide-badge-info" viewBox="0 0 24 24" id="badge-info" xmlns="http://www.w3.org/2000/svg"><path d="M3.85 8.62a4 4 0 0 1 4.78-4.77 4 4 0 0 1 6.74 0 4 4 0 0 1 4.78 4.78 4 4 0 0 1 0 6.74 4 4 0 0 1-4.77 4.78 4 4 0 0 1-6.75 0 4 4 0 0 1-4.78-4.77 4 4 0 0 1 0-6.76M12 16v-4M12 8h.01"></path></symbol><symbol viewBox="0 -960 960 960" id="book" xmlns="http://www.w3.org/2000/svg"><path d="M461.19-202.35q-8.96-2.42-16.42-6.65-42.54-25.39-88.89-38.19Q309.54-260 260-260q-36.61 0-71.92 8.11Q152.77-243.77 120-228q-21.38 9.84-40.69-2.93Q60-243.69 60-267.08v-433.53q0-12.93 6.85-23.89 6.84-10.96 19-15.96 40.61-19.77 84.65-29.65Q214.54-780 260-780q58.38 0 113.69 16.54T480-716.92v459.38q50.62-32 106.81-47.23T700-320q34.46 0 65.11 4.85 30.66 4.84 63.35 15.3 4.23 1.16 7.89.2 3.65-.97 3.65-6.35v-449.23q10.38 3.08 20.27 6.85 9.88 3.77 19.27 9.46 10.23 5 15.34 14.61 5.12 9.62 5.12 20.62v435.84q0 23.39-19.89 35.96-19.88 12.58-42.42 3.12-32.38-15.39-67.11-23.31T700-260q-49.92 0-97.04 12.81-47.11 12.8-89.65 38.19-7.46 4.23-16.23 6.65-8.77 2.43-17.85 2.43t-18.04-2.43m129.5-185.34q-9.23 8.23-19.96 3.31Q560-389.31 560-401.54v-310.84q0-3.62 1.5-7.12t4.12-6.11l163.07-163.08q9.23-9.23 20.27-4.42 11.04 4.8 11.04 17.65v327.23q0 4.61-1.81 7.92t-4.42 5.93zM420-286.54v-394.07q-37.61-18.62-77.92-29.01Q301.77-720 260-720q-37 0-70.27 6.81t-62.42 18.88q-3.08 1.16-5.19 3.27-2.12 2.12-2.12 5.19v381.23q0 5.39 3.65 6.35 3.66.96 7.89-.58 29.61-10.69 61.04-15.92Q224-320 260-320q44.85 0 85.69 9.27 40.85 9.27 74.31 24.19m0 0v-394.07z"></path></symbol><symbol viewBox="0 0 24 24" id="book-cog-outline" xmlns="http://www.w3.org/2000/svg"><path d="M18 4h-5v8l-2.5-2.25L8 12V4H6v16h6.08c.1.71.31 1.38.61 2H6c-1.11 0-2-.89-2-2V4a2 2 0 0 1 2-2h12a2 2 0 0 1 2 2v8.08c-.33-.05-.66-.08-1-.08s-.67.03-1 .08zm5.8 16.4c.1 0 .1.1 0 .2l-1 1.7c-.1.1-.2.1-.3.1l-1.2-.4c-.3.2-.5.3-.8.5l-.2 1.3c0 .1-.1.2-.2.2h-2c-.1 0-.2-.1-.3-.2l-.2-1.3c-.3-.1-.6-.3-.8-.5l-1.2.5c-.1 0-.2 0-.3-.1l-1-1.7c-.1-.1 0-.2.1-.3l1.1-.8v-1l-1.1-.8c-.1-.1-.1-.2-.1-.3l1-1.7c.1-.1.2-.1.3-.1l1.2.5c.3-.2.5-.3.8-.5l.2-1.3c0-.1.1-.2.3-.2h2c.1 0 .2.1.2.2l.2 1.3c.3.1.6.3.9.5l1.2-.5c.1 0 .3 0 .3.1l1 1.7c.1.1 0 .2-.1.3l-1.1.8v1zM20.5 19c0-.8-.7-1.5-1.5-1.5s-1.5.7-1.5 1.5.7 1.5 1.5 1.5 1.5-.7 1.5-1.5"></path></symbol><symbol fill="currentColor" viewBox="0 0 24 24" id="book-marked" xmlns="http://www.w3.org/2000/svg"><path d="M3 18.5V5a3 3 0 0 1 3-3h14a1 1 0 0 1 1 1v18a1 1 0 0 1-1 1H6.5A3.5 3.5 0 0 1 3 18.5M19 20v-3H6.5a1.5 1.5 0 0 0 0 3zM10 4H6a1 1 0 0 0-1 1v10.337A3.5 3.5 0 0 1 6.5 15H19V4h-2v8l-3.5-2-3.5 2z"></path></symbol><symbol viewBox="0 0 24 24" id="book-open-variant" xmlns="http://www.w3.org/2000/svg"><path d="M17.5 14.33c.79 0 1.63.08 2.5.24v1.5c-.62-.16-1.46-.24-2.5-.24-1.9 0-3.39.33-4.5.99v-1.69c1.17-.53 2.67-.8 4.5-.8M13 12.46c1.29-.53 2.79-.79 4.5-.79.79 0 1.63.07 2.5.23v1.5c-.62-.16-1.46-.24-2.5-.24-1.9 0-3.39.34-4.5.99m4.5-3.65c-1.9 0-3.39.32-4.5 1V9.84Q14.845 9 17.5 9c.79 0 1.63.08 2.5.23v1.55c-.74-.19-1.59-.28-2.5-.28m3.5 8V7c-1.04-.33-2.21-.5-3.5-.5-2.05 0-3.88.5-5.5 1.5v11.5c1.62-1 3.45-1.5 5.5-1.5 1.19 0 2.36.16 3.5.5m-3.5-14c2.35 0 4.19.5 5.5 1.5v14.56c0 .12-.05.24-.16.35-.11.09-.23.17-.34.17s-.19-.02-.25-.05C20.97 20.34 19.38 20 17.5 20c-2.05 0-3.88.5-5.5 1.5-1.34-1-3.17-1.5-5.5-1.5-1.66 0-3.25.36-4.75 1.07-.03.01-.07.01-.12.03-.04.01-.08.02-.13.02-.11 0-.23-.04-.34-.12a.48.48 0 0 1-.16-.35V6c1.34-1 3.18-1.5 5.5-1.5 2.33 0 4.16.5 5.5 1.5 1.34-1 3.17-1.5 5.5-1.5"></path></symbol><symbol viewBox="0 0 24 24" id="book-white" xmlns="http://www.w3.org/2000/svg"><path d="m19 1-5 5v11l5-4.5zm2 4v13.5c-1.1-.35-2.3-.5-3.5-.5-1.7 0-4.15.65-5.5 1.5V6c-1.45-1.1-3.55-1.5-5.5-1.5S2.45 4.9 1 6v14.65c0 .25.25.5.5.5.1 0 .15-.05.25-.05C3.1 20.45 5.05 20 6.5 20c1.95 0 4.05.4 5.5 1.5 1.35-.85 3.8-1.5 5.5-1.5 1.65 0 3.35.3 4.75 1.05.1.05.15.05.25.05.25 0 .5-.25.5-.5V6c-.6-.45-1.25-.75-2-1M10 18.41C8.75 18.09 7.5 18 6.5 18c-1.06 0-2.32.19-3.5.5V7.13c.91-.4 2.14-.63 3.5-.63s2.59.23 3.5.63z"></path></symbol><symbol viewBox="0 -960 960 960" id="bookmark" xmlns="http://www.w3.org/2000/svg"><path d="M200-120v-640q0-33 23.5-56.5T280-840h400q33 0 56.5 23.5T760-760v640L480-240zm80-122 200-86 200 86v-518H280zm0-518h400z"></path></symbol><symbol viewBox="0 -960 960 960" id="bookmark-add" xmlns="http://www.w3.org/2000/svg"><path d="M200-120v-640q0-33 23.5-56.5T280-840h240v80H280v518l200-86 200 86v-278h80v400L480-240zm80-640h240zm400 160v-80h-80v-80h80v-80h80v80h80v80h-80v80z"></path></symbol><symbol viewBox="0 -960 960 960" id="bookmark-added" xmlns="http://www.w3.org/2000/svg"><path d="M713-600 600-713l56-57 57 57 141-142 57 57zM200-120v-640q0-33 23.5-56.5T280-840h240v80H280v518l200-86 200 86v-278h80v400L480-240zm80-640h240z"></path></symbol><symbol viewBox="0 -960 960 960" id="bookmark-remove" xmlns="http://www.w3.org/2000/svg"><path d="M840-680H600v-80h240zM200-120v-640q0-33 23.5-56.5T280-840h240v80H280v518l200-86 200 86v-278h80v400L480-240zm80-640h240z"></path></symbol><symbol viewBox="0 0 24 24" id="bookshelf" xmlns="http://www.w3.org/2000/svg"><path d="M9 3v15h3V3zm3 2 4 13 3-1-4-13zM5 5v13h3V5zM3 19v2h18v-2z"></path></symbol><symbol viewBox="0 0 24 24" id="bookshelf-white" xmlns="http://www.w3.org/2000/svg"><path d="M9 3v15h3V3zm3 2 4 13 3-1-4-13zM5 5v13h3V5zM3 19v2h18v-2z"></path></symbol><symbol fill="none" stroke="currentColor" stroke-linecap="round" stroke-linejoin="round" stroke-width="2" class="lucide lucide-bot-icon lucide-bot" viewBox="0 0 24 24" id="bot" xmlns="http://www.w3.org/2000/svg"><path d="M12 8V4H8"></path><rect width="16" height="12" x="4" y="8" rx="2"></rect><path d="M2 14h2M20 14h2M15 13v2M9 13v2"></path></symbol><symbol viewBox="0 0 24 24" id="build" xmlns="http://www.w3.org/2000/svg"><path fill="none" d="M0 0h24v24H0z" clip-rule="evenodd"></path><path d="m22.7 19-9.1-9.1c.9-2.3.4-5-1.5-6.9-2-2-5-2.4-7.4-1.3L9 6 6 9 1.6 4.7C.4 7.1.9 10.1 2.9 12.1c1.9 1.9 4.6 2.4 6.9 1.5l9.1 9.1c.4.4 1 .4 1.4 0l2.3-2.3c.5-.4.5-1.1.1-1.4"></path></symbol><symbol viewBox="0 0 1024 1024" id="chapter" xmlns="http://www.w3.org/2000/svg"><path d="M192 128h400l240 230.4V896H192zm160 153.6v76.8h160v-76.8zM672 512v-76.8H352V512z"></path></symbol><symbol viewBox="0 0 24 24" id="chart-box-outline" xmlns="http://www.w3.org/2000/svg"><path d="M9 17H7v-7h2zm4 0h-2V7h2zm4 0h-2v-4h2zm2 2H5V5h14v14.1M19 3H5c-1.1 0-2 .9-2 2v14c0 1.1.9 2 2 2h14c1.1 0 2-.9 2-2V5c0-1.1-.9-2-2-2"></path></symbol><symbol viewBox="0 -960 960 960" id="check" xmlns="http://www.w3.org/2000/svg"><path d="m382-339.38 345.54-345.54q8.92-8.93 20.88-9.12t21.27 9.12 9.31 21.38q0 12.08-9.31 21.39l-362.38 363q-10.85 10.84-25.31 10.84t-25.31-10.84l-167-167q-8.92-8.93-8.8-21.2.11-12.26 9.42-21.57t21.38-9.31q12.08 0 21.39 9.31z"></path></symbol><symbol viewBox="0 -960 960 960" id="check_circle" xmlns="http://www.w3.org/2000/svg"><path d="m423.23-394.15-92.92-92.93q-8.31-8.3-20.89-8.5-12.57-.19-21.27 8.5-8.69 8.7-8.69 21.08t8.69 21.08l109.77 109.77q10.85 10.84 25.31 10.84t25.31-10.84l222.54-222.54q8.3-8.31 8.5-20.89.19-12.57-8.5-21.27-8.7-8.69-21.08-8.69t-21.08 8.69zM480.07-100q-78.84 0-148.21-29.92t-120.68-81.21-81.25-120.63Q100-401.1 100-479.93q0-78.84 29.92-148.21t81.21-120.68 120.63-81.25Q401.1-860 479.93-860q78.84 0 148.21 29.92t120.68 81.21 81.25 120.63Q860-558.9 860-480.07q0 78.84-29.92 148.21t-81.21 120.68-120.63 81.25Q558.9-100 480.07-100m-.07-60q134 0 227-93t93-227-93-227-227-93-227 93-93 227 93 227 227 93m0-320"></path></symbol><symbol viewBox="0 -960 960 960" id="chevron_down" xmlns="http://www.w3.org/2000/svg"><path d="M465-363.5q-7-2.5-13-8.5L268-556q-11-11-11-28t11-28 28-11 28 11l156 156 156-156q11-11 28-11t28 11 11 28-11 28L508-372q-6 6-13 8.5t-15 2.5-15-2.5"></path></symbol><symbol viewBox="0 -960 960 960" id="close" xmlns="http://www.w3.org/2000/svg"><path d="M480-437.85 277.08-234.92q-8.31 8.3-20.89 8.5-12.57.19-21.27-8.5-8.69-8.7-8.69-21.08t8.69-21.08L437.85-480 234.92-682.92q-8.3-8.31-8.5-20.89-.19-12.57 8.5-21.27 8.7-8.69 21.08-8.69t21.08 8.69L480-522.15l202.92-202.93q8.31-8.3 20.89-8.5 12.57-.19 21.27 8.5 8.69 8.7 8.69 21.08t-8.69 21.08L522.15-480l202.93 202.92q8.3 8.31 8.5 20.89.19 12.57-8.5 21.27-8.7 8.69-21.08 8.69t-21.08-8.69z"></path></symbol><symbol viewBox="0 -960 960 960" id="cog-outline" xmlns="http://www.w3.org/2000/svg"><path d="M435.69-100q-20.46 0-35.34-13.58-14.89-13.57-18.12-33.42l-9.77-74.85q-16.07-5.38-32.96-15.07-16.88-9.7-30.19-20.77L240-228.23q-18.85 8.31-37.88 1.61-19.04-6.69-29.58-24.3l-45.08-78.16q-10.54-17.61-6.07-37.27 4.46-19.65 20.46-32.42l59.92-45q-1.38-8.92-1.96-17.92t-.58-17.93q0-8.53.58-17.34t1.96-19.27l-59.92-45q-16-12.77-20.27-32.62-4.27-19.84 6.27-37.46l44.69-77q10.54-17.23 29.58-24.11 19.03-6.89 37.88 1.42l68.92 29.08q14.47-11.46 30.89-20.96t32.27-15.27L382.23-813q3.23-19.85 18.12-33.42Q415.23-860 435.69-860h88.62q20.46 0 35.34 13.58 14.89 13.57 18.12 33.42l9.77 75.23q18 6.54 32.57 15.27 14.58 8.73 29.43 20.58L720.39-731q18.84-8.31 37.88-1.42 19.04 6.88 29.57 24.11l44.7 77.39q10.54 17.61 6.07 37.27-4.46 19.65-20.46 32.42l-61.46 46.15q2.15 9.69 2.35 18.12.19 8.42.19 16.96 0 8.15-.39 16.58-.38 8.42-2.76 19.27l60.3 45.38q16 12.77 20.66 32.42 4.65 19.66-5.89 37.27l-45.31 77.77q-10.53 17.62-29.76 24.31T718-228.62l-68.46-29.46q-14.85 11.85-30.31 20.96-15.46 9.12-31.69 14.89L577.77-147q-3.23 19.85-18.12 33.42Q544.77-100 524.31-100zm4.31-60h78.62L533-267.15q30.62-8 55.96-22.73 25.35-14.74 48.89-37.89L737.23-286l39.39-68-86.77-65.38q5-15.54 6.8-30.47 1.81-14.92 1.81-30.15 0-15.62-1.81-30.15-1.8-14.54-6.8-29.7L777.38-606 738-674l-100.54 42.38q-20.08-21.46-48.11-37.92-28.04-16.46-56.73-23.31L520-800h-79.38l-13.24 106.77q-30.61 7.23-56.53 22.15-25.93 14.93-49.47 38.46L222-674l-39.38 68L269-541.62q-5 14.24-7 29.62t-2 32.38q0 15.62 2 30.62t6.62 29.62l-86 65.38L222-286l99-42q22.77 23.38 48.69 38.31 25.93 14.92 57.31 22.92zm40.46-200q49.92 0 84.96-35.04T600.46-480t-35.04-84.96T480.46-600q-50.54 0-85.27 35.04T360.46-480t34.73 84.96T480.46-360M480-480"></path></symbol><symbol viewBox="0 0 1024 1024" id="comment" xmlns="http://www.w3.org/2000/svg"><path fill="none" d="M0 0h24v24H0zm0 0h24v24H0z"></path><path d="M930.27 506.415a418.53 418.53 0 0 1-50.632 199.68 418.4 418.4 0 0 1-139.75 151.315 418.22 418.22 0 0 1-397.968 31.45l-247.686 35.4 35.391-247.76a418.57 418.57 0 0 1 9.793-359.957A418.4 418.4 0 0 1 257.5 174.697a418.27 418.27 0 0 1 352.125-74.794 418.29 418.29 0 0 1 279.83 226.508A418.5 418.5 0 0 1 930.32 506.42h-.05z"></path></symbol><symbol viewBox="0 -960 960 960" id="content_copy" xmlns="http://www.w3.org/2000/svg"><path d="M362.31-260Q332-260 311-281t-21-51.31v-455.38Q290-818 311-839t51.31-21h335.38Q728-860 749-839t21 51.31v455.38Q770-302 749-281t-51.31 21zm0-60h335.38q4.62 0 8.46-3.85 3.85-3.84 3.85-8.46v-455.38q0-4.62-3.85-8.46-3.84-3.85-8.46-3.85H362.31q-4.62 0-8.46 3.85-3.85 3.84-3.85 8.46v455.38q0 4.62 3.85 8.46 3.84 3.85 8.46 3.85m-140 200Q192-120 171-141t-21-51.31v-485.38q0-12.77 8.62-21.39 8.61-8.61 21.38-8.61t21.39 8.61q8.61 8.62 8.61 21.39v485.38q0 4.62 3.85 8.46 3.84 3.85 8.46 3.85h365.38q12.77 0 21.39 8.61 8.61 8.62 8.61 21.39t-8.61 21.38q-8.62 8.62-21.39 8.62zM350-320v-480z"></path></symbol><symbol viewBox="0 0 54 47" id="creem" xmlns="http://www.w3.org/2000/svg"><path d="M7.447 1H1l25.97 45L53 1H9.393L26.97 31.486l3.223-5.466L19.063 6.71H43.33L26.97 34.887z"></path></symbol><symbol viewBox="0 -960 960 960" id="crown" xmlns="http://www.w3.org/2000/svg"><path d="M200-160v-80h560v80zm0-140-51-321q-2 0-4.5.5t-4.5.5q-25 0-42.5-17.5T80-680t17.5-42.5T140-740t42.5 17.5T200-680q0 7-1.5 13t-3.5 11l125 56 125-171q-11-8-18-21t-7-28q0-25 17.5-42.5T480-880t42.5 17.5T540-820q0 15-7 28t-18 21l125 171 125-56q-2-5-3.5-11t-1.5-13q0-25 17.5-42.5T820-740t42.5 17.5T880-680t-17.5 42.5T820-620q-2 0-4.5-.5t-4.5-.5l-51 321zm68-80h424l26-167-105 46-133-183-133 183-105-46zm212 0"></path></symbol><symbol viewBox="0 -960 960 960" id="delete" xmlns="http://www.w3.org/2000/svg"><path d="M292.31-140q-29.83 0-51.07-21.24T220-212.31V-720h-10q-12.75 0-21.37-8.63-8.63-8.63-8.63-21.38 0-12.76 8.63-21.37Q197.25-780 210-780h150q0-14.69 10.35-25.04 10.34-10.34 25.03-10.34h169.24q14.69 0 25.03 10.34Q600-794.69 600-780h150q12.75 0 21.37 8.63 8.63 8.63 8.63 21.38 0 12.76-8.63 21.37Q762.75-720 750-720h-10v507.69q0 29.83-21.24 51.07T667.69-140zM680-720H280v507.69q0 5.39 3.46 8.85t8.85 3.46h375.38q5.39 0 8.85-3.46t3.46-8.85zM427.54-288.62q8.61-8.63 8.61-21.38v-300q0-12.75-8.63-21.38-8.62-8.62-21.38-8.62-12.75 0-21.37 8.62-8.61 8.63-8.61 21.38v300q0 12.75 8.62 21.38 8.63 8.62 21.39 8.62 12.75 0 21.37-8.62m147.69 0q8.61-8.63 8.61-21.38v-300q0-12.75-8.62-21.38-8.63-8.62-21.39-8.62-12.75 0-21.37 8.62-8.61 8.63-8.61 21.38v300q0 12.75 8.63 21.38 8.62 8.62 21.38 8.62 12.75 0 21.37-8.62M280-720v520z"></path></symbol><symbol viewBox="0 -960 960 960" id="delete-sweep" xmlns="http://www.w3.org/2000/svg"><path d="M214.62-220q-30.31 0-51.31-21t-21-51.31V-640h-7.69q-12.75 0-21.38-8.63-8.62-8.63-8.62-21.38 0-12.76 8.62-21.37 8.63-8.62 21.38-8.62h122.3v-12.31q0-14.69 10.35-25.03 10.35-10.35 25.04-10.35h60q14.69 0 25.04 10.35 10.34 10.34 10.34 25.03V-700H510q12.75 0 21.37 8.63 8.63 8.63 8.63 21.38 0 12.76-8.63 21.37Q522.75-640 510-640h-7.69v347.69q0 30.31-21 51.31T430-220zm413.07-30q-12.75 0-21.37-8.63-8.63-8.63-8.63-21.38 0-12.76 8.63-21.37 8.62-8.62 21.37-8.62h80q12.75 0 21.38 8.63 8.62 8.63 8.62 21.38 0 12.76-8.62 21.37-8.63 8.62-21.38 8.62zm0-160q-12.75 0-21.37-8.63-8.63-8.63-8.63-21.38 0-12.76 8.63-21.37 8.62-8.62 21.37-8.62h160q12.75 0 21.38 8.63 8.62 8.63 8.62 21.38 0 12.76-8.62 21.37-8.63 8.62-21.38 8.62zm0-160q-12.75 0-21.37-8.63-8.63-8.63-8.63-21.38 0-12.76 8.63-21.37 8.62-8.62 21.37-8.62h200q12.75 0 21.38 8.63 8.62 8.63 8.62 21.38 0 12.76-8.62 21.37-8.63 8.62-21.38 8.62zm-425.38-70v347.69q0 4.62 3.84 8.46Q210-280 214.62-280H430q4.61 0 8.46-3.85 3.85-3.84 3.85-8.46V-640z"></path></symbol><symbol viewBox="0 -960 960 960" id="dictionary" xmlns="http://www.w3.org/2000/svg"><path d="M228.38-457.38h109.39l19 52.61q2 5.23 6.23 8.15 4.23 2.93 10.08 2.93 9.46 0 14.8-7.66 5.35-7.65 2.12-16.5l-81.23-214.84q-2-5.62-7.12-9.23-5.11-3.62-11.34-3.62h-14.46q-6.23 0-11.35 3.62-5.11 3.61-7.11 9.23l-81.24 215.23q-3.23 8.46 2.12 16.11 5.35 7.66 14.81 7.66 5.84 0 10.38-2.93 4.54-2.92 6.54-8.76zm12.24-33.16 41.46-114.69h2l41.46 114.69zM260-319.23q49.69 0 96.69 11.27T450-272.61v-393.24q-42.15-27.46-91.23-41.19T260-720.77q-36 0-67.27 5.65-31.27 5.66-64.27 18.5-4.61 1.54-6.54 4.43-1.92 2.88-1.92 6.34v378.31q0 5.39 3.85 7.89 3.84 2.5 8.46.57 28.46-9.69 60.07-14.92 31.62-5.23 67.62-5.23m250 46.62q46.31-24.08 93.31-35.35T700-319.23q36 0 67.62 5.23 31.61 5.23 60.07 14.92 4.62 1.93 8.46-.57 3.85-2.5 3.85-7.89v-378.31q0-3.46-1.92-6.15-1.93-2.69-6.54-4.62-33-12.84-64.27-18.5-31.27-5.65-67.27-5.65-49.69 0-98.77 13.73T510-665.85zm-51.88 68.3q-10.2-2.92-19.27-7.77-41.31-23.38-86.43-35.27-45.11-11.88-92.42-11.88-36.61 0-71.92 8.11Q152.77-243 120-227.23q-21.38 9.84-40.69-3.12T60-267.08v-434.3q0-12.93 6.66-24.27Q73.31-737 85.85-742q40.61-19.77 84.65-29.27t89.5-9.5q58.38 0 114.08 15.96 55.69 15.97 105.92 47.12 50.23-31.15 105.92-47.12 55.7-15.96 114.08-15.96 45.46 0 89.5 9.5T874.15-742q12.54 5 19.19 16.35 6.66 11.34 6.66 24.27v434.3q0 23.77-20.08 36.35-20.08 12.57-42.23 2.73-32.38-15.39-67.11-23.31T700-259.23q-47.31 0-92.42 11.88-45.12 11.89-86.43 35.27-9.07 4.85-19.27 7.77-10.19 2.92-21.88 2.92t-21.88-2.92m99.57-401.23q0-6.69 4.77-13.69t10.85-9.62q29.77-11.92 61.27-18.07 31.5-6.16 65.42-6.16 19.62 0 37.96 2.31 18.35 2.31 36.96 6.31 7.08 1.61 12.23 7.69 5.16 6.08 5.16 14.15 0 13.54-8.5 19.81t-22.04 3.04q-14.39-3-29.89-4.31T700-605.39q-29.08 0-56.96 5.58-27.89 5.58-53.19 15.12-14.16 5.46-23.16-.62-9-6.07-9-20.23m0 218.46q0-6.69 4.77-13.88t10.85-9.81q29-11.92 61.27-17.88t65.42-5.96q19.62 0 37.96 2.3 18.35 2.31 36.96 6.31 7.08 1.62 12.23 7.69 5.16 6.08 5.16 14.16 0 13.53-8.5 19.8t-22.04 3.04q-14.39-3-29.89-4.31-15.5-1.3-31.88-1.3-28.69 0-56.39 5.46-27.69 5.46-53 15.38-14.15 5.85-23.53-.3-9.39-6.16-9.39-20.7m0-108.84q0-6.69 4.77-13.69t10.85-9.62q29.77-11.92 61.27-18.08 31.5-6.15 65.42-6.15 19.62 0 37.96 2.31 18.35 2.3 36.96 6.3 7.08 1.62 12.23 7.7 5.16 6.07 5.16 14.15 0 13.54-8.5 19.81t-22.04 3.04q-14.39-3-29.89-4.31T700-495.77q-29.08 0-56.96 5.58-27.89 5.57-53.19 15.11-14.16 5.46-23.16-.61-9-6.08-9-20.23"></path></symbol><symbol viewBox="0 0 127.14 96.36" id="discord" xmlns="http://www.w3.org/2000/svg"><path d="M107.7 8.07A105.2 105.2 0 0 0 81.47 0a72 72 0 0 0-3.36 6.83 97.7 97.7 0 0 0-29.11 0A72 72 0 0 0 45.64 0a106 106 0 0 0-26.25 8.09C2.79 32.65-1.71 56.6.54 80.21a105.7 105.7 0 0 0 32.17 16.15 77.7 77.7 0 0 0 6.89-11.11 68.4 68.4 0 0 1-10.85-5.18c.91-.66 1.8-1.34 2.66-2a75.57 75.57 0 0 0 64.32 0c.87.71 1.76 1.39 2.66 2a68.7 68.7 0 0 1-10.87 5.19 77 77 0 0 0 6.89 11.1 105.3 105.3 0 0 0 32.19-16.14c2.64-27.38-4.51-51.11-18.9-72.15M42.45 65.69C36.18 65.69 31 60 31 53s5-12.74 11.43-12.74S54 46 53.89 53s-5.05 12.69-11.44 12.69m42.24 0C78.41 65.69 73.25 60 73.25 53s5-12.74 11.44-12.74S96.23 46 96.12 53s-5.04 12.69-11.43 12.69" class="cls-1"></path></symbol><symbol viewBox="0 -960 960 960" id="dock_to_right" xmlns="http://www.w3.org/2000/svg"><path d="M212.31-140q-29.92 0-51.12-21.19Q140-182.39 140-212.31v-535.38q0-29.92 21.19-51.12Q182.39-820 212.31-820h535.38q29.92 0 51.12 21.19Q820-777.61 820-747.69v535.38q0 29.92-21.19 51.12Q777.61-140 747.69-140zM320-200v-560H212.31q-4.62 0-8.46 3.85-3.85 3.84-3.85 8.46v535.38q0 4.62 3.85 8.46 3.84 3.85 8.46 3.85zm60 0h367.69q4.62 0 8.46-3.85 3.85-3.84 3.85-8.46v-535.38q0-4.62-3.85-8.46-3.84-3.85-8.46-3.85H380zm-60 0H200z"></path></symbol><symbol viewBox="0 -960 960 960" id="edit" xmlns="http://www.w3.org/2000/svg"><path d="M400-360q-17 0-28.5-11.5T360-400v-97q0-16 6-30.5t17-25.5l344-344q12-12 27-18t30-6q16 0 30.5 6t26.5 18l56 57q11 12 17 26.5t6 29.5-5.5 29.5T897-728L553-384q-11 11-25.5 17.5T497-360zm384-368 57-56-56-56-57 56zM200-120q-33 0-56.5-23.5T120-200v-560q0-33 23.5-56.5T200-840h260q14 0 23 7t14 18 3.5 22-11.5 21L303-586q-11 11-17 25.5t-6 30.5v170q0 33 23.5 56.5T360-280h169q16 0 30.5-6t25.5-17l187-187q10-10 21-11.5t22 3.5 18 14 7 23v261q0 33-23.5 56.5T760-120z"></path></symbol><symbol viewBox="0 -960 960 960" id="emergency" xmlns="http://www.w3.org/2000/svg"><path d="M480-79q-16 0-30.5-6T423-102L102-423q-11-12-17-26.5T79-480t6-31 17-26l321-321q12-12 26.5-17.5T480-881t31 5.5 26 17.5l321 321q12 11 17.5 26t5.5 31-5.5 30.5T858-423L537-102q-11 11-26 17t-31 6m0-80 321-321-321-321-321 321zm-40-281h80v-240h-80zm40 120q17 0 28.5-11.5T520-360t-11.5-28.5T480-400t-28.5 11.5T440-360t11.5 28.5T480-320m0-160"></path></symbol><symbol fill="none" stroke="currentColor" stroke-linecap="round" stroke-linejoin="round" stroke-width="2" class="lucide lucide-external-link-icon lucide-external-link" viewBox="0 0 24 24" id="external-link" xmlns="http://www.w3.org/2000/svg"><path d="M15 3h6v6M10 14 21 3M18 13v6a2 2 0 0 1-2 2H5a2 2 0 0 1-2-2V8a2 2 0 0 1 2-2h6"></path></symbol><symbol viewBox="0 0 24 24" id="eye-fill" xmlns="http://www.w3.org/2000/svg"><path d="M12 9a3 3 0 0 0-3 3 3 3 0 0 0 3 3 3 3 0 0 0 3-3 3 3 0 0 0-3-3m0 8a5 5 0 0 1-5-5 5 5 0 0 1 5-5 5 5 0 0 1 5 5 5 5 0 0 1-5 5m0-12.5C7 4.5 2.73 7.61 1 12c1.73 4.39 6 7.5 11 7.5s9.27-3.11 11-7.5c-1.73-4.39-6-7.5-11-7.5"></path></symbol><symbol viewBox="0 -960 960 960" id="fast-release" xmlns="http://www.w3.org/2000/svg"><path d="m354-334 356-94q15-4 22.5-18.5T736-476t-17.5-22.5T690-502l-98 26-160-150-56 14 96 168-96 24-50-38-38 10zm446 174H160q-33 0-56.5-23.5T80-240v-160q33 0 56.5-23.5T160-480t-23.5-56.5T80-560v-160q0-33 23.5-56.5T160-800h640q33 0 56.5 23.5T880-720v480q0 33-23.5 56.5T800-160"></path></symbol><symbol viewBox="0 -960 960 960" id="first_page" xmlns="http://www.w3.org/2000/svg"><path d="M251.5-251.5Q240-263 240-280v-400q0-17 11.5-28.5T280-720t28.5 11.5T320-680v400q0 17-11.5 28.5T280-240t-28.5-11.5M552-480l156 156q11 11 11 28t-11 28-28 11-28-11L468-452q-6-6-8.5-13t-2.5-15 2.5-15 8.5-13l184-184q11-11 28-11t28 11 11 28-11 28z"></path></symbol><symbol fill="none" stroke="currentColor" stroke-linecap="round" stroke-linejoin="round" stroke-width="2" class="lucide lucide-funnel-icon lucide-funnel" viewBox="0 0 24 24" id="funnel" xmlns="http://www.w3.org/2000/svg"><path d="M10 20a1 1 0 0 0 .553.895l2 1A1 1 0 0 0 14 21v-7a2 2 0 0 1 .517-1.341L21.74 4.67A1 1 0 0 0 21 3H3a1 1 0 0 0-.742 1.67l7.225 7.989A2 2 0 0 1 10 14z"></path></symbol><symbol viewBox="0 -960 960 960" id="g_translate" xmlns="http://www.w3.org/2000/svg"><path d="m480-80-40-120H160q-33 0-56.5-23.5T80-280v-520q0-33 23.5-56.5T160-880h240l35 120h365q35 0 57.5 22.5T880-680v520q0 33-22.5 56.5T800-80zM286-376q69 0 113.5-44.5T444-536q0-8-.5-14.5T441-564H283v62h89q-8 28-30.5 43.5T287-443q-39 0-67-28t-28-69 28-69 67-28q18 0 34 6.5t29 19.5l49-47q-21-22-50.5-34T286-704q-67 0-114.5 47.5T124-540t47.5 116.5T286-376m268 20 22-21q-14-17-25.5-33T528-444zm50-51q28-33 42.5-63t19.5-47H507l12 42h40q8 15 19 32.5t26 35.5m-84 287h280q18 0 29-11.5t11-28.5v-520q0-18-11-29t-29-11H447l47 162h79v-42h41v42h146v41h-51q-10 38-30 74t-47 67l109 107-29 29-108-108-36 37 32 111z"></path></symbol><symbol viewBox="0 -960 960 960" id="gift" xmlns="http://www.w3.org/2000/svg"><path d="M160-80v-440H80v-240h208q-5-9-6.5-19t-1.5-21q0-50 35-85t85-35q23 0 43 8.5t37 23.5q17-16 37-24t43-8q50 0 85 35t35 85q0 11-2 20.5t-6 19.5h208v240h-80v440zm400-760q-17 0-28.5 11.5T520-800t11.5 28.5T560-760t28.5-11.5T600-800t-11.5-28.5T560-840m-200 40q0 17 11.5 28.5T400-760t28.5-11.5T440-800t-11.5-28.5T400-840t-28.5 11.5T360-800M160-680v80h280v-80zm280 520v-360H240v360zm80 0h200v-360H520zm280-440v-80H520v80z"></path></symbol><symbol viewBox="0 0 24 24" id="github" xmlns="http://www.w3.org/2000/svg"><path d="M12 .297c-6.63 0-12 5.373-12 12 0 5.303 3.438 9.8 8.205 11.385.6.113.82-.258.82-.577 0-.285-.01-1.04-.015-2.04-3.338.724-4.042-1.61-4.042-1.61C4.422 18.07 3.633 17.7 3.633 17.7c-1.087-.744.084-.729.084-.729 1.205.084 1.838 1.236 1.838 1.236 1.07 1.835 2.809 1.305 3.495.998.108-.776.417-1.305.76-1.605-2.665-.3-5.466-1.332-5.466-5.93 0-1.31.465-2.38 1.235-3.22-.135-.303-.54-1.523.105-3.176 0 0 1.005-.322 3.3 1.23.96-.267 1.98-.399 3-.405 1.02.006 2.04.138 3 .405 2.28-1.552 3.285-1.23 3.285-1.23.645 1.653.24 2.873.12 3.176.765.84 1.23 1.91 1.23 3.22 0 4.61-2.805 5.625-5.475 5.92.42.36.81 1.096.81 2.22 0 1.606-.015 2.896-.015 3.286 0 .315.21.69.825.57C20.565 22.092 24 17.592 24 12.297c0-6.627-5.373-12-12-12"></path></symbol><symbol viewBox="0 0 24 24" id="google" xmlns="http://www.w3.org/2000/svg"><path d="M12.48 10.92v3.28h7.84c-.24 1.84-.853 3.187-1.787 4.133-1.147 1.147-2.933 2.4-6.053 2.4-4.827 0-8.6-3.893-8.6-8.72s3.773-8.72 8.6-8.72c2.6 0 4.507 1.027 5.907 2.347l2.307-2.307C18.747 1.44 16.133 0 12.48 0 5.867 0 .307 5.387.307 12s5.56 12 12.173 12c3.573 0 6.267-1.173 8.373-3.36 2.16-2.16 2.84-5.213 2.84-7.667 0-.76-.053-1.467-.173-2.053z"></path></symbol><symbol viewBox="0 -960 960 960" id="grid" xmlns="http://www.w3.org/2000/svg"><path d="M200-120q-33 0-56.5-23.5T120-200v-560q0-33 23.5-56.5T200-840h560q33 0 56.5 23.5T840-760v560q0 33-23.5 56.5T760-120zm0-80h133v-133H200zm213 0h134v-133H413zm214 0h133v-133H627zM200-413h133v-134H200zm213 0h134v-134H413zm214 0h133v-134H627zM200-627h133v-133H200zm213 0h134v-133H413zm214 0h133v-133H627z"></path></symbol><symbol viewBox="0 -960 960 960" id="group" xmlns="http://www.w3.org/2000/svg"><path d="M40-160v-112q0-34 17.5-62.5T104-378q62-31 126-46.5T360-440t130 15.5T616-378q29 15 46.5 43.5T680-272v112zm720 0v-120q0-44-24.5-84.5T666-434q51 6 96 20.5t84 35.5q36 20 55 44.5t19 53.5v120zM247-527q-47-47-47-113t47-113 113-47 113 47 47 113-47 113-113 47-113-47m466 0q-47 47-113 47-11 0-28-2.5t-28-5.5q27-32 41.5-71t14.5-81-14.5-81-41.5-71q14-5 28-6.5t28-1.5q66 0 113 47t47 113-47 113M120-240h480v-32q0-11-5.5-20T580-306q-54-27-109-40.5T360-360t-111 13.5T140-306q-9 5-14.5 14t-5.5 20zm296.5-343.5Q440-607 440-640t-23.5-56.5T360-720t-56.5 23.5T280-640t23.5 56.5T360-560t56.5-23.5M360-640"></path></symbol><symbol fill="none" stroke="currentColor" stroke-linecap="round" stroke-linejoin="round" stroke-width="2" class="lucide lucide-heart-icon lucide-heart" viewBox="0 0 24 24" id="heart" xmlns="http://www.w3.org/2000/svg"><path d="M2 9.5a5.5 5.5 0 0 1 9.591-3.676.56.56 0 0 0 .818 0A5.49 5.49 0 0 1 22 9.5c0 2.29-1.5 4-3 5.5l-5.492 5.313a2 2 0 0 1-3 .019L5 15c-1.5-1.5-3-3.2-3-5.5"></path></symbol><symbol viewBox="0 -960 960 960" id="help" xmlns="http://www.w3.org/2000/svg"><path d="M478-240q21 0 35.5-14.5T528-290t-14.5-35.5T478-340t-35.5 14.5T428-290t14.5 35.5T478-240m-36-154h74q0-33 7.5-52t42.5-52q26-26 41-49.5t15-56.5q0-56-41-86t-97-30q-57 0-92.5 30T342-618l66 26q5-18 22.5-39t53.5-21q32 0 48 17.5t16 38.5q0 20-12 37.5T506-526q-44 39-54 59t-10 73m38 314q-83 0-156-31.5T197-197t-85.5-127T80-480t31.5-156T197-763t127-85.5T480-880t156 31.5T763-763t85.5 127T880-480t-31.5 156T763-197t-127 85.5T480-80m0-80q134 0 227-93t93-227-93-227-227-93-227 93-93 227 93 227 227 93m0-320"></path></symbol><symbol viewBox="0 -960 960 960" id="history" xmlns="http://www.w3.org/2000/svg"><path d="M480-120q-138 0-240.5-91.5T122-440h82q14 104 92.5 172T480-200q117 0 198.5-81.5T760-480t-81.5-198.5T480-760q-69 0-129 32t-101 88h110v80H120v-240h80v94q51-64 124.5-99T480-840q75 0 140.5 28.5t114 77 77 114T840-480t-28.5 140.5-77 114-114 77T480-120m112-192L440-464v-216h80v184l128 128z"></path></symbol><symbol viewBox="0 -960 960 960" id="home" xmlns="http://www.w3.org/2000/svg"><path d="M240-200h133.85v-201.54q0-15.36 10.39-25.76 10.4-10.39 25.76-10.39h140q15.36 0 25.76 10.39 10.39 10.4 10.39 25.76V-200H720v-353.85q0-3.07-1.35-5.57-1.34-2.5-3.65-4.43L487.31-735q-3.08-2.69-7.31-2.69t-7.31 2.69L245-563.85q-2.31 1.93-3.65 4.43-1.35 2.5-1.35 5.57zm-60 0v-353.85q0-17.17 7.68-32.53 7.69-15.37 21.24-25.31l227.7-171.54q18.95-14.46 43.32-14.46t43.44 14.46l227.7 171.54q13.55 9.94 21.24 25.31 7.68 15.36 7.68 32.53V-200q0 24.54-17.73 42.27T720-140H562.31q-15.37 0-25.76-10.4-10.4-10.39-10.4-25.76v-201.53h-92.3v201.53q0 15.37-10.4 25.76-10.39 10.4-25.76 10.4H240q-24.54 0-42.27-17.73T180-200m300-269.23"></path></symbol><symbol viewBox="0 -960 960 960" id="inbox" xmlns="http://www.w3.org/2000/svg"><path d="M200-120q-33 0-56.5-23.5T120-200v-560q0-33 23.5-56.5T200-840h560q33 0 56.5 23.5T840-760v560q0 33-23.5 56.5T760-120zm0-80h560v-120H640q-30 38-71.5 59T480-240t-88.5-21-71.5-59H200zm280-120q38 0 69-22t43-58h168v-360H200v360h168q12 36 43 58t69 22M200-200h560z"></path></symbol><symbol fill="none" stroke="currentColor" stroke-linecap="round" stroke-linejoin="round" stroke-width="2" class="lucide lucide-info-icon lucide-info" viewBox="0 0 24 24" id="info" xmlns="http://www.w3.org/2000/svg"><circle cx="12" cy="12" r="10"></circle><path d="M12 16v-4M12 8h.01"></path></symbol><symbol viewBox="0 0 24 24" id="library-shelves" xmlns="http://www.w3.org/2000/svg"><path d="M19.5 9V1.5h-3V9h-3V1.5h-3V9h-3V1.5H4.65V9H3v1.5h18V9zm0 4.5h-3V21h-3v-7.5h-3V21h-3v-7.5H4.65V21H3v1.5h18V21h-1.5z"></path></symbol><symbol viewBox="0 0 24 24" id="like" xmlns="http://www.w3.org/2000/svg"><path fill="none" d="M0 0h24v24H0zm0 0h24v24H0z"></path><path d="M13.12 2.06 7.58 7.6c-.37.37-.58.88-.58 1.41V19c0 1.1.9 2 2 2h9c.8 0 1.52-.48 1.84-1.21l3.26-7.61C23.94 10.2 22.49 8 20.34 8h-5.65l.95-4.58c.1-.5-.05-1.01-.41-1.37-.59-.58-1.53-.58-2.11.01M3 21c1.1 0 2-.9 2-2v-8c0-1.1-.9-2-2-2s-2 .9-2 2v8c0 1.1.9 2 2 2"></path></symbol><symbol viewBox="0 -960 960 960" id="list" xmlns="http://www.w3.org/2000/svg"><path d="M280-600v-80h560v80zm0 160v-80h560v80zm0 160v-80h560v80zM160-600q-17 0-28.5-11.5T120-640t11.5-28.5T160-680t28.5 11.5T200-640t-11.5 28.5T160-600m0 160q-17 0-28.5-11.5T120-480t11.5-28.5T160-520t28.5 11.5T200-480t-11.5 28.5T160-440m0 160q-17 0-28.5-11.5T120-320t11.5-28.5T160-360t28.5 11.5T200-320t-11.5 28.5T160-280"></path></symbol><symbol viewBox="0 0 24 24" id="list_check" xmlns="http://www.w3.org/2000/svg"><path fill="none" d="M0 0h24v24H0z"></path><path d="M3 10h11v2H3zM3 6h11v2H3zM3 14h7v2H3zM20.59 11.93l-4.25 4.24-2.12-2.12-1.41 1.41L16.34 19 22 13.34z"></path></symbol><symbol viewBox="0 -960 960 960" id="lock" xmlns="http://www.w3.org/2000/svg"><path d="M240-80q-33 0-56.5-23.5T160-160v-400q0-33 23.5-56.5T240-640h40v-80q0-83 58.5-141.5T480-920t141.5 58.5T680-720v80h40q33 0 56.5 23.5T800-560v400q0 33-23.5 56.5T720-80zm0-80h480v-400H240zm296.5-143.5Q560-327 560-360t-23.5-56.5T480-440t-56.5 23.5T400-360t23.5 56.5T480-280t56.5-23.5M360-640h240v-80q0-50-35-85t-85-35-85 35-35 85zM240-160v-400z"></path></symbol><symbol viewBox="0 0 24 24" id="login" xmlns="http://www.w3.org/2000/svg"><path d="M10 17v-3H3v-4h7V7l5 5zm0-15h9a2 2 0 0 1 2 2v16a2 2 0 0 1-2 2h-9a2 2 0 0 1-2-2v-2h2v2h9V4h-9v2H8V4a2 2 0 0 1 2-2"></path></symbol><symbol viewBox="0 0 24 24" id="logout" xmlns="http://www.w3.org/2000/svg"><path d="M16 17v-3H9v-4h7V7l5 5zM14 2a2 2 0 0 1 2 2v2h-2V4H5v16h9v-2h2v2a2 2 0 0 1-2 2H5a2 2 0 0 1-2-2V4a2 2 0 0 1 2-2z"></path></symbol><symbol viewBox="0 0 24 24" id="magnify" xmlns="http://www.w3.org/2000/svg"><path d="M9.5 3A6.5 6.5 0 0 1 16 9.5c0 1.61-.59 3.09-1.56 4.23l.27.27h.79l5 5-1.5 1.5-5-5v-.79l-.27-.27A6.52 6.52 0 0 1 9.5 16 6.5 6.5 0 0 1 3 9.5 6.5 6.5 0 0 1 9.5 3m0 2C7 5 5 7 5 9.5S7 14 9.5 14 14 12 14 9.5 12 5 9.5 5"></path></symbol><symbol viewBox="0 -960 960 960" id="mail" xmlns="http://www.w3.org/2000/svg"><path d="M160-160q-33 0-56.5-23.5T80-240v-480q0-33 23.5-56.5T160-800h640q33 0 56.5 23.5T880-720v480q0 33-23.5 56.5T800-160zm320-280L160-640v400h640v-400zm0-80 320-200H160zM160-640v-80 480z"></path></symbol><symbol fill="none" stroke="currentColor" stroke-linecap="round" stroke-linejoin="round" stroke-width="2" class="lucide lucide-mail-check-icon lucide-mail-check" viewBox="0 0 24 24" id="mail-check" xmlns="http://www.w3.org/2000/svg"><path d="M22 13V6a2 2 0 0 0-2-2H4a2 2 0 0 0-2 2v12c0 1.1.9 2 2 2h8"></path><path d="m22 7-8.97 5.7a1.94 1.94 0 0 1-2.06 0L2 7M16 19l2 2 4-4"></path></symbol><symbol viewBox="0 0 24 24" id="menu" xmlns="http://www.w3.org/2000/svg"><path d="M3 6h18v2H3zm0 5h18v2H3zm0 5h18v2H3z"></path></symbol><symbol fill="none" stroke="currentColor" stroke-linecap="round" stroke-linejoin="round" stroke-width="2" class="lucide lucide-minus-icon lucide-minus" viewBox="0 0 24 24" id="minus" xmlns="http://www.w3.org/2000/svg"><path d="M5 12h14"></path></symbol><symbol fill="none" stroke="currentColor" stroke-linecap="round" stroke-linejoin="round" stroke-width="2" class="lucide lucide-monitor-play-icon lucide-monitor-play" viewBox="0 0 24 24" id="monitor-play" xmlns="http://www.w3.org/2000/svg"><path d="M15.033 9.44a.647.647 0 0 1 0 1.12l-4.065 2.352a.645.645 0 0 1-.968-.56V7.648a.645.645 0 0 1 .967-.56zM12 17v4M8 21h8"></path><rect width="20" height="14" x="2" y="3" rx="2"></rect></symbol><symbol viewBox="0 -960 960 960" id="more" xmlns="http://www.w3.org/2000/svg"><path d="M480-160q-33 0-56.5-23.5T400-240t23.5-56.5T480-320t56.5 23.5T560-240t-23.5 56.5T480-160m0-240q-33 0-56.5-23.5T400-480t23.5-56.5T480-560t56.5 23.5T560-480t-23.5 56.5T480-400m0-240q-33 0-56.5-23.5T400-720t23.5-56.5T480-800t56.5 23.5T560-720t-23.5 56.5T480-640"></path></symbol><symbol viewBox="0 -960 960 960" id="more_horiz" xmlns="http://www.w3.org/2000/svg"><path d="M249.23-420q-24.75 0-42.37-17.63-17.63-17.62-17.63-42.37t17.63-42.37Q224.48-540 249.23-540t42.38 17.63q17.62 17.62 17.62 42.37t-17.62 42.37Q273.98-420 249.23-420M480-420q-24.75 0-42.37-17.63Q420-455.25 420-480t17.63-42.37Q455.25-540 480-540t42.37 17.63Q540-504.75 540-480t-17.63 42.37Q504.75-420 480-420m230.77 0q-24.75 0-42.38-17.63-17.62-17.62-17.62-42.37t17.62-42.37Q686.02-540 710.77-540t42.37 17.63q17.63 17.62 17.63 42.37t-17.63 42.37Q735.52-420 710.77-420"></path></symbol><symbol viewBox="0 -960 960 960" id="news" xmlns="http://www.w3.org/2000/svg"><path d="M162.31-130q-29.92 0-51.12-21.19Q90-172.39 90-202.31v-591.77q0-6.23 5.42-8.53 5.43-2.31 10.04 2.3L147-758.77l53.15-54.15q5.62-5.62 12.85-5.62t12.85 5.62L280-758.77l54.15-54.15q5.62-5.62 12.85-5.62t12.85 5.62L413-758.77l54.15-54.15q5.62-5.62 12.85-5.62t12.85 5.62L547-758.77l53.15-54.15q5.62-5.62 12.85-5.62t12.85 5.62L680-758.77l54.15-54.15q5.62-5.62 12.85-5.62t12.85 5.62L813-758.77l41.54-41.54q4.61-4.61 10.04-2.3 5.42 2.3 5.42 8.53v591.77q0 29.92-21.19 51.12Q827.61-130 797.69-130zm0-60H450v-260H150v247.69q0 5.39 3.46 8.85t8.85 3.46M510-190h287.69q5.39 0 8.85-3.46t3.46-8.85V-290H510zm0-160h300v-100H510zM150-510h660v-143.85H150z"></path></symbol><symbol viewBox="0 0 24 24" id="nexus" xmlns="http://www.w3.org/2000/svg"><g fill="none" stroke="currentColor" stroke-linecap="round" stroke-linejoin="round" stroke-width="1.5"><path d="M12 12 3.34 7 12 2l8.66 5zM7.67 4.5 12 7"></path><path d="M12 12 3.34 7v10L12 22zM3.34 7 12 22M12 12l8.66-5v10L12 22zM12 12l8.66 5M20.66 7 12 22"></path></g></symbol><symbol viewBox="0 0 24 24" id="outlined_all_inclusive" xmlns="http://www.w3.org/2000/svg"><path d="M18.6 6.62c-1.44 0-2.8.56-3.77 1.53L7.8 14.39c-.64.64-1.49.99-2.4.99-1.87 0-3.39-1.51-3.39-3.38S3.53 8.62 5.4 8.62c.91 0 1.76.35 2.44 1.03l1.13 1 1.51-1.34L9.22 8.2A5.37 5.37 0 0 0 5.4 6.62C2.42 6.62 0 9.04 0 12s2.42 5.38 5.4 5.38c1.44 0 2.8-.56 3.77-1.53l7.03-6.24c.64-.64 1.49-.99 2.4-.99 1.87 0 3.39 1.51 3.39 3.38s-1.52 3.38-3.39 3.38c-.9 0-1.76-.35-2.44-1.03l-1.14-1.01-1.51 1.34 1.27 1.12a5.39 5.39 0 0 0 3.82 1.57c2.98 0 5.4-2.41 5.4-5.38s-2.42-5.37-5.4-5.37"></path></symbol><symbol viewBox="0 0 24 24" id="outlined_done_all" xmlns="http://www.w3.org/2000/svg"><path d="m18 7-1.41-1.41-6.34 6.34 1.41 1.41zm4.24-1.41L11.66 16.17 7.48 12l-1.41 1.41L11.66 19l12-12zM.41 13.41 6 19l1.41-1.41L1.83 12z"></path></symbol><symbol viewBox="0 0 24 24" id="outlined_folder" xmlns="http://www.w3.org/2000/svg"><path d="m9.17 6 2 2H20v10H4V6zM10 4H4c-1.1 0-1.99.9-1.99 2L2 18c0 1.1.9 2 2 2h16c1.1 0 2-.9 2-2V8c0-1.1-.9-2-2-2h-8z"></path></symbol><symbol viewBox="0 0 24 24" id="outlined_update" xmlns="http://www.w3.org/2000/svg"><path d="M11 8v5l4.25 2.52.77-1.28-3.52-2.09V8zm10 2V3l-2.64 2.64A8.94 8.94 0 0 0 12 3a9 9 0 1 0 9 9h-2c0 3.86-3.14 7-7 7s-7-3.14-7-7 3.14-7 7-7c1.93 0 3.68.79 4.95 2.05L14 10z"></path></symbol><symbol viewBox="0 0 24 24" id="patreon" xmlns="http://www.w3.org/2000/svg"><path d="M22.957 7.21c-.004-3.064-2.391-5.576-5.191-6.482-3.478-1.125-8.064-.962-11.384.604C2.357 3.231 1.093 7.391 1.046 11.54c-.039 3.411.302 12.396 5.369 12.46 3.765.047 4.326-4.804 6.068-7.141 1.24-1.662 2.836-2.132 4.801-2.618 3.376-.836 5.678-3.501 5.673-7.031"></path></symbol><symbol viewBox="0 -960 960 960" id="pause" xmlns="http://www.w3.org/2000/svg"><path d="M560-200v-560h160v560zm-320 0v-560h160v560z"></path></symbol><symbol viewBox="0 0 1024 1024" id="pen" xmlns="http://www.w3.org/2000/svg"><path d="M891.688 330.594 706.063 617.469 317.938 739.812l244.687-244.687c12.656 4.219 29.531 0 42.188-12.656 16.875-16.875 16.875-42.188 0-59.063s-42.188-16.875-59.063 0c-12.656 12.657-12.656 25.313-12.656 42.188L288.406 710.28 410.75 322.156l282.656-189.843C638.563 107 575.281 90.125 512 90.125 279.969 90.125 90.125 279.969 90.125 512S279.969 933.875 512 933.875 933.875 744.031 933.875 512c0-63.281-16.875-126.562-42.187-181.406"></path></symbol><symbol fill="none" stroke="currentColor" stroke-linecap="round" stroke-linejoin="round" stroke-width="2" class="lucide lucide-pencil-icon lucide-pencil" viewBox="0 0 24 24" id="pencil" xmlns="http://www.w3.org/2000/svg"><path d="M21.174 6.812a1 1 0 0 0-3.986-3.987L3.842 16.174a2 2 0 0 0-.5.83l-1.321 4.352a.5.5 0 0 0 .623.622l4.353-1.32a2 2 0 0 0 .83-.497zM15 5l4 4"></path></symbol><symbol viewBox="0 -960 960 960" id="play" xmlns="http://www.w3.org/2000/svg"><path d="M320-200v-560l440 280z"></path></symbol><symbol viewBox="0 0 24 24" id="plus-box-outline" xmlns="http://www.w3.org/2000/svg"><path fill="currentColor" d="M19 19H5V5h14m0-2H5a2 2 0 0 0-2 2v14a2 2 0 0 0 2 2h14a2 2 0 0 0 2-2V5a2 2 0 0 0-2-2m-8 4h2v4h4v2h-4v4h-2v-4H7v-2h4z"></path></symbol><symbol viewBox="0 -960 960 960" id="premium" xmlns="http://www.w3.org/2000/svg"><path d="m387-412 35-114-92-74h114l36-112 36 112h114l-93 74 35 114-92-71zM240-40v-309q-38-42-59-96t-21-115q0-134 93-227t227-93 227 93 93 227q0 61-21 115t-59 96v309l-240-80zm240-280q100 0 170-70t70-170-70-170-170-70-170 70-70 170 70 170 170 70M320-159l160-41 160 41v-124q-35 20-75.5 31.5T480-240t-84.5-11.5T320-283zm160-62"></path></symbol><symbol viewBox="0 -960 960 960" id="priority" xmlns="http://www.w3.org/2000/svg"><path d="M360-120q-100 0-170-70t-70-170v-240q0-100 70-170t170-70h240q100 0 170 70t70 170v240q0 100-70 170t-170 70zm80-200 240-240-56-56-184 184-88-88-56 56zm-80 120h240q66 0 113-47t47-113v-240q0-66-47-113t-113-47H360q-66 0-113 47t-47 113v240q0 66 47 113t113 47m120-280"></path></symbol><symbol fill="none" viewBox="0 0 24 24" id="profile" xmlns="http://www.w3.org/2000/svg"><path fill="currentColor" d="M12 12c2.21 0 4-1.79 4-4s-1.79-4-4-4-4 1.79-4 4 1.79 4 4 4m0 2c-2.67 0-8 1.34-8 4v2h16v-2c0-2.66-5.33-4-8-4"></path></symbol><symbol viewBox="0 -960 960 960" id="profile2" xmlns="http://www.w3.org/2000/svg"><path d="M480-480q-66 0-113-47t-47-113 47-113 113-47 113 47 47 113-47 113-113 47M160-160v-112q0-34 17.5-62.5T224-378q62-31 126-46.5T480-440t130 15.5T736-378q29 15 46.5 43.5T800-272v112zm80-80h480v-32q0-11-5.5-20T700-306q-54-27-109-40.5T480-360t-111 13.5T260-306q-9 5-14.5 14t-5.5 20zm240-320q33 0 56.5-23.5T560-640t-23.5-56.5T480-720t-56.5 23.5T400-640t23.5 56.5T480-560m0 320"></path></symbol><symbol viewBox="0 -960 960 960" id="profile3" xmlns="http://www.w3.org/2000/svg"><path d="M400-492.309q-57.749 0-98.874-41.124t-41.125-98.874 41.125-98.874T400-772.306t98.874 41.125q41.125 41.124 41.125 98.874t-41.125 98.874T400-492.309M100.001-187.694v-88.922q0-30.307 15.462-54.884 15.461-24.576 43.153-38.038 49.847-24.846 107.692-41.5Q324.154-427.691 400-427.691h11.692q4.846 0 10.462 1.23-6.077 14.154-10.039 28.846t-6.576 29.922H400q-69.077 0-122.307 15.885t-91.539 35.807q-13.615 7.308-19.885 17.077-6.269 9.77-6.269 22.308v28.923h252q4.461 15.23 11.577 30.922t15.653 29.077zm544.23 29.614-8.923-53.076q-14.308-4.231-26.923-11.077-12.616-6.846-24-17.154l-50.692 17.615-28.461-48.383 41.384-33.846q-4.307-15.538-4.307-30.615 0-15.078 4.307-30.616l-40.999-34.615 28.461-48.383 50.307 18q11-10.308 23.808-16.962t27.115-10.885l8.923-53.076h56.921l8.539 53.076q14.308 4.231 27.115 11.193 12.808 6.961 23.808 17.884l50.307-19.23 28.461 49.614-41 34.615q4.308 14.429 4.308 30.061t-4.308 29.939l41.385 33.846-28.461 48.383-50.692-17.615q-11.384 10.308-24 17.154t-26.923 11.077l-8.539 53.076zm28.11-100.383q31.428 0 53.774-22.38t22.346-53.807-22.38-53.774-53.808-22.346-53.773 22.38-22.346 53.808q0 31.427 22.38 53.773t53.807 22.346M400-552.307q33 0 56.5-23.5t23.5-56.5-23.5-56.5-56.5-23.5-56.5 23.5-23.5 56.5 23.5 56.5 56.5 23.5m12 304.614"></path></symbol><symbol viewBox="0 -960 960 960" id="progress_activity" xmlns="http://www.w3.org/2000/svg"><path d="M332.55-129.9q-69.29-29.9-121.02-81.63T129.9-332.55q-29.9-69.3-29.9-147.8 0-78.51 29.96-147.5 29.96-69 81.58-120.61 51.61-51.62 121.04-81.58T480-860q12.75 0 21.37 8.63 8.63 8.63 8.63 21.38 0 12.76-8.63 21.37Q492.75-800 480-800q-133 0-226.5 93.5T160-480t93.5 226.5T480-160t226.5-93.5T800-480q0-12.77 8.63-21.38 8.63-8.62 21.38-8.62 12.76 0 21.37 8.63Q860-492.75 860-480q0 77.99-29.96 147.42t-81.58 121.04q-51.61 51.62-120.61 81.58Q558.86-100 480.35-100q-78.5 0-147.8-29.9"></path></symbol><symbol viewBox="0 -960 960 960" id="react" xmlns="http://www.w3.org/2000/svg"><path d="M480-80q-83 0-156-31.5T197-197t-85.5-127T80-480t31.5-156T197-763t127-85.5T480-880q43 0 83 8.5t77 24.5v90q-35-20-75.5-31.5T480-800q-133 0-226.5 93.5T160-480t93.5 226.5T480-160t226.5-93.5T800-480q0-32-6.5-62T776-600h86q9 29 13.5 58.5T880-480q0 83-31.5 156T763-197t-127 85.5T480-80m320-600v-80h-80v-80h80v-80h80v80h80v80h-80v80zM620-520q25 0 42.5-17.5T680-580t-17.5-42.5T620-640t-42.5 17.5T560-580t17.5 42.5T620-520m-280 0q25 0 42.5-17.5T400-580t-17.5-42.5T340-640t-42.5 17.5T280-580t17.5 42.5T340-520m140 260q68 0 123.5-38.5T684-400H276q25 63 80.5 101.5T480-260"></path></symbol><symbol viewBox="0 -960 960 960" id="refresh" xmlns="http://www.w3.org/2000/svg"><path d="M480-160q-134 0-227-93t-93-227 93-227 227-93q69 0 132 28.5T720-690v-110h80v280H520v-80h168q-32-56-87.5-88T480-720q-100 0-170 70t-70 170 70 170 170 70q77 0 139-44t87-116h84q-28 106-114 173t-196 67"></path></symbol><symbol viewBox="0 -960 960 960" id="reply" xmlns="http://www.w3.org/2000/svg"><path d="M440-400h80v-120h120v-80H520v-120h-80v120H320v80h120zM80-80v-720q0-33 23.5-56.5T160-880h640q33 0 56.5 23.5T880-800v480q0 33-23.5 56.5T800-240H240zm126-240h594v-480H160v525zm-46 0v-480z"></path></symbol><symbol viewBox="0 -960 960 960" id="resume" xmlns="http://www.w3.org/2000/svg"><path d="M240-240v-480h80v480zm160 0 400-240-400-240z"></path></symbol><symbol viewBox="0 -960 960 960" id="review" xmlns="http://www.w3.org/2000/svg"><path d="m363-390 117-71 117 71-31-133 104-90-137-11-53-126-53 126-137 11 104 90zM80-80v-720q0-33 23.5-56.5T160-880h640q33 0 56.5 23.5T880-800v480q0 33-23.5 56.5T800-240H240zm126-240h594v-480H160v525zm-46 0v-480z"></path></symbol><symbol viewBox="0 -960 960 960" id="save" xmlns="http://www.w3.org/2000/svg"><path d="M212.31-140Q182-140 161-161t-21-51.31v-535.38Q140-778 161-799t51.31-21h429.3q14.47 0 27.81 5.62 13.35 5.61 23.19 15.46l106.31 106.31q9.85 9.84 15.46 23.19 5.62 13.34 5.62 27.81v429.3Q820-182 799-161t-51.31 21zM760-646 646-760H212.31q-5.39 0-8.85 3.46t-3.46 8.85v535.38q0 5.39 3.46 8.85t8.85 3.46h535.38q5.39 0 8.85-3.46t3.46-8.85zM550.77-298.46Q580-327.69 580-369.23T550.77-440 480-469.23 409.23-440 380-369.23t29.23 70.77T480-269.23t70.77-29.23M291.54-564.62h256.15q15.46 0 25.81-10.34 10.34-10.35 10.34-25.81v-67.69q0-15.46-10.34-25.81-10.35-10.34-25.81-10.34H291.54q-15.46 0-25.81 10.34-10.34 10.35-10.34 25.81v67.69q0 15.46 10.34 25.81 10.35 10.34 25.81 10.34M200-646v446-560z"></path></symbol><symbol viewBox="0 0 24 24" id="search" xmlns="http://www.w3.org/2000/svg"><path d="M9.5 3A6.5 6.5 0 0 1 16 9.5c0 1.61-.59 3.09-1.56 4.23l.27.27h.79l5 5-1.5 1.5-5-5v-.79l-.27-.27A6.52 6.52 0 0 1 9.5 16 6.5 6.5 0 0 1 3 9.5 6.5 6.5 0 0 1 9.5 3m0 2C7 5 5 7 5 9.5S7 14 9.5 14 14 12 14 9.5 12 5 9.5 5"></path></symbol><symbol viewBox="0 0 24 24" id="shuffle" xmlns="http://www.w3.org/2000/svg"><path d="m17 3 5.25 4.5L17 12l5.25 4.5L17 21v-3h-2.74l-2.82-2.82 2.12-2.12L15.5 15H17V9h-1.5l-9 9H2v-3h3.26l9-9H17zM2 6h4.5l2.82 2.82-2.12 2.12L5.26 9H2z"></path></symbol><symbol viewBox="0 0 24 24" id="sort-variant" xmlns="http://www.w3.org/2000/svg"><path d="M3 13h12v-2H3m0-5v2h18V6M3 18h6v-2H3z"></path></symbol><symbol viewBox="0 0 24 24" id="sort-variant-white" xmlns="http://www.w3.org/2000/svg"><path d="M3 13h12v-2H3m0-5v2h18V6M3 18h6v-2H3z"></path></symbol><symbol viewBox="0 -960 960 960" id="sos" xmlns="http://www.w3.org/2000/svg"><path d="M420-280q-33 0-56.5-23.5T340-360v-240q0-33 23.5-56.5T420-680h120q33 0 56.5 23.5T620-600v240q0 33-23.5 56.5T540-280zm-380 0v-80h160v-80h-80q-33 0-56.5-23.5T40-520v-80q0-33 23.5-56.5T120-680h160v80H120v80h80q33 0 56.5 23.5T280-440v80q0 33-23.5 56.5T200-280zm640 0v-80h160v-80h-80q-33 0-56.5-23.5T680-520v-80q0-33 23.5-56.5T760-680h160v80H760v80h80q33 0 56.5 23.5T920-440v80q0 33-23.5 56.5T840-280zm-260-80h120v-240H420z"></path></symbol><symbol viewBox="0 -960 960 960" id="star" xmlns="http://www.w3.org/2000/svg"><path d="m480-292.46-155.61 93.84q-8.7 5.08-17.43 4.27t-15.8-5.88q-7.08-5.08-10.93-13.27-3.84-8.19-1.23-18.12l41.31-176.69-137.38-118.92q-7.7-6.69-9.81-15.5-2.12-8.81 1.11-17.12 3.23-8.3 9.31-13.57t16.62-6.89l181.3-15.84L451.85-763q3.84-9.31 11.65-13.77t16.5-4.46 16.5 4.46T508.15-763l70.39 166.85 181.3 15.84q10.54 1.62 16.62 6.89t9.31 13.57q3.23 8.31 1.11 17.12-2.11 8.81-9.81 15.5L639.69-408.31 681-231.62q2.61 9.93-1.23 18.12-3.85 8.19-10.93 13.27-7.07 5.07-15.8 5.88t-17.43-4.27z"></path></symbol><symbol fill="none" stroke="currentColor" stroke-linecap="round" stroke-linejoin="round" stroke-width="2" class="lucide lucide-stone-icon lucide-stone" viewBox="0 0 24 24" id="stone" xmlns="http://www.w3.org/2000/svg"><path d="M11.264 2.205A4 4 0 0 0 6.42 4.211l-4 8a4 4 0 0 0 1.359 5.117l6 4a4 4 0 0 0 4.438 0l6-4a4 4 0 0 0 1.576-4.592l-2-6a4 4 0 0 0-2.53-2.53z"></path><path d="M11.99 22 14 12l7.822 3.184M14 12 8.47 2.302"></path></symbol><symbol viewBox="0 -960 960 960" id="stop" xmlns="http://www.w3.org/2000/svg"><path d="M240-240v-480h480v480z"></path></symbol><symbol viewBox="0 -960 960 960" id="text_fields" xmlns="http://www.w3.org/2000/svg"><path d="M280-160v-520H80v-120h520v120H400v520zm360 0v-320H520v-120h360v120H760v320z"></path></symbol><symbol viewBox="0 0 24 24" id="theme-light-dark" xmlns="http://www.w3.org/2000/svg"><path d="M7.5 2c-1.79 1.15-3 3.18-3 5.5s1.21 4.35 3.03 5.5C4.46 13 2 10.54 2 7.5A5.5 5.5 0 0 1 7.5 2m11.57 1.5 1.43 1.43L4.93 20.5 3.5 19.07zm-6.18 2.43L11.41 5 9.97 6l.42-1.7L9 3.24l1.75-.12.58-1.65L12 3.1l1.73.03-1.35 1.13zm-3.3 3.61-1.16-.73-1.12.78.34-1.32-1.09-.83 1.36-.09.45-1.29.51 1.27 1.36.03-1.05.87zM19 13.5a5.5 5.5 0 0 1-5.5 5.5c-1.22 0-2.35-.4-3.26-1.07l7.69-7.69c.67.91 1.07 2.04 1.07 3.26m-4.4 6.58 2.77-1.15-.24 3.35zm4.33-2.7 1.15-2.77 2.2 2.54zm1.15-4.96-1.14-2.78 3.34.24zM9.63 18.93l2.77 1.15-2.53 2.19z"></path></symbol><symbol viewBox="0 -960 960 960" id="ticket" xmlns="http://www.w3.org/2000/svg"><path d="M160-160q-33 0-56.5-23.5T80-240v-135q0-11 7-19t18-10q24-8 39.5-29t15.5-47-15.5-47-39.5-29q-11-2-18-10t-7-19v-135q0-33 23.5-56.5T160-800h640q33 0 56.5 23.5T880-720v135q0 11-7 19t-18 10q-24 8-39.5 29T800-480t15.5 47 39.5 29q11 2 18 10t7 19v135q0 33-23.5 56.5T800-160z"></path><path d="M332.472-290.32c-17.915 0-30.648-7.98-38.199-23.97-7.553-15.99-5.708-30.82 5.532-44.53l116.97-142.25v-126.46H395.7q-8.958 0-15.017-6.06c-4.04-4.04-6.06-9.05-6.06-15.02 0-5.98 2.02-10.96 6.06-15.02 4.04-4.02 9.044-6.05 15.017-6.05h168.605c5.97 0 10.975 2.03 15.014 6.05 4.04 4.06 6.061 9.04 6.061 15.02 0 5.97-2.021 10.98-6.061 15.02q-6.059 6.06-15.014 6.06h-21.076v126.46l116.968 142.25c11.238 13.71 13.084 28.54 5.532 44.53-7.549 15.99-20.284 23.97-38.198 23.97zm42.151-63.23H585.38l-71.658-84.31h-67.441z" style="fill:#fff"></path></symbol><symbol viewBox="0 -960 960 960" id="trending-up" xmlns="http://www.w3.org/2000/svg"><path d="m136-240-56-56 296-298 160 160 208-206H640v-80h240v240h-80v-104L536-320 376-480z"></path></symbol><symbol fill="none" stroke="currentColor" stroke-linecap="round" stroke-linejoin="round" stroke-width="2" class="lucide lucide-triangle-alert-icon lucide-triangle-alert" viewBox="0 0 24 24" id="triangle-alert" xmlns="http://www.w3.org/2000/svg"><path d="m21.73 18-8-14a2 2 0 0 0-3.48 0l-8 14A2 2 0 0 0 4 21h16a2 2 0 0 0 1.73-3M12 9v4M12 17h.01"></path></symbol><symbol viewBox="0 -960 960 960" id="tts" xmlns="http://www.w3.org/2000/svg"><path d="M160-80q-33 0-56.5-23.5T80-160v-640q0-33 23.5-56.5T160-880h326l-80 80H160v640h440v-80h80v80q0 33-23.5 56.5T600-80zm80-160v-80h280v80zm0-120v-80h200v80zm360 0L440-520H320v-200h120l160-160zm80-122v-276q36 21 58 57t22 81-22 81-58 57m0 172v-84q70-25 115-86.5T840-620t-45-139.5T680-846v-84q104 27 172 112.5T920-620t-68 197.5T680-310"></path></symbol><symbol viewBox="0 -960 960 960" id="unarchive" xmlns="http://www.w3.org/2000/svg"><path d="M480-552.46 349.85-422.31 387-385.16l67-67v160h52v-160l67 67 37.15-37.15zm-264-75.39v399.54q0 5.39 3.46 8.85t8.85 3.46h503.38q5.39 0 8.85-3.46t3.46-8.85v-399.54zM231.39-164q-26.63 0-47.01-20.38T164-231.39v-439.38q0-12.84 4.87-24.5 4.86-11.65 14.59-21.5l60.16-60.92q9.84-9.85 21.05-14.08t23.79-4.23h382.31q12.58 0 23.98 4.23T716-777.69L776.54-716q9.73 9.85 14.59 21.69 4.87 11.85 4.87 24.7v438.22q0 26.63-20.38 47.01T728.61-164zm-9.77-515.84H738l-57.62-59.93q-1.92-1.92-4.42-3.08-2.5-1.15-5.19-1.15H288.85q-2.69 0-5.2 1.15-2.5 1.16-4.42 3.08zM480-421.92"></path></symbol><symbol fill="none" stroke="currentColor" stroke-linecap="round" stroke-linejoin="round" stroke-width="2" class="lucide lucide-user-icon lucide-user" viewBox="0 0 24 24" id="user" xmlns="http://www.w3.org/2000/svg"><path d="M19 21v-2a4 4 0 0 0-4-4H9a4 4 0 0 0-4 4v2"></path><circle cx="12" cy="7" r="4"></circle></symbol><symbol viewBox="0 -960 960 960" id="view_list" xmlns="http://www.w3.org/2000/svg"><path d="M360-240h440v-107H360zM160-613h120v-107H160zm0 187h120v-107H160zm0 186h120v-107H160zm200-186h440v-107H360zm0-187h440v-107H360zM160-160q-33 0-56.5-23.5T80-240v-480q0-33 23.5-56.5T160-800h640q33 0 56.5 23.5T880-720v480q0 33-23.5 56.5T800-160z"></path></symbol><symbol viewBox="0 0 24 24" id="vote" xmlns="http://www.w3.org/2000/svg"><path fill="none" d="M0 0h24v24H0z"></path><path d="M18 13h-.68l-2 2h1.91L19 17H5l1.78-2h2.05l-2-2H6l-3 3v4c0 1.1.89 2 1.99 2H19a2 2 0 0 0 2-2v-4zm-1-5.05-4.95 4.95-3.54-3.54 4.95-4.95zm-4.24-5.66L6.39 8.66a.996.996 0 0 0 0 1.41l4.95 4.95c.39.39 1.02.39 1.41 0l6.36-6.36a.996.996 0 0 0 0-1.41L14.16 2.3a.975.975 0 0 0-1.4-.01"></path></symbol><symbol viewBox="0 -960 960 960" id="zoom_out" xmlns="http://www.w3.org/2000/svg"><path d="M256-200h64q17 0 28.5 11.5T360-160t-11.5 28.5T320-120H160q-17 0-28.5-11.5T120-160v-160q0-17 11.5-28.5T160-360t28.5 11.5T200-320v64l96-96q11-11 28-11t28 11 11 28-11 28zm448 0-96-96q-11-11-11-28t11-28 28-11 28 11l96 96v-64q0-17 11.5-28.5T800-360t28.5 11.5T840-320v160q0 17-11.5 28.5T800-120H640q-17 0-28.5-11.5T600-160t11.5-28.5T640-200zM200-704v64q0 17-11.5 28.5T160-600t-28.5-11.5T120-640v-160q0-17 11.5-28.5T160-840h160q17 0 28.5 11.5T360-800t-11.5 28.5T320-760h-64l96 96q11 11 11 28t-11 28-28 11-28-11zm560 0-96 96q-11 11-28 11t-28-11-11-28 11-28l96-96h-64q-17 0-28.5-11.5T600-800t11.5-28.5T640-840h160q17 0 28.5 11.5T840-800v160q0 17-11.5 28.5T800-600t-28.5-11.5T760-640z"></path></symbol></svg></div><script>if(typeof CSSLayerBlockRule==="undefined"){var l=document.createElement("link");l.rel="stylesheet";l.href="/legacy.min.css";document.head.appendChild(l)}</script><svg class="hide"><defs><linearGradient id="color-1"><stop offset="0" style="stop-color: #2193b0"></stop><stop offset="1" style="stop-color: #6dd5ed"></stop></linearGradient><linearGradient id="color-2"><stop style="stop-color: #f12711;" offset="0"></stop><stop style="stop-color: #f5af19;" offset="1"></stop></linearGradient><linearGradient id="color-3"><stop style="stop-color: #7f8c8d" offset="0"></stop><stop style="stop-color: #bdc3c7" offset="1"></stop></linearGradient><linearGradient id="color-4"><stop style="stop-color: #f12711" offset="0"></stop><stop style="stop-color: #f5af19" offset="0.5"></stop><stop style="stop-color: #7f8c8d" offset="0.5"></stop><stop style="stop-color: #bdc3c7" offset="1"></stop></linearGradient></defs></svg><div id="__next"><style>
:root {
  --bprogress-color: var(--primary);
  --bprogress-height: 3px;
  --bprogress-spinner-size: 18px;
  --bprogress-spinner-animation-duration: 400ms;
  --bprogress-spinner-border-size: 2px;
  --bprogress-box-shadow: 0 0 10px var(--primary), 0 0 5px var(--primary);
  --bprogress-z-index: 99999;
  --bprogress-spinner-top: 15px;
  --bprogress-spinner-bottom: auto;
  --bprogress-spinner-right: 15px;
  --bprogress-spinner-left: auto;
}

.bprogress {
  width: 0;
  height: 0;
  pointer-events: none;
  z-index: var(--bprogress-z-index);
}

.bprogress .bar {
  background: var(--bprogress-color);
  position: fixed;
  z-index: var(--bprogress-z-index);
  top: 0;
  left: 0;
  width: 100%;
  height: var(--bprogress-height);
}

/* Fancy blur effect */
.bprogress .peg {
  display: block;
  position: absolute;
  right: 0;
  width: 100px;
  height: 100%;
  box-shadow: var(--bprogress-box-shadow);
  opacity: 1.0;
  transform: rotate(3deg) translate(0px, -4px);
}

/* Remove these to get rid of the spinner */
.bprogress .spinner {
  display: block;
  position: fixed;
  z-index: var(--bprogress-z-index);
  top: var(--bprogress-spinner-top);
  bottom: var(--bprogress-spinner-bottom);
  right: var(--bprogress-spinner-right);
  left: var(--bprogress-spinner-left);
}

.bprogress .spinner-icon {
  width: var(--bprogress-spinner-size);
  height: var(--bprogress-spinner-size);
  box-sizing: border-box;
  border: solid var(--bprogress-spinner-border-size) transparent;
  border-top-color: var(--bprogress-color);
  border-left-color: var(--bprogress-color);
  border-radius: 50%;
  -webkit-animation: bprogress-spinner var(--bprogress-spinner-animation-duration) linear infinite;
  animation: bprogress-spinner var(--bprogress-spinner-animation-duration) linear infinite;
}

.bprogress-custom-parent {
  overflow: hidden;
  position: relative;
}

.bprogress-custom-parent .bprogress .spinner,
.bprogress-custom-parent .bprogress .bar {
  position: absolute;
}

.bprogress .indeterminate {
  position: fixed;
  top: 0;
  left: 0;
  width: 100%;
  height: var(--bprogress-height);
  overflow: hidden;
}

.bprogress .indeterminate .inc,
.bprogress .indeterminate .dec {
  position: absolute;
  top: 0;
  height: 100%;
  background-color: var(--bprogress-color);
}

.bprogress .indeterminate .inc {
  animation: bprogress-indeterminate-increase 2s infinite;
}

.bprogress .indeterminate .dec {
  animation: bprogress-indeterminate-decrease 2s 0.5s infinite;
}

@-webkit-keyframes bprogress-spinner {
  0%   { -webkit-transform: rotate(0deg); transform: rotate(0deg); }
  100% { -webkit-transform: rotate(360deg); transform: rotate(360deg); }
}

@keyframes bprogress-spinner {
  0%   { transform: rotate(0deg); }
  100% { transform: rotate(360deg); }
}

@keyframes bprogress-indeterminate-increase {
  from { left: -5%; width: 5%; }
  to { left: 130%; width: 100%; }
}

@keyframes bprogress-indeterminate-decrease {
  from { left: -80%; width: 80%; }
  to { left: 110%; width: 10%; }
}
</style><script>((e,t,n,r,o,a,i,s)=>{let l=document.documentElement,u=["light","dark"];function c(t){var n;(Array.isArray(e)?e:[e]).forEach(e=>{let n="class"===e,r=n&&a?o.map(e=>a[e]||e):o;n?(l.classList.remove(...r),l.classList.add(a&&a[t]?a[t]:t)):l.setAttribute(e,t)}),n=t,s&&u.includes(n)&&(l.style.colorScheme=n)}if(r)c(r);else try{let e=localStorage.getItem(t)||n,r=i&&"system"===e?window.matchMedia("(prefers-color-scheme: dark)").matches?"dark":"light":e;c(r)}catch(e){}})(["class","data-color-scheme"],"theme","light",null,["light","dark"],null,false,true)</script><nav class="fixed inset-x-0 top-0 z-40 border-b border-white/10 bg-navbar py-2 text-white shadow-md transition-transform duration-200 [body.hide-navbar:not(.popover-open)_&amp;]:-translate-y-14 font-sans antialiased" translate="no"><div class="mx-auto flex w-full max-w-7xl items-center px-2"><button type="button" tabindex="0" data-slot="button" class="group/button inline-flex cursor-pointer shrink-0 items-center justify-center rounded-lg bg-clip-padding text-sm font-medium whitespace-nowrap transition-all outline-none select-none focus-visible:ring-3 focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-3 aria-invalid:ring-destructive/20 dark:aria-invalid:border-destructive/50 dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 [&amp;_svg:not([class*='size-'])]:size-4 border aria-expanded:bg-muted aria-expanded:text-foreground dark:border-input dark:bg-input/30 dark:hover:bg-input/50 size-10 border-white/30 bg-transparent text-white/85 hover:bg-white/10 hover:text-white focus-visible:border-white/50 focus-visible:bg-white/10 focus-visible:text-white lg:w-10 mr-1 ml-0"><svg class="icon inline-flex shrink-0 size-6"><use href="#menu"></use></svg></button><div data-closed="" data-swipe-direction="right" role="presentation" aria-hidden="true" id="base-ui-_r_2_" data-slot="drawer-swipe-area" class="data-[swipe-direction=left]:inset-y-0 data-[swipe-direction=left]:right-0 data-[swipe-direction=left]:w-10 data-[swipe-direction=right]:inset-y-0 data-[swipe-direction=right]:left-0 data-[swipe-direction=right]:w-10 data-[swipe-direction=up]:inset-x-0 data-[swipe-direction=up]:bottom-0 data-[swipe-direction=up]:h-10 data-[swipe-direction=down]:inset-x-0 data-[swipe-direction=down]:top-0 data-[swipe-direction=down]:h-10 fixed inset-y-0 left-0 z-50 w-4!" style="touch-action: pan-y;"></div><a class="me-4 inline-flex items-center gap-2 whitespace-nowrap text-[18px] font-semibold text-white/90 hover:no-underline transition-colors hover:text-white" href="/en"><svg xmlns="http://www.w3.org/2000/svg" xml:space="preserve" fill="#fff" viewBox="0 0 512 512" class="icon size-[30px] shrink-0"><path d="M111.155 160.828c-3.042 4.341-5.697 9.139-7.873 14.218a87 87 0 0 0-1.948 4.925l-.062.172s-.267.813-.375 1.134a87 87 0 0 0-.542 1.653l-.086.287a154 154 0 0 1-.501 1.697l-.086.298a108 108 0 0 0-1.038 4.259l-.485 2.326c-.182.974-.363 1.95-.511 2.922-.308 1.987-.541 4.133-.715 6.602-.06.915-.12 1.86-.151 2.769a76 76 0 0 0-.002 5.276q.025.928.072 1.831l-.115.115.531 4.776q.097.868.217 1.743l.879 7.126h.455q.063.27.128.53c.209.865.42 1.727.656 2.565.187.716.395 1.452.606 2.17.119.456.247.83.344 1.094.449 1.476.925 2.83 1.447 4.121.226.635.544 1.48.956 2.374a77 77 0 0 0 5.191 10.225l11.25 18.624 1.598 2.697.879 1.413 1.045 1.751 4.34 7.197 9.06 14.983 32.641-51.534.36-.613 1.93-2.993 5.37-8.545 23.01-36.365a74 74 0 0 0 4.559-8.363 78 78 0 0 0 3.121-7.667c.33-.953.62-1.904.92-2.856l.04-.128a75.8 75.8 0 0 0 3.13-21.639V18zm303.865 38.938a12 12 0 0 0-.12-1.262c-.02-.223-.05-.506-.1-.83a45 45 0 0 0-.66-4.576c-.02-.126-.05-.25-.08-.373v-3.172l-.87-.856c-.06-.269-.12-.539-.19-.814-.17-.681-.34-1.338-.54-2.015-.2-.814-.42-1.606-.65-2.396l-.03-.124a97 97 0 0 0-1.19-3.733l-.04-.132c-.32-.892-.67-1.752-.98-2.519a79 79 0 0 0-8.65-16.134L300.86 17.94v136.058c0 8.219 1.24 16.233 3.69 23.818l.03.096c.26.794.53 1.588.82 2.382a63 63 0 0 0 3.17 7.486 75 75 0 0 0 3.88 6.878l57.74 91.177h11.22l22.53-37.818c8.76-14.581 12.6-31.262 11.08-48.251" style="fill-opacity:0.3"></path><path d="M442.17 461.082c-8.23 14.426-22.52 23.172-38.99 23.172H109.01c-16.444 0-31.031-8.746-38.997-23.172-7.859-14.318-7.079-31.651 1.534-45.08l72.424-113.98.377-.888 22.792-36.145 7.94 8.693c5.22 5.598 13.94 6.083 19.65 1.023l29.5-25.945 26.59 30.655c1.26 1.507 3.69 1.749 5.7 1.749 1.89 0 4.31-1.399 5.57-2.906l25.06-28.879.51-.619 29.53 25.945c5.68 5.06 14.56 4.575 19.62-1.023l8.13-8.639.88 1.427 21.4 33.911.51.753 72.94 114.868c8.61 13.429 9.12 30.385 1.5 45.08M116.627 242.893c-.404-.753-.781-1.372-1.157-2.018-.377-.754-.889-1.534-1.131-2.154a82 82 0 0 1-2.045-4.44c-.081-.162-.135-.35-.189-.512a18 18 0 0 1-.753-1.884 39 39 0 0 1-1.319-3.767c-.054-.162-.135-.324-.161-.485a65 65 0 0 1-.619-2.207c-.216-.753-.404-1.534-.592-2.314-.189-.754-.35-1.535-.512-2.315a87 87 0 0 1-.942-5.813s.027-.027 0-.027c-.215-1.965-.376-3.903-.43-5.867a67 67 0 0 1 0-4.656c.027-.808.08-1.642.134-2.45.135-1.911.323-3.821.619-5.732.135-.888.296-1.75.458-2.611l.403-1.938c.296-1.291.593-2.583.942-3.875.189-.619.377-1.265.565-1.911.27-.861.566-1.696.835-2.53a77 77 0 0 1 1.722-4.36c1.884-4.306 4.172-8.451 6.944-12.38L201.4 49.734l.059-.081v104.344c0 6.325-.89 12.542-2.669 18.49-.02.108-.05.188-.101.296-.25.861-.509 1.75-.839 2.584a56 56 0 0 1-2.23 5.732c-.161.323-.32.619-.49.942-.561 1.077-1.07 2.154-1.66 3.23a67 67 0 0 1-2.401 4.064l-22.899 36.333-5.33 8.478-2.05 3.176-.37.646-15.318 24.061-7.348 11.492-1.265 2.045-.242-.376-.431-.646-.726-1.131-3.607-6.055-1.076-1.804-.808-1.291-1.345-2.234-.296-.539-.216-.349-1.103-1.831zm204.243-53.558a63 63 0 0 1-3.36-6.029 53 53 0 0 1-2.69-6.378c-.27-.727-.54-1.454-.75-2.18-2.13-6.594-3.21-13.565-3.21-20.751V49.653l81.79 116.994c1.54 2.287 3.04 4.575 4.34 6.971 1.18 2.314 2.28 4.736 3.28 7.024.3.727.59 1.453.86 2.207.4 1.076.7 2.207 1.05 3.31.4 1.373.81 2.772 1.13 4.199.14.511.22.996.32 1.507.11.511.22 1.023.3 1.534.05.108.08.161.11.242 0 .027 0 .081.02.108v.134c.03.054.06.135.06.216.03.107.05.215.08.323.03.161.08.323.11.511a32 32 0 0 1 .4 3.149c.05.269.11.565.16.834.06.269.08.566.11.835.05.296.05.619.08.915.03.269.03.538.03.807 1.13 14.184-2.18 28.636-9.77 41.42l-19.62 32.942-19.11-30.035-6.97-11.142zm136.89 215.793-69.9-110.319 24.82-41.635c18.22-30.412 16.82-69.033-3.42-98.288L309.09 11.651c-1.91-2.503-5.06-4.171-8.35-4.171-1.02 0-2.02.269-3.04.511-4.33 1.265-7.1 5.06-7.1 9.635v136.371c0 16.579 4.57 32.539 13.18 45.969l28.24 44.569 5.41 8.558c-.35.243-.7.539-1 .862l-10.12 10.496-29.01-25.945c-2.77-2.53-6.33-3.822-10.26-3.552-3.79.242-7.21 2.045-9.63 4.817l-1.27 1.4-20.1 23.28-21.42-24.68c-2.53-2.772-5.95-4.575-9.75-4.817-3.93-.27-7.34 1.022-10.14 3.552l-29.5 25.945-9.74-10.496c-.3-.296-.57-.539-.89-.781l1.29-1.911 1.24-1.911 4.569-7.347.521-.754 26.07-41.285c8.75-13.538 13.17-29.39 13.17-45.969V17.626c0-4.575-2.64-8.37-6.98-9.635-.99-.242-2.01-.511-3.28-.511-3.15 0-6.32 1.668-8.1 4.171L102.82 154.886c-20.266 29.255-21.8 67.876-3.552 98.288l25.083 41.635L54.43 405.128c-12.919 20.239-13.673 44.919-2.019 65.939 11.519 20.912 32.781 33.453 56.599 33.453h294.17c23.95 0 45.08-12.541 56.62-33.453 11.39-21.127 10.61-45.942-2.04-65.939"></path><path d="M223.26 364.678c0 16.53-13.4 29.93-29.931 29.93-16.529 0-29.929-13.4-29.929-29.93 0-16.529 13.4-29.929 29.929-29.929 16.531 0 29.931 13.4 29.931 29.929m125.38.228c0 16.529-13.4 29.929-29.93 29.929s-29.93-13.4-29.93-29.929c0-16.53 13.4-29.93 29.93-29.93s29.93 13.4 29.93 29.93"></path></svg><span>WTR-LAB</span></a><div class="ml-auto flex lg:hidden"><button type="button" tabindex="0" data-slot="button" class="group/button inline-flex cursor-pointer shrink-0 items-center justify-center rounded-lg bg-clip-padding text-sm font-medium whitespace-nowrap transition-all outline-none select-none focus-visible:ring-3 focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-3 aria-invalid:ring-destructive/20 dark:aria-invalid:border-destructive/50 dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 [&amp;_svg:not([class*='size-'])]:size-4 border aria-expanded:bg-muted aria-expanded:text-foreground dark:border-input dark:bg-input/30 dark:hover:bg-input/50 size-10 ml-1 border-white/30 bg-transparent text-white/85 hover:bg-white/10 hover:text-white focus-visible:border-white/50 focus-visible:bg-white/10 focus-visible:text-white lg:w-10"><svg class="icon inline-flex shrink-0 size-6"><use href="#search"></use></svg></button></div><div class="absolute inset-y-0 left-0 right-0 z-50 items-center bg-navbar lg:static lg:z-auto lg:bg-transparent hidden lg:flex"><button type="button" tabindex="0" data-slot="button" class="group/button inline-flex cursor-pointer shrink-0 items-center justify-center rounded-lg bg-clip-padding text-sm font-medium whitespace-nowrap transition-all outline-none select-none focus-visible:border-ring focus-visible:ring-3 focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-3 aria-invalid:ring-destructive/20 dark:aria-invalid:border-destructive/50 dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 [&amp;_svg:not([class*='size-'])]:size-4 border border-border bg-card hover:bg-muted hover:text-foreground aria-expanded:bg-muted aria-expanded:text-foreground dark:border-input dark:bg-input/30 dark:hover:bg-input/50 size-10 lg:hidden ml-2 border-white/30! bg-transparent! text-white/85! hover:bg-white/10! hover:text-white! focus:bg-white/10! focus:text-white! lg:w-10"><svg class="icon inline-flex shrink-0 size-6 rotate-180"><use href="#arrow_forward"></use></svg></button><div class="relative flex-1 px-2 lg:px-0 lg:w-64"><form><input id="base-ui-_R_6kal6_" type="search" data-slot="input" placeholder="Search..." aria-label="Search" class="w-full min-w-0 rounded-lg border px-2.5 py-1 text-base transition-colors outline-none file:inline-flex file:h-6 file:border-0 file:bg-transparent file:text-sm file:font-medium file:text-foreground focus-visible:ring-ring/50 disabled:pointer-events-none disabled:cursor-not-allowed disabled:bg-input/50 disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-3 aria-invalid:ring-destructive/20 md:text-sm dark:bg-input/30 dark:disabled:bg-input/80 dark:aria-invalid:border-destructive/50 dark:aria-invalid:ring-destructive/40 pl-8 lg:pl-8 h-9 bg-input/20 border-white/20 text-white placeholder:text-white/50 focus-visible:ring-0 focus-visible:border-white/50" value=""></form><svg class="icon inline-flex shrink-0 absolute left-4 lg:left-2.5 top-1/2 -translate-y-1/2 size-5 text-white/50"><use href="#search"></use></svg></div></div><div class="hidden lg:flex items-center ml-auto"><a class="inline-flex items-center gap-1 rounded-md px-2 py-2 text-sm transition-colors hover:text-white/80 hover:no-underline [&amp;&gt;svg]:opacity-55 hover:[&amp;&gt;svg]:opacity-80 text-white/55" href="/en/library"><span class="inline-flex shrink-0"><svg class="icon inline-flex shrink-0 size-6"><use href="#book-marked"></use></svg></span><span>Library</span></a><a class="inline-flex items-center gap-1 rounded-md px-2 py-2 text-sm transition-colors hover:text-white/80 hover:no-underline [&amp;&gt;svg]:opacity-55 hover:[&amp;&gt;svg]:opacity-80 text-white/55" href="/en/novel-list"><span class="inline-flex shrink-0"><svg class="icon inline-flex shrink-0 size-6"><use href="#bookshelf-white"></use></svg></span><span>Novels</span></a><a class="inline-flex items-center gap-1 rounded-md px-2 py-2 text-sm transition-colors hover:text-white/80 hover:no-underline [&amp;&gt;svg]:opacity-55 hover:[&amp;&gt;svg]:opacity-80 text-white/55" href="/en/ranking/daily"><span class="inline-flex shrink-0"><svg class="icon inline-flex shrink-0 size-6"><use href="#sort-variant-white"></use></svg></span><span>Ranking</span></a><a class="inline-flex items-center gap-1 rounded-md px-2 py-2 text-sm transition-colors hover:text-white/80 hover:no-underline [&amp;&gt;svg]:opacity-55 hover:[&amp;&gt;svg]:opacity-80 text-white/55" href="/en/leaderboard"><span class="inline-flex shrink-0"><svg class="icon inline-flex shrink-0 size-6"><use href="#crown"></use></svg></span><span>Leaderboard</span></a></div><button type="button" tabindex="0" data-slot="button" class="group/button inline-flex cursor-pointer shrink-0 items-center justify-center rounded-lg bg-clip-padding text-sm font-medium whitespace-nowrap transition-all outline-none select-none focus-visible:ring-3 focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-3 aria-invalid:ring-destructive/20 dark:aria-invalid:border-destructive/50 dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 [&amp;_svg:not([class*='size-'])]:size-4 border aria-expanded:bg-muted aria-expanded:text-foreground dark:border-input dark:bg-input/30 dark:hover:bg-input/50 size-10 ml-1 border-white/30 bg-transparent text-white/85 hover:bg-white/10 hover:text-white focus-visible:border-white/50 focus-visible:bg-white/10 focus-visible:text-white lg:w-10 relative"><svg class="icon inline-flex shrink-0 size-6"><use href="#inbox"></use></svg><span class="absolute right-1 top-1 inline-block size-3 rounded-full border-2 border-white bg-destructive"></span></button><button type="button" tabindex="0" data-slot="dropdown-menu-trigger" aria-haspopup="menu" id="base-ui-_r_6_" class="group/button inline-flex cursor-pointer shrink-0 items-center justify-center rounded-lg bg-clip-padding text-sm font-medium whitespace-nowrap transition-all outline-none select-none focus-visible:ring-3 focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-3 aria-invalid:ring-destructive/20 dark:aria-invalid:border-destructive/50 dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 [&amp;_svg:not([class*='size-'])]:size-4 border aria-expanded:bg-muted aria-expanded:text-foreground dark:border-input dark:bg-input/30 dark:hover:bg-input/50 size-10 ml-1 border-white/30 bg-transparent text-white/85 hover:bg-white/10 hover:text-white focus-visible:border-white/50 focus-visible:bg-white/10 focus-visible:text-white lg:w-10" aria-expanded="false"><svg class="icon inline-flex shrink-0 size-6"><use href="#profile3"></use></svg></button></div></nav><div class="header-spacer"></div><div class="layout-body flex flex-col max-w-7xl container mx-auto font-sans antialiased"><div><div data-slot="card" data-size="default" data-variant="default" class="group/card flex flex-col gap-0 bg-card text-sm text-card-foreground has-[&gt;img:first-child]:pt-0 *:[img:first-child]:rounded-t-xl *:[img:last-child]:rounded-b-xl shadow rounded-md ring-1 ring-foreground/20 fix-size fix-edge mb-2 overflow-hidden"><div data-slot="card-content" class="p-2"><div class="flex items-center justify-between gap-2 mb-3"><div class="flex-1 min-w-0"><h1 class="text-base text-card-foreground sm:text-xl font-bold uppercase tracking-wide leading-tight wrap-break-word">Mortal Bones</h1><p class="text-xs sm:text-sm text-foreground mt-0.5 wrap-break-word">凡骨</p></div><div class="shrink-0 flex items-center gap-1"><button type="button" tabindex="0" data-slot="button" class="group/button inline-flex cursor-pointer items-center justify-center bg-clip-padding text-sm font-medium whitespace-nowrap transition-all outline-none select-none focus-visible:border-ring focus-visible:ring-3 focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-3 aria-invalid:ring-destructive/20 dark:aria-invalid:border-destructive/50 dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 [&amp;_svg:not([class*='size-'])]:size-4 border bg-card hover:text-foreground aria-expanded:bg-muted aria-expanded:text-foreground dark:border-input dark:bg-input/30 dark:hover:bg-input/50 size-8 h-9 w-9 shrink-0 p-0 border-amber-200 text-amber-600 hover:bg-amber-50 rounded-lg self-start"><svg class="icon inline-flex shrink-0 size-4"><use href="#alert-outline"></use></svg></button></div></div><div class="flex w-full rounded-lg border bg-card/50 overflow-hidden mb-3"><div class="shrink-0"><div class="image-wrap relative rounded-md h-full w-30 sm:w-40 rounded-none zoom group cursor-pointer"><div class="absolute inset-0 z-10 flex items-center justify-center bg-black/0 group-hover:bg-black/30 transition-colors duration-200 rounded-md"><svg class="icon inline-flex shrink-0 size-8 text-white opacity-0 group-hover:opacity-100 transition-opacity duration-200 drop-shadow-lg"><use href="#zoom_out"></use></svg></div><div class="absolute inset-0 overflow-hidden z-0"><img alt="" aria-hidden="true" loading="lazy" class="absolute inset-0 w-full h-full object-cover scale-110 blur-sm opacity-60 pointer-events-none select-none" src="/api/v2/img?src=s3%3A%2F%2Fwtrimg%2Fseries%2FP0DFglC6FSy273hoZ3e48A.jpg&amp;w=344"></div><img class="relative z-1" alt="Mortal Bones" loading="lazy" src="/api/v2/img?src=s3%3A%2F%2Fwtrimg%2Fseries%2FP0DFglC6FSy273hoZ3e48A.jpg&amp;w=344" style="object-fit: contain;"><span class="status corner st-0"></span></div></div><div class="flex flex-1 flex-col min-w-0 p-2 gap-2"><div class="grid grid-cols-2 sm:grid-cols-3 gap-1.5"><div class="flex flex-col rounded-lg bg-muted/40 border border-border/50 dark:border-white/15 px-2.5 py-1.5 items-center text-center gap-0.5"><span class="text-[9px] uppercase tracking-wide font-bold leading-none text-foreground/80" translate="no">Status</span><div class="flex items-center gap-1"><span class="w-1.5 h-1.5 rounded-full shrink-0 bg-emerald-500"></span><span class="text-xs font-semibold leading-none text-emerald-600 dark:text-emerald-400"><span>Ongoing</span></span></div></div><div class="flex flex-col rounded-lg bg-muted/40 border border-border/50 dark:border-white/15 px-2.5 py-1.5 items-center text-center gap-0.5"><span class="text-[9px] uppercase tracking-wide font-bold leading-none text-foreground/80" translate="no">Views</span><span class="text-xs font-semibold leading-none" translate="no">147K</span></div><div class="flex flex-col rounded-lg bg-muted/40 border border-border/50 dark:border-white/15 px-2.5 py-1.5 items-center text-center gap-0.5"><span class="text-[9px] uppercase tracking-wide font-bold leading-none text-foreground/80" translate="no">Chapters</span><span class="text-xs font-semibold leading-none" translate="no">3992</span></div><div class="flex flex-col rounded-lg bg-muted/40 border border-border/50 dark:border-white/15 px-2.5 py-1.5 items-center text-center gap-0.5"><span class="text-[9px] uppercase tracking-wide font-bold leading-none text-foreground/80" translate="no">Rating</span><div class="flex items-center gap-0.5"><span class="text-amber-400 text-xs leading-none">★</span><span class="text-xs font-semibold leading-none" translate="no">3.4 (10)</span></div></div><div class="flex flex-col rounded-lg bg-muted/40 border border-border/50 dark:border-white/15 px-2.5 py-1.5 items-center text-center gap-0.5"><span class="text-[9px] uppercase tracking-wide font-bold leading-none text-foreground/80" translate="no">Readers</span><span class="text-xs font-semibold leading-none">227</span></div><div class="flex flex-col rounded-lg bg-muted/40 border border-border/50 dark:border-white/15 px-2.5 py-1.5 items-center text-center gap-0.5"><span class="text-[9px] uppercase tracking-wide font-bold leading-none text-foreground/80" translate="no">Chars</span><span class="text-xs font-semibold leading-none" translate="no">7.12M</span></div></div><div class="flex flex-col gap-1.5 mt-auto"><div class="flex gap-1.5"><div class="inline-flex w-full flex-1 h-9 rounded-lg" translate="no"><button type="button" tabindex="0" data-slot="dropdown-menu-trigger" aria-haspopup="menu" id="base-ui-_r_10j_" class="group/button inline-flex cursor-pointer shrink-0 items-center justify-center bg-clip-padding font-medium whitespace-nowrap transition-all outline-none select-none focus-visible:border-ring focus-visible:ring-3 focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-3 aria-invalid:ring-destructive/20 dark:aria-invalid:border-destructive/50 dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 border border-border bg-card hover:bg-muted hover:text-foreground aria-expanded:bg-muted aria-expanded:text-foreground dark:border-input dark:bg-input/30 dark:hover:bg-input/50 gap-1 rounded-[min(var(--radius-md),12px)] px-2.5 text-[0.8rem] in-data-[slot=button-group]:rounded-lg has-data-[icon=inline-end]:pr-1.5 has-data-[icon=inline-start]:pl-1.5 [&amp;_svg:not([class*='size-'])]:size-3.5 w-full h-full" aria-expanded="false"><div class="flex items-center gap-1.5 justify-center w-full"><svg class="icon inline-flex shrink-0 size-6"><use href="#book-open-variant"></use></svg><span translate="no">Reading</span></div></button></div></div><a role="button" tabindex="0" data-slot="button" translate="no" rel="nofollow" class="group/button inline-flex cursor-pointer shrink-0 items-center justify-center bg-clip-padding whitespace-nowrap transition-all outline-none select-none focus-visible:border-ring focus-visible:ring-3 focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-3 aria-invalid:ring-destructive/20 dark:aria-invalid:border-destructive/50 dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 [&amp;_svg:not([class*='size-'])]:size-4 border border-primary bg-transparent text-primary hover:bg-primary/10 aria-expanded:bg-primary/10 gap-1.5 has-data-[icon=inline-end]:pr-2 has-data-[icon=inline-start]:pl-2 relative isolate h-9 rounded-lg font-semibold w-full text-sm text-center px-4" href="/en/novel/7336/mortal-bones/chapter-1747"><div class="absolute inset-0 overflow-hidden rounded-[inherit] -z-10"><div class="absolute inset-y-0 left-0 bg-primary/10 transition-all duration-500" style="width: 43.7625%;"></div></div><span class="absolute -top-2 left-1/2 -translate-x-1/2 text-[10px] font-normal tabular-nums bg-black/50 text-white px-1.5 py-px rounded-sm whitespace-nowrap max-sm:text-[9px] max-sm:px-1 max-sm:py-0.5 max-sm:-top-2 max-sm:px-2" translate="no">1747 / 3992</span><span>Continue Reading</span></a></div></div></div><div class="flex flex-wrap items-center gap-1 max-sm:flex-nowrap max-sm:overflow-x-auto max-sm:pb-2 scroll-thin"><span class="inline-flex items-center capitalize text-xs border rounded px-1.5 py-0.5 whitespace-nowrap transition-colors border-blue-400/50 text-blue-600 dark:text-blue-400 bg-blue-50/50 dark:bg-blue-950/20 hover:bg-blue-100/70 dark:hover:bg-blue-950/40 font-semibold">Male</span><span class="inline-flex items-center capitalize text-xs font-medium border rounded px-1.5 py-0.5 whitespace-nowrap transition-colors border-border/70 text-card-foreground">Cultivation</span><span class="inline-flex items-center capitalize text-xs font-medium border rounded py-0.5 whitespace-nowrap transition-colors border-border/70 text-card-foreground px-2">action</span><span class="inline-flex items-center capitalize text-xs font-medium border rounded py-0.5 whitespace-nowrap transition-colors border-border/70 text-card-foreground px-2">adventure</span><span class="inline-flex items-center capitalize text-xs font-medium border rounded py-0.5 whitespace-nowrap transition-colors border-border/70 text-card-foreground px-2">drama</span><span class="inline-flex items-center capitalize text-xs font-medium border rounded py-0.5 whitespace-nowrap transition-colors border-border/70 text-card-foreground px-2">fantasy</span><span class="inline-flex items-center capitalize text-xs font-medium border rounded py-0.5 whitespace-nowrap transition-colors border-border/70 text-card-foreground px-2">xianxia</span></div></div></div><div class="unlock-progress-card fix-edge fix-size"><div class="upc-header"><div class="upc-label"><svg class="icon inline-flex shrink-0 size-6 icon-sm"><use href="#ticket" class="svg-ticket"></use></svg><span>AI-Unlock Progress</span></div><div class="upc-counter"><span class="upc-unlocked">3989</span><span class="upc-sep">/</span><span class="upc-total">3992</span></div></div><div class="upc-track"><div class="upc-fill is-complete" style="width: 100%;"></div></div><div class="upc-footer"><span class="upc-status"><span>3 chapters locked</span></span><button class="upc-batch-btn" translate="no"><svg class="icon inline-flex shrink-0 size-6 icon-sm"><use href="#ticket" class="svg-ticket"></use></svg><span>Batch Unlock</span></button></div></div><div data-slot="card" data-size="default" data-variant="default" class="group/card flex flex-col gap-0 bg-card text-sm text-card-foreground has-[&gt;img:first-child]:pt-0 *:[img:first-child]:rounded-t-xl *:[img:last-child]:rounded-b-xl shadow rounded-md ring-1 ring-foreground/20 fix-size fix-edge"><div data-slot="card-content" class="p-2 chapter-details"><div data-orientation="horizontal" data-activation-direction="none" data-slot="tabs" class="group/tabs flex gap-2 data-horizontal:flex-col new-pills"><div data-orientation="horizontal" data-activation-direction="none" role="tablist" data-slot="tabs-list" data-variant="wrap" class="group/tabs-list items-center justify-center rounded-lg text-muted-foreground group-data-vertical/tabs:h-fit group-data-vertical/tabs:flex-col data-[variant=line]:rounded-none bg-border h-auto group-data-horizontal/tabs:h-auto grid w-full grid-cols-2 sm:grid-cols-4 gap-1 p-1"><button type="button" data-active="" data-orientation="horizontal" aria-disabled="false" tabindex="0" role="tab" aria-selected="true" id="base-ui-_r_10n_" data-composite-item-active="" data-slot="tabs-trigger" class="relative inline-flex flex-1 cursor-pointer items-center justify-center gap-1.5 rounded-md border border-transparent px-1.5 text-sm font-medium whitespace-nowrap text-foreground/60 transition-all group-data-vertical/tabs:w-full group-data-vertical/tabs:justify-start hover:text-foreground focus-visible:border-ring focus-visible:ring-[3px] focus-visible:ring-ring/50 focus-visible:outline-1 focus-visible:outline-ring disabled:pointer-events-none disabled:opacity-50 has-data-[icon=inline-end]:pr-1 has-data-[icon=inline-start]:pl-1 aria-disabled:pointer-events-none aria-disabled:opacity-50 dark:text-muted-foreground dark:hover:text-foreground group-data-[variant=default]/tabs-list:data-active:shadow-sm group-data-[variant=line]/tabs-list:data-active:shadow-none [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 [&amp;_svg:not([class*='size-'])]:size-4 group-data-[variant=line]/tabs-list:bg-transparent group-data-[variant=line]/tabs-list:data-active:bg-transparent dark:group-data-[variant=line]/tabs-list:data-active:border-transparent dark:group-data-[variant=line]/tabs-list:data-active:bg-transparent data-active:bg-background data-active:text-foreground dark:data-active:border-input dark:data-active:bg-input/30 dark:data-active:text-foreground after:absolute after:bg-foreground after:opacity-0 after:transition-opacity group-data-horizontal/tabs:after:inset-x-0 group-data-horizontal/tabs:after:bottom-[-5px] group-data-horizontal/tabs:after:h-0.5 group-data-vertical/tabs:after:inset-y-0 group-data-vertical/tabs:after:-right-1 group-data-vertical/tabs:after:w-0.5 group-data-[variant=line]/tabs-list:data-active:after:opacity-100 py-2 h-10" aria-controls="base-ui-_r_10r_">About</button><button type="button" data-orientation="horizontal" aria-disabled="false" tabindex="-1" role="tab" aria-selected="false" id="base-ui-_r_10o_" data-slot="tabs-trigger" class="relative inline-flex flex-1 cursor-pointer items-center justify-center gap-1.5 rounded-md border border-transparent px-1.5 text-sm font-medium whitespace-nowrap text-foreground/60 transition-all group-data-vertical/tabs:w-full group-data-vertical/tabs:justify-start hover:text-foreground focus-visible:border-ring focus-visible:ring-[3px] focus-visible:ring-ring/50 focus-visible:outline-1 focus-visible:outline-ring disabled:pointer-events-none disabled:opacity-50 has-data-[icon=inline-end]:pr-1 has-data-[icon=inline-start]:pl-1 aria-disabled:pointer-events-none aria-disabled:opacity-50 dark:text-muted-foreground dark:hover:text-foreground group-data-[variant=default]/tabs-list:data-active:shadow-sm group-data-[variant=line]/tabs-list:data-active:shadow-none [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 [&amp;_svg:not([class*='size-'])]:size-4 group-data-[variant=line]/tabs-list:bg-transparent group-data-[variant=line]/tabs-list:data-active:bg-transparent dark:group-data-[variant=line]/tabs-list:data-active:border-transparent dark:group-data-[variant=line]/tabs-list:data-active:bg-transparent data-active:bg-background data-active:text-foreground dark:data-active:border-input dark:data-active:bg-input/30 dark:data-active:text-foreground after:absolute after:bg-foreground after:opacity-0 after:transition-opacity group-data-horizontal/tabs:after:inset-x-0 group-data-horizontal/tabs:after:bottom-[-5px] group-data-horizontal/tabs:after:h-0.5 group-data-vertical/tabs:after:inset-y-0 group-data-vertical/tabs:after:-right-1 group-data-vertical/tabs:after:w-0.5 group-data-[variant=line]/tabs-list:data-active:after:opacity-100 py-2 h-10">Table of Contents</button><button type="button" data-orientation="horizontal" aria-disabled="false" tabindex="-1" role="tab" aria-selected="false" id="base-ui-_r_10p_" data-slot="tabs-trigger" class="relative inline-flex flex-1 cursor-pointer items-center justify-center gap-1.5 rounded-md border border-transparent px-1.5 text-sm font-medium whitespace-nowrap text-foreground/60 transition-all group-data-vertical/tabs:w-full group-data-vertical/tabs:justify-start hover:text-foreground focus-visible:border-ring focus-visible:ring-[3px] focus-visible:ring-ring/50 focus-visible:outline-1 focus-visible:outline-ring disabled:pointer-events-none disabled:opacity-50 has-data-[icon=inline-end]:pr-1 has-data-[icon=inline-start]:pl-1 aria-disabled:pointer-events-none aria-disabled:opacity-50 dark:text-muted-foreground dark:hover:text-foreground group-data-[variant=default]/tabs-list:data-active:shadow-sm group-data-[variant=line]/tabs-list:data-active:shadow-none [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 [&amp;_svg:not([class*='size-'])]:size-4 group-data-[variant=line]/tabs-list:bg-transparent group-data-[variant=line]/tabs-list:data-active:bg-transparent dark:group-data-[variant=line]/tabs-list:data-active:border-transparent dark:group-data-[variant=line]/tabs-list:data-active:bg-transparent data-active:bg-background data-active:text-foreground dark:data-active:border-input dark:data-active:bg-input/30 dark:data-active:text-foreground after:absolute after:bg-foreground after:opacity-0 after:transition-opacity group-data-horizontal/tabs:after:inset-x-0 group-data-horizontal/tabs:after:bottom-[-5px] group-data-horizontal/tabs:after:h-0.5 group-data-vertical/tabs:after:inset-y-0 group-data-vertical/tabs:after:-right-1 group-data-vertical/tabs:after:w-0.5 group-data-[variant=line]/tabs-list:data-active:after:opacity-100 py-2 h-10" aria-controls="base-ui-_r_110_">Reviews</button><button type="button" data-orientation="horizontal" aria-disabled="false" tabindex="-1" role="tab" aria-selected="false" id="base-ui-_r_10q_" data-slot="tabs-trigger" class="relative inline-flex flex-1 cursor-pointer items-center justify-center gap-1.5 rounded-md border border-transparent px-1.5 text-sm font-medium whitespace-nowrap text-foreground/60 transition-all group-data-vertical/tabs:w-full group-data-vertical/tabs:justify-start hover:text-foreground focus-visible:border-ring focus-visible:ring-[3px] focus-visible:ring-ring/50 focus-visible:outline-1 focus-visible:outline-ring disabled:pointer-events-none disabled:opacity-50 has-data-[icon=inline-end]:pr-1 has-data-[icon=inline-start]:pl-1 aria-disabled:pointer-events-none aria-disabled:opacity-50 dark:text-muted-foreground dark:hover:text-foreground group-data-[variant=default]/tabs-list:data-active:shadow-sm group-data-[variant=line]/tabs-list:data-active:shadow-none [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 [&amp;_svg:not([class*='size-'])]:size-4 group-data-[variant=line]/tabs-list:bg-transparent group-data-[variant=line]/tabs-list:data-active:bg-transparent dark:group-data-[variant=line]/tabs-list:data-active:border-transparent dark:group-data-[variant=line]/tabs-list:data-active:bg-transparent data-active:bg-background data-active:text-foreground dark:data-active:border-input dark:data-active:bg-input/30 dark:data-active:text-foreground after:absolute after:bg-foreground after:opacity-0 after:transition-opacity group-data-horizontal/tabs:after:inset-x-0 group-data-horizontal/tabs:after:bottom-[-5px] group-data-horizontal/tabs:after:h-0.5 group-data-vertical/tabs:after:inset-y-0 group-data-vertical/tabs:after:-right-1 group-data-vertical/tabs:after:w-0.5 group-data-[variant=line]/tabs-list:data-active:after:opacity-100 py-2 h-10">Recommendations</button></div><div data-orientation="horizontal" data-activation-direction="none" id="base-ui-_r_10r_" role="tabpanel" tabindex="0" data-index="0" data-slot="tabs-content" class="flex-1 text-sm outline-none" aria-labelledby="base-ui-_r_10n_"><div class="flex items-center justify-between gap-2 mt-4 mb-2 pl-2.5 border-l-2 border-l-primary"><span class="flex-1 text-base font-semibold text-card-foreground">Novel Summary</span></div><div class="desc-wrap text-foreground"><span class="description show"><p>There are four grades of spirit bones in the world. The first grade is the Heavenly Spirit Bone. The second grade is the Golden Spirit Bone. The third grade is the Profound Spirit Bone. The fourth grade is the White Spirit Bone. The rest are all Mortal Bones, with no chance of cultivation. Xu Taiping, a mere Mortal Bone, vows to prove to this cultivation world that Mortal Bones can also slay demons, Mortal Bones can also exorcise evil, Mortal Bones can also ascend to immortality!</p></span><div class="flex items-center justify-between mb-2"><div class="swap">Show less</div></div><span></span></div><div class="flex items-center justify-between gap-2 mt-4 mb-2 pl-2.5 border-l-2 border-l-primary"><span class="flex-1 text-base font-semibold text-card-foreground">Details</span><div class="flex gap-1"><button type="button" tabindex="0" data-slot="infotip-trigger" data-base-ui-click-trigger="" id="base-ui-_r_10t_" aria-haspopup="dialog" aria-expanded="false" class="group/button inline-flex cursor-pointer shrink-0 items-center justify-center bg-clip-padding font-medium whitespace-nowrap transition-all outline-none select-none focus-visible:border-ring focus-visible:ring-3 focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-3 aria-invalid:ring-destructive/20 dark:aria-invalid:border-destructive/50 dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 border border-border bg-card hover:bg-muted hover:text-foreground aria-expanded:bg-muted aria-expanded:text-foreground dark:border-input dark:bg-input/30 dark:hover:bg-input/50 h-7 gap-1 rounded-[min(var(--radius-md),12px)] px-2.5 text-[0.8rem] in-data-[slot=button-group]:rounded-lg has-data-[icon=inline-end]:pr-1.5 has-data-[icon=inline-start]:pl-1.5 [&amp;_svg:not([class*='size-'])]:size-3.5 ml-2">Edit</button><button type="button" tabindex="0" data-slot="button" class="group/button inline-flex cursor-pointer shrink-0 items-center justify-center bg-clip-padding font-medium whitespace-nowrap transition-all outline-none select-none focus-visible:border-ring focus-visible:ring-3 focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-3 aria-invalid:ring-destructive/20 dark:aria-invalid:border-destructive/50 dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 border border-border bg-card hover:bg-muted hover:text-foreground aria-expanded:bg-muted aria-expanded:text-foreground dark:border-input dark:bg-input/30 dark:hover:bg-input/50 h-7 gap-1 rounded-[min(var(--radius-md),12px)] px-2.5 text-[0.8rem] in-data-[slot=button-group]:rounded-lg has-data-[icon=inline-end]:pr-1.5 has-data-[icon=inline-start]:pl-1.5 [&amp;_svg:not([class*='size-'])]:size-3.5 ml-2">History</button></div></div><div class="rounded-lg overflow-hidden border border-border mb-0"><div class="grid grid-cols-[108px_1fr] items-center gap-x-2.5 px-3.5 py-[9px] border-b border-border last:border-b-0 even:bg-black/[0.016] dark:even:bg-white/[0.022] max-[400px]:grid-cols-1 max-[400px]:gap-0.5 max-[400px]:px-3 max-[400px]:py-2"><span class="text-[10.5px] font-bold uppercase tracking-[0.07em] whitespace-nowrap pt-px leading-6 text-muted-foreground">Titles</span><div class="text-[13px] leading-relaxed wrap-break-word [&amp;_a]:no-underline [&amp;_a]:font-semibold [&amp;_a:hover]:text-primary"><ul class="m-0 pl-0 list-none [&amp;_li]:py-px [&amp;_li]:text-[13px]"><li>Mortal Bones</li><li>凡骨</li></ul></div></div><div class="grid grid-cols-[108px_1fr] items-center gap-x-2.5 px-3.5 py-[9px] border-b border-border last:border-b-0 even:bg-black/[0.016] dark:even:bg-white/[0.022] max-[400px]:grid-cols-1 max-[400px]:gap-0.5 max-[400px]:px-3 max-[400px]:py-2"><span class="text-[10.5px] font-bold uppercase tracking-[0.07em] whitespace-nowrap pt-px leading-6 text-muted-foreground">Status</span><div class="text-[13px] leading-relaxed wrap-break-word [&amp;_a]:no-underline [&amp;_a]:font-semibold [&amp;_a:hover]:text-primary"><span>Ongoing</span></div></div><div class="grid grid-cols-[108px_1fr] items-center gap-x-2.5 px-3.5 py-[9px] border-b border-border last:border-b-0 even:bg-black/[0.016] dark:even:bg-white/[0.022] max-[400px]:grid-cols-1 max-[400px]:gap-0.5 max-[400px]:px-3 max-[400px]:py-2"><span class="text-[10.5px] font-bold uppercase tracking-[0.07em] whitespace-nowrap pt-px leading-6 text-muted-foreground">Date Added</span><div class="text-[13px] leading-relaxed wrap-break-word [&amp;_a]:no-underline [&amp;_a]:font-semibold [&amp;_a:hover]:text-primary"><span>June 6, 2024</span></div></div><div class="grid grid-cols-[108px_1fr] items-center gap-x-2.5 px-3.5 py-[9px] border-b border-border last:border-b-0 even:bg-black/[0.016] dark:even:bg-white/[0.022] max-[400px]:grid-cols-1 max-[400px]:gap-0.5 max-[400px]:px-3 max-[400px]:py-2"><span class="text-[10.5px] font-bold uppercase tracking-[0.07em] whitespace-nowrap pt-px leading-6 text-muted-foreground">Author</span><div class="text-[13px] leading-relaxed wrap-break-word [&amp;_a]:no-underline [&amp;_a]:font-semibold [&amp;_a:hover]:text-primary flex flex-col gap-px"><a href="/en/author/%E5%A3%B9%E6%9B%B4%E5%A4%A7%E5%B8%88">壹更大师</a><a class="text-xs opacity-65" href="/en/author/%E5%A3%B9%E6%9B%B4%E5%A4%A7%E5%B8%88">Yi Geng Da Shi</a></div></div><div class="grid grid-cols-[108px_1fr] items-center gap-x-2.5 px-3.5 py-[9px] border-b border-border last:border-b-0 even:bg-black/[0.016] dark:even:bg-white/[0.022] max-[400px]:grid-cols-1 max-[400px]:gap-0.5 max-[400px]:px-3 max-[400px]:py-2"><span class="text-[10.5px] font-bold uppercase tracking-[0.07em] whitespace-nowrap pt-px leading-6 text-muted-foreground">Rankings</span><div class="text-[13px] leading-relaxed wrap-break-word flex flex-wrap gap-1.5"><a class="inline-flex flex-col items-center min-w-[58px] pt-[5px] px-2 pb-1 bg-black/3 border border-border rounded-[7px] no-underline hover:border-primary hover:bg-primary/20 hover:shadow-[0_2px_8px_rgba(253,126,20,0.12)] transition-[border-color,background,box-shadow] duration-200" href="/en/ranking/weekly"><span class="text-[9px] font-bold uppercase tracking-[0.1em] text-muted-foreground leading-[1.3]">Weekly</span><span class="text-sm font-bold text-foreground leading-[1.4]">#24929</span></a><a class="inline-flex flex-col items-center min-w-[58px] pt-[5px] px-2 pb-1 bg-black/3 border border-border rounded-[7px] no-underline hover:border-primary hover:bg-primary/20 hover:shadow-[0_2px_8px_rgba(253,126,20,0.12)] transition-[border-color,background,box-shadow] duration-200" href="/en/ranking/monthly"><span class="text-[9px] font-bold uppercase tracking-[0.1em] text-muted-foreground leading-[1.3]">Monthly</span><span class="text-sm font-bold text-foreground leading-[1.4]">#14939</span></a><a class="inline-flex flex-col items-center min-w-[58px] pt-[5px] px-2 pb-1 bg-black/3 border border-border rounded-[7px] no-underline hover:border-primary hover:bg-primary/20 hover:shadow-[0_2px_8px_rgba(253,126,20,0.12)] transition-[border-color,background,box-shadow] duration-200" href="/en/ranking/all_time"><span class="text-[9px] font-bold uppercase tracking-[0.1em] text-muted-foreground leading-[1.3]">All Time</span><span class="text-sm font-bold text-foreground leading-[1.4]">#6343</span></a></div></div></div><div class="npc-root"><div class="npc-header flex items-center mt-4 mb-1.5 gap-[7px] pl-2.5 border-l-2 border-l-primary"><svg class="icon inline-flex size-[18px] text-ticket-gold shrink-0"><use href="#crown"></use></svg><span class="flex-1 text-base font-semibold text-muted-foreground">Novel Patrons</span><button class="npc-showmore-btn"><span>View All</span><span class="npc-showmore-count">4</span></button></div><div class="npc-rows"><div class="npc-row npc-gold"><span class="npc-rank npc-gold">1</span><div class="npc-row-user"><a target="_blank" rel="noopener noreferrer" class="user-name relative inline-flex items-center gap-1" href="/en/profile/1936e7e8-25d4-4be4-9290-d2c0e2f05870"><span>Gahai</span><span title="Lv. 3" class="w_lv -mr-1" style="background-position: 0px -72px;"></span></a></div><span class="npc-ticket-pill" translate="no"><svg class="icon inline-flex shrink-0 size-6 npc-icon"><use href="#ticket" class="svg-silver"></use></svg><span class="npc-amount">374.17</span></span></div><div class="npc-row npc-silver"><span class="npc-rank npc-silver">2</span><div class="npc-row-user"><a target="_blank" rel="noopener noreferrer" class="user-name relative inline-flex items-center gap-1" href="/en/profile/512a39ec-3f8c-4430-bc3b-a6d44008f2d4"><span>HeavenWeaver</span><span title="Lv. 2" class="w_lv -mr-1" style="background-position: 0px -48px;"></span></a></div><span class="npc-ticket-pill" translate="no"><svg class="icon inline-flex shrink-0 size-6 npc-icon"><use href="#ticket" class="svg-silver"></use></svg><span class="npc-amount">108.85</span></span></div><div class="npc-row npc-bronze"><span class="npc-rank npc-bronze">3</span><div class="npc-row-user"><a target="_blank" rel="noopener noreferrer" class="user-name relative inline-flex items-center gap-1" href="/en/profile/24028d3e-0adc-40b1-824a-c4913dc20d23"><span>Kuzzon</span><span title="Lv. 1" class="w_lv -mr-1" style="background-position: 0px -24px;"></span></a></div><span class="npc-ticket-pill" translate="no"><svg class="icon inline-flex shrink-0 size-6 npc-icon"><use href="#ticket" class="svg-ticket"></use></svg><span class="npc-amount">77.75</span></span></div></div><button class="npc-footer-btn"><span>+1 more patrons</span></button></div><div class="flex items-center justify-between gap-2 mt-4 mb-2 pl-2.5 border-l-2 border-l-primary"><span class="flex-1 text-base font-semibold text-card-foreground">Genre &amp; Tags</span></div><div class="rounded-lg overflow-hidden border border-border mb-0"><div class="flex flex-wrap items-center gap-1.5 px-3.5 py-[9px] border-b border-border last:border-b-0 even:bg-black/[0.016] dark:even:bg-white/[0.022]"><div class="flex items-center w-full gap-1.5 text-[10px] uppercase tracking-wider font-bold text-muted-foreground"><span>Genre</span><span class="text-[9px] font-normal text-muted-foreground/60">5</span></div><a href="/en/novel-list?genre=1"><span class="inline-flex items-center capitalize text-xs font-medium border rounded px-1.5 py-0.5 whitespace-nowrap transition-colors border-border/70 text-card-foreground hover:bg-muted/50 hover:text-card-foreground hover:border-primary/50">action</span></a><a href="/en/novel-list?genre=3"><span class="inline-flex items-center capitalize text-xs font-medium border rounded px-1.5 py-0.5 whitespace-nowrap transition-colors border-border/70 text-card-foreground hover:bg-muted/50 hover:text-card-foreground hover:border-primary/50">adventure</span></a><a href="/en/novel-list?genre=5"><span class="inline-flex items-center capitalize text-xs font-medium border rounded px-1.5 py-0.5 whitespace-nowrap transition-colors border-border/70 text-card-foreground hover:bg-muted/50 hover:text-card-foreground hover:border-primary/50">drama</span></a><a href="/en/novel-list?genre=9"><span class="inline-flex items-center capitalize text-xs font-medium border rounded px-1.5 py-0.5 whitespace-nowrap transition-colors border-border/70 text-card-foreground hover:bg-muted/50 hover:text-card-foreground hover:border-primary/50">fantasy</span></a><a href="/en/novel-list?genre=37"><span class="inline-flex items-center capitalize text-xs font-medium border rounded px-1.5 py-0.5 whitespace-nowrap transition-colors border-border/70 text-card-foreground hover:bg-muted/50 hover:text-card-foreground hover:border-primary/50">xianxia</span></a></div><div class="flex flex-wrap items-center gap-1.5 px-3.5 py-[9px] border-b border-border last:border-b-0 even:bg-black/[0.016] dark:even:bg-white/[0.022]"><div class="flex items-center w-full gap-1.5 text-[10px] uppercase tracking-wider font-bold text-muted-foreground"><span>Protagonist Archetypes</span><span class="text-[9px] font-normal text-muted-foreground/60">9</span></div><a href="/en/novel-finder?ti=417"><span class="inline-flex items-center capitalize text-xs border rounded px-1.5 py-0.5 whitespace-nowrap transition-colors border-blue-400/50 text-blue-600 dark:text-blue-400 bg-blue-50/50 dark:bg-blue-950/20 hover:bg-blue-100/70 dark:hover:bg-blue-950/40 font-semibold">Male</span></a><a href="/en/novel-finder?ti=115"><span class="inline-flex items-center capitalize text-xs font-medium border rounded px-1.5 py-0.5 whitespace-nowrap transition-colors border-border/70 text-card-foreground hover:bg-muted/50 hover:text-card-foreground hover:border-primary/50">Caring</span></a><a href="/en/novel-finder?ti=125"><span class="inline-flex items-center capitalize text-xs font-medium border rounded px-1.5 py-0.5 whitespace-nowrap transition-colors border-border/70 text-card-foreground hover:bg-muted/50 hover:text-card-foreground hover:border-primary/50">Child</span></a><a href="/en/novel-finder?ti=197"><span class="inline-flex items-center capitalize text-xs font-medium border rounded px-1.5 py-0.5 whitespace-nowrap transition-colors border-border/70 text-card-foreground hover:bg-muted/50 hover:text-card-foreground hover:border-primary/50">Determined</span></a><a href="/en/novel-finder?ti=539"><span class="inline-flex items-center capitalize text-xs font-medium border rounded px-1.5 py-0.5 whitespace-nowrap transition-colors border-border/70 text-card-foreground hover:bg-muted/50 hover:text-card-foreground hover:border-primary/50">Poor</span></a><a href="/en/novel-finder?ti=590"><span class="inline-flex items-center capitalize text-xs font-medium border rounded px-1.5 py-0.5 whitespace-nowrap transition-colors border-border/70 text-card-foreground hover:bg-muted/50 hover:text-card-foreground hover:border-primary/50">Righteous</span></a><a href="/en/novel-finder?ti=715"><span class="inline-flex items-center capitalize text-xs font-medium border rounded px-1.5 py-0.5 whitespace-nowrap transition-colors border-border/70 text-card-foreground hover:bg-muted/50 hover:text-card-foreground hover:border-primary/50">Tragic Past</span></a><a href="/en/novel-finder?ti=731"><span class="inline-flex items-center capitalize text-xs font-medium border rounded px-1.5 py-0.5 whitespace-nowrap transition-colors border-border/70 text-card-foreground hover:bg-muted/50 hover:text-card-foreground hover:border-primary/50">Underestimated</span></a><a href="/en/novel-finder?ti=750"><span class="inline-flex items-center capitalize text-xs font-medium border rounded px-1.5 py-0.5 whitespace-nowrap transition-colors border-border/70 text-card-foreground hover:bg-muted/50 hover:text-card-foreground hover:border-primary/50">Weak to Strong</span></a></div><div class="flex flex-wrap items-center gap-1.5 px-3.5 py-[9px] border-b border-border last:border-b-0 even:bg-black/[0.016] dark:even:bg-white/[0.022]"><div class="flex items-center w-full gap-1.5 text-[10px] uppercase tracking-wider font-bold text-muted-foreground"><span>Power Systems</span><span class="text-[9px] font-normal text-muted-foreground/60">2</span></div><a href="/en/novel-finder?ti=93"><span class="inline-flex items-center capitalize text-xs font-medium border rounded px-1.5 py-0.5 whitespace-nowrap transition-colors border-border/70 text-card-foreground hover:bg-muted/50 hover:text-card-foreground hover:border-primary/50">Bloodlines</span></a><a href="/en/novel-finder?ti=169"><span class="inline-flex items-center capitalize text-xs font-medium border rounded px-1.5 py-0.5 whitespace-nowrap transition-colors border-border/70 text-card-foreground hover:bg-muted/50 hover:text-card-foreground hover:border-primary/50">Cultivation</span></a></div><div class="flex flex-wrap items-center gap-1.5 px-3.5 py-[9px] border-b border-border last:border-b-0 even:bg-black/[0.016] dark:even:bg-white/[0.022]"><div class="flex items-center w-full gap-1.5 text-[10px] uppercase tracking-wider font-bold text-muted-foreground"><span>Worldbuilding</span><span class="text-[9px] font-normal text-muted-foreground/60">1</span></div><a href="/en/novel-finder?ti=265"><span class="inline-flex items-center capitalize text-xs font-medium border rounded px-1.5 py-0.5 whitespace-nowrap transition-colors border-border/70 text-card-foreground hover:bg-muted/50 hover:text-card-foreground hover:border-primary/50">Fantasy World</span></a></div><div class="flex flex-wrap items-center gap-1.5 px-3.5 py-[9px] border-b border-border last:border-b-0 even:bg-black/[0.016] dark:even:bg-white/[0.022]"><div class="flex items-center w-full gap-1.5 text-[10px] uppercase tracking-wider font-bold text-muted-foreground"><span>Socio-Political Structures</span><span class="text-[9px] font-normal text-muted-foreground/60">1</span></div><a href="/en/novel-finder?ti=680"><span class="inline-flex items-center capitalize text-xs font-medium border rounded px-1.5 py-0.5 whitespace-nowrap transition-colors border-border/70 text-card-foreground hover:bg-muted/50 hover:text-card-foreground hover:border-primary/50">Strength-based Social Hierarchy</span></a></div><div class="flex flex-wrap items-center gap-1.5 px-3.5 py-[9px] border-b border-border last:border-b-0 even:bg-black/[0.016] dark:even:bg-white/[0.022]"><div class="flex items-center w-full gap-1.5 text-[10px] uppercase tracking-wider font-bold text-muted-foreground"><span>Relationship Tropes</span><span class="text-[9px] font-normal text-muted-foreground/60">2</span></div><a href="/en/novel-finder?ti=184"><span class="inline-flex items-center capitalize text-xs font-medium border rounded px-1.5 py-0.5 whitespace-nowrap transition-colors border-border/70 text-card-foreground hover:bg-muted/50 hover:text-card-foreground hover:border-primary/50">Death of Loved Ones</span></a><a href="/en/novel-finder?ti=257"><span class="inline-flex items-center capitalize text-xs font-medium border rounded px-1.5 py-0.5 whitespace-nowrap transition-colors border-border/70 text-card-foreground hover:bg-muted/50 hover:text-card-foreground hover:border-primary/50">Family</span></a></div><div class="flex flex-wrap items-center gap-1.5 px-3.5 py-[9px] border-b border-border last:border-b-0 even:bg-black/[0.016] dark:even:bg-white/[0.022]"><div class="flex items-center w-full gap-1.5 text-[10px] uppercase tracking-wider font-bold text-muted-foreground"><span>Narrative</span><span class="text-[9px] font-normal text-muted-foreground/60">1</span></div><a href="/en/novel-finder?ti=692"><span class="inline-flex items-center capitalize text-xs font-medium border rounded px-1.5 py-0.5 whitespace-nowrap transition-colors border-border/70 text-card-foreground hover:bg-muted/50 hover:text-card-foreground hover:border-primary/50">Survival</span></a></div><div class="flex flex-wrap items-center gap-1.5 px-3.5 py-[9px] border-b border-border last:border-b-0 even:bg-black/[0.016] dark:even:bg-white/[0.022]"><div class="flex items-center w-full gap-1.5 text-[10px] uppercase tracking-wider font-bold text-muted-foreground"><span>Beings &amp; Factions</span><span class="text-[9px] font-normal text-muted-foreground/60">1</span></div><a href="/en/novel-finder?ti=357"><span class="inline-flex items-center capitalize text-xs font-medium border rounded px-1.5 py-0.5 whitespace-nowrap transition-colors border-border/70 text-card-foreground hover:bg-muted/50 hover:text-card-foreground hover:border-primary/50">Immortals</span></a></div><div class="flex flex-wrap items-center gap-1.5 px-3.5 py-[9px] border-b border-border last:border-b-0 even:bg-black/[0.016] dark:even:bg-white/[0.022]"><div class="flex items-center w-full gap-1.5 text-[10px] uppercase tracking-wider font-bold text-muted-foreground"><span>Tone &amp; Atmosphere</span><span class="text-[9px] font-normal text-muted-foreground/60">2</span></div><a href="/en/novel-finder?ti=181"><span class="inline-flex items-center capitalize text-xs font-medium border rounded px-1.5 py-0.5 whitespace-nowrap transition-colors border-border/70 text-card-foreground hover:bg-muted/50 hover:text-card-foreground hover:border-primary/50">Dark</span></a><a href="/en/novel-finder?ti=193"><span class="inline-flex items-center capitalize text-xs font-medium border rounded px-1.5 py-0.5 whitespace-nowrap transition-colors border-border/70 text-card-foreground hover:bg-muted/50 hover:text-card-foreground hover:border-primary/50">Depictions of Cruelty</span></a></div></div></div><div data-hidden="" data-orientation="horizontal" data-activation-direction="none" hidden="" id="base-ui-_r_110_" role="tabpanel" tabindex="-1" inert="" data-index="1" data-slot="tabs-content" class="flex-1 text-sm outline-none" aria-labelledby="base-ui-_r_10p_"><div><div class="mb-3"><div class="flex items-center justify-between mb-2"><span class="font-bold"><span>Rating 3.4/5.0</span></span><span class="text-foreground text-sm">10 votes</span></div><div class="flex flex-col"><div class="flex items-center mb-1 group/rating"><span class="mr-2" style="min-width: 12px; font-size: 0.875rem;">5</span><div class="grow mr-2"><div class="h-3.5 w-full overflow-hidden rounded-full bg-accent/50 flex"><div class="h-full bg-[var(--review-rate-5)]" style="width: 10%;"></div><div class="h-full bg-[var(--review-rate-5)] opacity-35" style="width: 30%;"></div></div></div><span class="text-muted-foreground text-xs" style="min-width: 65px; text-align: right;">40% (4)</span></div><div class="flex items-center mb-1 group/rating"><span class="mr-2" style="min-width: 12px; font-size: 0.875rem;">4</span><div class="grow mr-2"><div class="h-3.5 w-full overflow-hidden rounded-full bg-accent/50 flex"><div class="h-full bg-[var(--review-rate-4)]" style="width: 10%;"></div><div class="h-full bg-[var(--review-rate-4)] opacity-35" style="width: 10%;"></div></div></div><span class="text-muted-foreground text-xs" style="min-width: 65px; text-align: right;">20% (2)</span></div><div class="flex items-center mb-1 group/rating"><span class="mr-2" style="min-width: 12px; font-size: 0.875rem;">3</span><div class="grow mr-2"><div class="h-3.5 w-full overflow-hidden rounded-full bg-accent/50 flex"><div class="h-full bg-[var(--review-rate-3)] opacity-35" style="width: 10%;"></div></div></div><span class="text-muted-foreground text-xs" style="min-width: 65px; text-align: right;">10% (1)</span></div><div class="flex items-center mb-1 group/rating"><span class="mr-2" style="min-width: 12px; font-size: 0.875rem;">2</span><div class="grow mr-2"><div class="h-3.5 w-full overflow-hidden rounded-full bg-accent/50 flex"></div></div><span class="text-muted-foreground text-xs" style="min-width: 65px; text-align: right;">0% (0)</span></div><div class="flex items-center mb-1 group/rating"><span class="mr-2" style="min-width: 12px; font-size: 0.875rem;">1</span><div class="grow mr-2"><div class="h-3.5 w-full overflow-hidden rounded-full bg-accent/50 flex"><div class="h-full bg-[var(--review-rate-1)]" style="width: 30%;"></div></div></div><span class="text-muted-foreground text-xs" style="min-width: 65px; text-align: right;">30% (3)</span></div></div></div><div><div class="flex justify-between items-center mb-3"><h5 class="font-semibold">Reviews</h5><div class="flex gap-2 items-center"><button type="button" tabindex="0" data-slot="button" class="group/button inline-flex cursor-pointer shrink-0 items-center justify-center bg-clip-padding font-medium whitespace-nowrap transition-all outline-none select-none focus-visible:border-ring focus-visible:ring-3 focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-3 aria-invalid:ring-destructive/20 dark:aria-invalid:border-destructive/50 dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 border border-border bg-card hover:bg-muted hover:text-foreground aria-expanded:bg-muted aria-expanded:text-foreground dark:border-input dark:bg-input/30 dark:hover:bg-input/50 h-7 gap-1 rounded-[min(var(--radius-md),12px)] px-2.5 text-[0.8rem] in-data-[slot=button-group]:rounded-lg has-data-[icon=inline-end]:pr-1.5 has-data-[icon=inline-start]:pl-1.5 [&amp;_svg:not([class*='size-'])]:size-3.5">Comments Only</button><div data-orientation="horizontal" aria-orientation="horizontal" role="group" data-slot="toggle-group" data-variant="outline" data-size="sm" data-spacing="0" class="group/toggle-group flex w-fit bg-card dark:bg-input/30 flex-row items-center gap-[--spacing(var(--gap))] rounded-lg data-[size=sm]:rounded-[min(var(--radius-md),10px)] data-vertical:flex-col data-vertical:items-stretch" style="--gap: 0;"><button type="button" data-pressed="" aria-disabled="false" tabindex="0" aria-pressed="true" data-slot="toggle-group-item" data-variant="outline" data-size="sm" data-spacing="0" class="shrink-0 group-data-[spacing=0]/toggle-group:rounded-none group-data-[spacing=0]/toggle-group:px-2 focus:z-10 focus-visible:z-10 group-data-[spacing=0]/toggle-group:has-data-[icon=inline-end]:pr-1.5 group-data-[spacing=0]/toggle-group:has-data-[icon=inline-start]:pl-1.5 group-data-horizontal/toggle-group:data-[spacing=0]:first:rounded-l-lg group-data-vertical/toggle-group:data-[spacing=0]:first:rounded-t-lg group-data-horizontal/toggle-group:data-[spacing=0]:last:rounded-r-lg group-data-vertical/toggle-group:data-[spacing=0]:last:rounded-b-lg group-data-horizontal/toggle-group:data-[spacing=0]:data-[variant=outline]:border-l-0 group-data-vertical/toggle-group:data-[spacing=0]:data-[variant=outline]:border-t-0 group-data-horizontal/toggle-group:data-[spacing=0]:data-[variant=outline]:first:border-l group-data-vertical/toggle-group:data-[spacing=0]:data-[variant=outline]:first:border-t cursor-pointer group/toggle inline-flex items-center justify-center gap-1 font-medium whitespace-nowrap transition-all outline-none hover:text-foreground focus-visible:border-ring focus-visible:ring-[3px] focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-destructive/20 aria-pressed:bg-muted data-[state=on]:bg-muted dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 border border-input bg-transparent hover:bg-muted h-7 min-w-7 rounded-[min(var(--radius-md),12px)] px-2.5 text-[0.8rem] has-data-[icon=inline-end]:pr-1.5 has-data-[icon=inline-start]:pl-1.5 [&amp;_svg:not([class*='size-'])]:size-3.5">Best</button><button type="button" aria-disabled="false" tabindex="-1" aria-pressed="false" data-slot="toggle-group-item" data-variant="outline" data-size="sm" data-spacing="0" class="shrink-0 group-data-[spacing=0]/toggle-group:rounded-none group-data-[spacing=0]/toggle-group:px-2 focus:z-10 focus-visible:z-10 group-data-[spacing=0]/toggle-group:has-data-[icon=inline-end]:pr-1.5 group-data-[spacing=0]/toggle-group:has-data-[icon=inline-start]:pl-1.5 group-data-horizontal/toggle-group:data-[spacing=0]:first:rounded-l-lg group-data-vertical/toggle-group:data-[spacing=0]:first:rounded-t-lg group-data-horizontal/toggle-group:data-[spacing=0]:last:rounded-r-lg group-data-vertical/toggle-group:data-[spacing=0]:last:rounded-b-lg group-data-horizontal/toggle-group:data-[spacing=0]:data-[variant=outline]:border-l-0 group-data-vertical/toggle-group:data-[spacing=0]:data-[variant=outline]:border-t-0 group-data-horizontal/toggle-group:data-[spacing=0]:data-[variant=outline]:first:border-l group-data-vertical/toggle-group:data-[spacing=0]:data-[variant=outline]:first:border-t cursor-pointer group/toggle inline-flex items-center justify-center gap-1 font-medium whitespace-nowrap transition-all outline-none hover:text-foreground focus-visible:border-ring focus-visible:ring-[3px] focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-destructive/20 aria-pressed:bg-muted data-[state=on]:bg-muted dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 border border-input bg-transparent hover:bg-muted h-7 min-w-7 rounded-[min(var(--radius-md),12px)] px-2.5 text-[0.8rem] has-data-[icon=inline-end]:pr-1.5 has-data-[icon=inline-start]:pl-1.5 [&amp;_svg:not([class*='size-'])]:size-3.5">Newest</button></div></div></div><div class="flex flex-col gap-2 mb-2"><div class="transition-all duration-200 border rounded-md p-2 border-transparent border-b-border last:border-b-transparent hover:border-secondary hover:border-b-secondary  "><div class="mb-2"><div class="flex justify-between items-start"><div class="flex items-center"><div class="mr-1"><div class="font-bold flex items-center"><a target="_blank" rel="noopener noreferrer" class="user-name relative inline-flex items-center gap-1" href="/en/profile/387faadc-b91e-446d-a14a-ef916bd87ac2"><span>Death_012</span><span title="Lv. 4" class="w_lv -mr-1" style="background-position: 0px -96px;"></span></a></div><div class="flex items-center"><ul class="rc-rate mr-2 rc-rate-disabled" tabindex="-1"><li class="rc-rate-star rc-rate-star-full"><div role="radio" aria-checked="true" aria-posinset="1" aria-setsize="5" tabindex="-1"><div class="rc-rate-star-first">★</div><div class="rc-rate-star-second">★</div></div></li><li class="rc-rate-star rc-rate-star-zero"><div role="radio" aria-checked="false" aria-posinset="2" aria-setsize="5" tabindex="-1"><div class="rc-rate-star-first">★</div><div class="rc-rate-star-second">★</div></div></li><li class="rc-rate-star rc-rate-star-zero"><div role="radio" aria-checked="false" aria-posinset="3" aria-setsize="5" tabindex="-1"><div class="rc-rate-star-first">★</div><div class="rc-rate-star-second">★</div></div></li><li class="rc-rate-star rc-rate-star-zero"><div role="radio" aria-checked="false" aria-posinset="4" aria-setsize="5" tabindex="-1"><div class="rc-rate-star-first">★</div><div class="rc-rate-star-second">★</div></div></li><li class="rc-rate-star rc-rate-star-zero"><div role="radio" aria-checked="false" aria-posinset="5" aria-setsize="5" tabindex="-1"><div class="rc-rate-star-first">★</div><div class="rc-rate-star-second">★</div></div></li></ul></div></div></div><div class="flex flex-col text-right gap-1"><div class="text-muted-foreground text-sm flex items-center ml-auto"><span>Jun 29, 2024</span><button type="button" tabindex="0" data-slot="dropdown-menu-trigger" aria-haspopup="menu" id="base-ui-_r_1le_" class="group/button inline-flex cursor-pointer shrink-0 items-center justify-center bg-clip-padding font-medium whitespace-nowrap transition-all outline-none select-none focus-visible:border-ring focus-visible:ring-3 focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-3 aria-invalid:ring-destructive/20 dark:aria-invalid:border-destructive/50 dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 border-border bg-card hover:bg-muted hover:text-foreground aria-expanded:bg-muted aria-expanded:text-foreground dark:border-input dark:bg-input/30 dark:hover:bg-input/50 h-7 gap-1 rounded-[min(var(--radius-md),12px)] text-[0.8rem] in-data-[slot=button-group]:rounded-lg has-data-[icon=inline-end]:pr-1.5 has-data-[icon=inline-start]:pl-1.5 [&amp;_svg:not([class*='size-'])]:size-3.5 p-0 border-0 ml-1" aria-expanded="false"><svg class="icon inline-flex shrink-0 size-6"><use href="#more"></use></svg></button></div><div class="flex items-center gap-1.5 text-[12px] ml-auto text-muted-foreground"><span>Read 28</span><span aria-hidden="true" class="select-none">|</span><span><b>Ch. #26</b></span></div></div></div></div><div class="mb-3 wrap-break-word text-base"><div class="tiptap-content text-base "><p>MC seems NAIVE &amp;amp; KIND.</p></div><button type="button" tabindex="0" data-slot="button" class="group/button cursor-pointer shrink-0 justify-center border-transparent bg-clip-padding font-medium whitespace-nowrap transition-all outline-none select-none focus-visible:border-ring focus-visible:ring-3 focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-3 aria-invalid:ring-destructive/20 dark:aria-invalid:border-destructive/50 dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 underline-offset-4 hover:underline h-7 rounded-[min(var(--radius-md),12px)] text-[0.8rem] in-data-[slot=button-group]:rounded-lg has-data-[icon=inline-end]:pr-1.5 has-data-[icon=inline-start]:pl-1.5 [&amp;_svg:not([class*='size-'])]:size-3.5 mt-1 flex items-center gap-1 border-0 p-0 text-muted-foreground"><svg class="icon inline-flex shrink-0 size-6"><use href="#g_translate"></use></svg><span>Translate to British English</span></button></div><div class="flex items-center [&amp;_.btn-sm]:min-h-[26px] [&amp;_.btn-sm]:min-w-[26px] [&amp;_svg]:size-5"><button type="button" tabindex="0" data-slot="button" class="group/button inline-flex cursor-pointer shrink-0 items-center justify-center border-transparent bg-clip-padding whitespace-nowrap transition-all outline-none select-none focus-visible:border-ring focus-visible:ring-3 focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-3 aria-invalid:ring-destructive/20 dark:aria-invalid:border-destructive/50 dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 underline-offset-4 hover:underline h-7 gap-1 rounded-[min(var(--radius-md),12px)] text-[0.8rem] in-data-[slot=button-group]:rounded-lg has-data-[icon=inline-end]:pr-1.5 has-data-[icon=inline-start]:pl-1.5 [&amp;_svg:not([class*='size-'])]:size-3.5 text-primary p-0 border-0 font-semibold uppercase"><svg class="icon inline-flex shrink-0 size-6 mr-1"><use href="#review"></use></svg><span>View 1 Reply</span></button><div class="flex items-center gap-2 ml-auto"><button type="button" tabindex="0" data-slot="button" class="group/button cursor-pointer shrink-0 justify-center border-transparent bg-clip-padding font-medium whitespace-nowrap transition-all outline-none select-none focus-visible:border-ring focus-visible:ring-3 focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-3 aria-invalid:ring-destructive/20 dark:aria-invalid:border-destructive/50 dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 h-7 gap-1 rounded-[min(var(--radius-md),12px)] text-[0.8rem] in-data-[slot=button-group]:rounded-lg has-data-[icon=inline-end]:pr-1.5 has-data-[icon=inline-start]:pl-1.5 [&amp;_svg:not([class*='size-'])]:size-3.5 text-muted-foreground hover:text-primary p-0 flex items-center"><svg class="icon inline-flex shrink-0 size-6"><use href="#premium"></use></svg></button><span class="like-button flex items-center cursor-pointer text-muted-foreground hover-primary-text" title="Like" style="cursor: pointer;"><svg class="icon inline-flex shrink-0 size-6 like"><use href="#like"></use></svg><span class="ml-1">2</span></span><button type="button" tabindex="0" data-slot="button" class="group/button cursor-pointer shrink-0 items-center justify-center border-transparent bg-clip-padding font-medium whitespace-nowrap transition-all outline-none select-none focus-visible:border-ring focus-visible:ring-3 focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-3 aria-invalid:ring-destructive/20 dark:aria-invalid:border-destructive/50 dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 flex h-7 gap-1 rounded-[min(var(--radius-md),12px)] text-[0.8rem] in-data-[slot=button-group]:rounded-lg has-data-[icon=inline-end]:pr-1.5 has-data-[icon=inline-start]:pl-1.5 [&amp;_svg:not([class*='size-'])]:size-3.5 text-muted-foreground hover:text-primary p-0"><svg class="icon inline-flex shrink-0 size-6"><use href="#reply"></use></svg></button></div></div></div><div class="transition-all duration-200 border rounded-md p-2 border-transparent border-b-border last:border-b-transparent hover:border-secondary hover:border-b-secondary  "><div class="mb-2"><div class="flex justify-between items-start"><div class="flex items-center"><div class="mr-1"><div class="font-bold flex items-center"><a target="_blank" rel="noopener noreferrer" class="user-name relative inline-flex items-center gap-1" href="/en/profile/6fd0bf5b-8d1b-43a0-86df-5723b17e7423"><span>ABSenior</span><span title="Lv. 3" class="w_lv -mr-1" style="background-position: 0px -72px;"></span></a></div><div class="flex items-center"><ul class="rc-rate mr-2 rc-rate-disabled" tabindex="-1"><li class="rc-rate-star rc-rate-star-full"><div role="radio" aria-checked="true" aria-posinset="1" aria-setsize="5" tabindex="-1"><div class="rc-rate-star-first">★</div><div class="rc-rate-star-second">★</div></div></li><li class="rc-rate-star rc-rate-star-zero"><div role="radio" aria-checked="false" aria-posinset="2" aria-setsize="5" tabindex="-1"><div class="rc-rate-star-first">★</div><div class="rc-rate-star-second">★</div></div></li><li class="rc-rate-star rc-rate-star-zero"><div role="radio" aria-checked="false" aria-posinset="3" aria-setsize="5" tabindex="-1"><div class="rc-rate-star-first">★</div><div class="rc-rate-star-second">★</div></div></li><li class="rc-rate-star rc-rate-star-zero"><div role="radio" aria-checked="false" aria-posinset="4" aria-setsize="5" tabindex="-1"><div class="rc-rate-star-first">★</div><div class="rc-rate-star-second">★</div></div></li><li class="rc-rate-star rc-rate-star-zero"><div role="radio" aria-checked="false" aria-posinset="5" aria-setsize="5" tabindex="-1"><div class="rc-rate-star-first">★</div><div class="rc-rate-star-second">★</div></div></li></ul></div></div></div><div class="flex flex-col text-right gap-1"><div class="text-muted-foreground text-sm flex items-center ml-auto"><span>Apr 23, 2026</span><button type="button" tabindex="0" data-slot="dropdown-menu-trigger" aria-haspopup="menu" id="base-ui-_r_1lk_" class="group/button inline-flex cursor-pointer shrink-0 items-center justify-center bg-clip-padding font-medium whitespace-nowrap transition-all outline-none select-none focus-visible:border-ring focus-visible:ring-3 focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-3 aria-invalid:ring-destructive/20 dark:aria-invalid:border-destructive/50 dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 border-border bg-card hover:bg-muted hover:text-foreground aria-expanded:bg-muted aria-expanded:text-foreground dark:border-input dark:bg-input/30 dark:hover:bg-input/50 h-7 gap-1 rounded-[min(var(--radius-md),12px)] text-[0.8rem] in-data-[slot=button-group]:rounded-lg has-data-[icon=inline-end]:pr-1.5 has-data-[icon=inline-start]:pl-1.5 [&amp;_svg:not([class*='size-'])]:size-3.5 p-0 border-0 ml-1" aria-expanded="false"><svg class="icon inline-flex shrink-0 size-6"><use href="#more"></use></svg></button></div><div class="flex items-center gap-1.5 text-[12px] ml-auto text-muted-foreground"><span>Read 15</span><span aria-hidden="true" class="select-none">|</span><span><b>Ch. #16</b></span></div></div></div></div><div class="mb-3 wrap-break-word text-base"><div class="tiptap-content text-base "><p>mc just like any good hearted protagonist help people without benefits. He has a beautiful girl as his golden finger, and hates people who commit evil.</p></div><button type="button" tabindex="0" data-slot="button" class="group/button cursor-pointer shrink-0 justify-center border-transparent bg-clip-padding font-medium whitespace-nowrap transition-all outline-none select-none focus-visible:border-ring focus-visible:ring-3 focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-3 aria-invalid:ring-destructive/20 dark:aria-invalid:border-destructive/50 dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 underline-offset-4 hover:underline h-7 rounded-[min(var(--radius-md),12px)] text-[0.8rem] in-data-[slot=button-group]:rounded-lg has-data-[icon=inline-end]:pr-1.5 has-data-[icon=inline-start]:pl-1.5 [&amp;_svg:not([class*='size-'])]:size-3.5 mt-1 flex items-center gap-1 border-0 p-0 text-muted-foreground"><svg class="icon inline-flex shrink-0 size-6"><use href="#g_translate"></use></svg><span>Translate to British English</span></button></div><div class="flex items-center [&amp;_.btn-sm]:min-h-[26px] [&amp;_.btn-sm]:min-w-[26px] [&amp;_svg]:size-5"><button type="button" tabindex="0" data-slot="button" class="group/button inline-flex cursor-pointer shrink-0 items-center justify-center border-transparent bg-clip-padding whitespace-nowrap transition-all outline-none select-none focus-visible:border-ring focus-visible:ring-3 focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-3 aria-invalid:ring-destructive/20 dark:aria-invalid:border-destructive/50 dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 underline-offset-4 hover:underline h-7 gap-1 rounded-[min(var(--radius-md),12px)] text-[0.8rem] in-data-[slot=button-group]:rounded-lg has-data-[icon=inline-end]:pr-1.5 has-data-[icon=inline-start]:pl-1.5 [&amp;_svg:not([class*='size-'])]:size-3.5 text-primary p-0 border-0 font-semibold uppercase"><svg class="icon inline-flex shrink-0 size-6 mr-1"><use href="#review"></use></svg><span>View 0 Replies</span></button><div class="flex items-center gap-2 ml-auto"><button type="button" tabindex="0" data-slot="button" class="group/button cursor-pointer shrink-0 justify-center border-transparent bg-clip-padding font-medium whitespace-nowrap transition-all outline-none select-none focus-visible:border-ring focus-visible:ring-3 focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-3 aria-invalid:ring-destructive/20 dark:aria-invalid:border-destructive/50 dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 h-7 gap-1 rounded-[min(var(--radius-md),12px)] text-[0.8rem] in-data-[slot=button-group]:rounded-lg has-data-[icon=inline-end]:pr-1.5 has-data-[icon=inline-start]:pl-1.5 [&amp;_svg:not([class*='size-'])]:size-3.5 text-muted-foreground hover:text-primary p-0 flex items-center"><svg class="icon inline-flex shrink-0 size-6"><use href="#premium"></use></svg></button><span class="like-button flex items-center cursor-pointer text-muted-foreground hover-primary-text" title="Like" style="cursor: pointer;"><svg class="icon inline-flex shrink-0 size-6 like"><use href="#like"></use></svg><span class="ml-1">1</span></span><button type="button" tabindex="0" data-slot="button" class="group/button cursor-pointer shrink-0 items-center justify-center border-transparent bg-clip-padding font-medium whitespace-nowrap transition-all outline-none select-none focus-visible:border-ring focus-visible:ring-3 focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-3 aria-invalid:ring-destructive/20 dark:aria-invalid:border-destructive/50 dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 flex h-7 gap-1 rounded-[min(var(--radius-md),12px)] text-[0.8rem] in-data-[slot=button-group]:rounded-lg has-data-[icon=inline-end]:pr-1.5 has-data-[icon=inline-start]:pl-1.5 [&amp;_svg:not([class*='size-'])]:size-3.5 text-muted-foreground hover:text-primary p-0"><svg class="icon inline-flex shrink-0 size-6"><use href="#reply"></use></svg></button></div></div></div><div class="transition-all duration-200 border rounded-md p-2 border-transparent border-b-border last:border-b-transparent hover:border-secondary hover:border-b-secondary  "><div class="mb-2"><div class="flex justify-between items-start"><div class="flex items-center"><div class="mr-1"><div class="font-bold flex items-center"><a target="_blank" rel="noopener noreferrer" class="user-name relative inline-flex items-center gap-1" href="/en/profile/2d5f4a31-9d04-493d-ae74-1da87bf7ecec"><span>PrimeNumber</span><span title="Lv. 2" class="w_lv -mr-1" style="background-position: 0px -48px;"></span></a></div><div class="flex items-center"><ul class="rc-rate mr-2 rc-rate-disabled" tabindex="-1"><li class="rc-rate-star rc-rate-star-full"><div role="radio" aria-checked="true" aria-posinset="1" aria-setsize="5" tabindex="-1"><div class="rc-rate-star-first">★</div><div class="rc-rate-star-second">★</div></div></li><li class="rc-rate-star rc-rate-star-zero"><div role="radio" aria-checked="false" aria-posinset="2" aria-setsize="5" tabindex="-1"><div class="rc-rate-star-first">★</div><div class="rc-rate-star-second">★</div></div></li><li class="rc-rate-star rc-rate-star-zero"><div role="radio" aria-checked="false" aria-posinset="3" aria-setsize="5" tabindex="-1"><div class="rc-rate-star-first">★</div><div class="rc-rate-star-second">★</div></div></li><li class="rc-rate-star rc-rate-star-zero"><div role="radio" aria-checked="false" aria-posinset="4" aria-setsize="5" tabindex="-1"><div class="rc-rate-star-first">★</div><div class="rc-rate-star-second">★</div></div></li><li class="rc-rate-star rc-rate-star-zero"><div role="radio" aria-checked="false" aria-posinset="5" aria-setsize="5" tabindex="-1"><div class="rc-rate-star-first">★</div><div class="rc-rate-star-second">★</div></div></li></ul></div></div></div><div class="flex flex-col text-right gap-1"><div class="text-muted-foreground text-sm flex items-center ml-auto"><span>Feb 21, 2026</span><button type="button" tabindex="0" data-slot="dropdown-menu-trigger" aria-haspopup="menu" id="base-ui-_r_1lq_" class="group/button inline-flex cursor-pointer shrink-0 items-center justify-center bg-clip-padding font-medium whitespace-nowrap transition-all outline-none select-none focus-visible:border-ring focus-visible:ring-3 focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-3 aria-invalid:ring-destructive/20 dark:aria-invalid:border-destructive/50 dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 border-border bg-card hover:bg-muted hover:text-foreground aria-expanded:bg-muted aria-expanded:text-foreground dark:border-input dark:bg-input/30 dark:hover:bg-input/50 h-7 gap-1 rounded-[min(var(--radius-md),12px)] text-[0.8rem] in-data-[slot=button-group]:rounded-lg has-data-[icon=inline-end]:pr-1.5 has-data-[icon=inline-start]:pl-1.5 [&amp;_svg:not([class*='size-'])]:size-3.5 p-0 border-0 ml-1" aria-expanded="false"><svg class="icon inline-flex shrink-0 size-6"><use href="#more"></use></svg></button></div><div class="flex items-center gap-1.5 text-[12px] ml-auto text-muted-foreground"><span>Read 15</span><span aria-hidden="true" class="select-none">|</span><span><b>Ch. #32</b></span></div></div></div></div><div class="mb-3 wrap-break-word text-base"><div class="tiptap-content text-base "><p>Power fantasy.</p></div><button type="button" tabindex="0" data-slot="button" class="group/button cursor-pointer shrink-0 justify-center border-transparent bg-clip-padding font-medium whitespace-nowrap transition-all outline-none select-none focus-visible:border-ring focus-visible:ring-3 focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-3 aria-invalid:ring-destructive/20 dark:aria-invalid:border-destructive/50 dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 underline-offset-4 hover:underline h-7 rounded-[min(var(--radius-md),12px)] text-[0.8rem] in-data-[slot=button-group]:rounded-lg has-data-[icon=inline-end]:pr-1.5 has-data-[icon=inline-start]:pl-1.5 [&amp;_svg:not([class*='size-'])]:size-3.5 mt-1 flex items-center gap-1 border-0 p-0 text-muted-foreground"><svg class="icon inline-flex shrink-0 size-6"><use href="#g_translate"></use></svg><span>Translate to British English</span></button></div><div class="flex items-center [&amp;_.btn-sm]:min-h-[26px] [&amp;_.btn-sm]:min-w-[26px] [&amp;_svg]:size-5"><button type="button" tabindex="0" data-slot="button" class="group/button inline-flex cursor-pointer shrink-0 items-center justify-center border-transparent bg-clip-padding whitespace-nowrap transition-all outline-none select-none focus-visible:border-ring focus-visible:ring-3 focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-3 aria-invalid:ring-destructive/20 dark:aria-invalid:border-destructive/50 dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 underline-offset-4 hover:underline h-7 gap-1 rounded-[min(var(--radius-md),12px)] text-[0.8rem] in-data-[slot=button-group]:rounded-lg has-data-[icon=inline-end]:pr-1.5 has-data-[icon=inline-start]:pl-1.5 [&amp;_svg:not([class*='size-'])]:size-3.5 text-primary p-0 border-0 font-semibold uppercase"><svg class="icon inline-flex shrink-0 size-6 mr-1"><use href="#review"></use></svg><span>View 0 Replies</span></button><div class="flex items-center gap-2 ml-auto"><button type="button" tabindex="0" data-slot="button" class="group/button cursor-pointer shrink-0 justify-center border-transparent bg-clip-padding font-medium whitespace-nowrap transition-all outline-none select-none focus-visible:border-ring focus-visible:ring-3 focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-3 aria-invalid:ring-destructive/20 dark:aria-invalid:border-destructive/50 dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 h-7 gap-1 rounded-[min(var(--radius-md),12px)] text-[0.8rem] in-data-[slot=button-group]:rounded-lg has-data-[icon=inline-end]:pr-1.5 has-data-[icon=inline-start]:pl-1.5 [&amp;_svg:not([class*='size-'])]:size-3.5 text-muted-foreground hover:text-primary p-0 flex items-center"><svg class="icon inline-flex shrink-0 size-6"><use href="#premium"></use></svg></button><span class="like-button flex items-center cursor-pointer text-muted-foreground hover-primary-text" title="Like" style="cursor: pointer;"><svg class="icon inline-flex shrink-0 size-6 like"><use href="#like"></use></svg><span class="ml-1">1</span></span><button type="button" tabindex="0" data-slot="button" class="group/button cursor-pointer shrink-0 items-center justify-center border-transparent bg-clip-padding font-medium whitespace-nowrap transition-all outline-none select-none focus-visible:border-ring focus-visible:ring-3 focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-3 aria-invalid:ring-destructive/20 dark:aria-invalid:border-destructive/50 dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 flex h-7 gap-1 rounded-[min(var(--radius-md),12px)] text-[0.8rem] in-data-[slot=button-group]:rounded-lg has-data-[icon=inline-end]:pr-1.5 has-data-[icon=inline-start]:pl-1.5 [&amp;_svg:not([class*='size-'])]:size-3.5 text-muted-foreground hover:text-primary p-0"><svg class="icon inline-flex shrink-0 size-6"><use href="#reply"></use></svg></button></div></div></div><div class="transition-all duration-200 border rounded-md p-2 border-transparent border-b-border last:border-b-transparent hover:border-secondary hover:border-b-secondary  "><div class="mb-2"><div class="flex justify-between items-start"><div class="flex items-center"><div class="mr-1"><div class="font-bold flex items-center"><a target="_blank" rel="noopener noreferrer" class="user-name relative inline-flex items-center gap-1" href="/en/profile/df601d40-4866-4b82-962d-edef79bc1236"><span>Billy</span><span title="Lv. 4" class="w_lv -mr-1" style="background-position: 0px -96px;"></span></a></div><div class="flex items-center"><ul class="rc-rate mr-2 rc-rate-disabled" tabindex="-1"><li class="rc-rate-star rc-rate-star-full"><div role="radio" aria-checked="true" aria-posinset="1" aria-setsize="5" tabindex="-1"><div class="rc-rate-star-first">★</div><div class="rc-rate-star-second">★</div></div></li><li class="rc-rate-star rc-rate-star-full"><div role="radio" aria-checked="true" aria-posinset="2" aria-setsize="5" tabindex="-1"><div class="rc-rate-star-first">★</div><div class="rc-rate-star-second">★</div></div></li><li class="rc-rate-star rc-rate-star-full"><div role="radio" aria-checked="true" aria-posinset="3" aria-setsize="5" tabindex="-1"><div class="rc-rate-star-first">★</div><div class="rc-rate-star-second">★</div></div></li><li class="rc-rate-star rc-rate-star-full"><div role="radio" aria-checked="true" aria-posinset="4" aria-setsize="5" tabindex="-1"><div class="rc-rate-star-first">★</div><div class="rc-rate-star-second">★</div></div></li><li class="rc-rate-star rc-rate-star-zero"><div role="radio" aria-checked="false" aria-posinset="5" aria-setsize="5" tabindex="-1"><div class="rc-rate-star-first">★</div><div class="rc-rate-star-second">★</div></div></li></ul></div></div></div><div class="flex flex-col text-right gap-1"><div class="text-muted-foreground text-sm flex items-center ml-auto"><span>Jul 13, 2024</span><button type="button" tabindex="0" data-slot="dropdown-menu-trigger" aria-haspopup="menu" id="base-ui-_r_1m0_" class="group/button inline-flex cursor-pointer shrink-0 items-center justify-center bg-clip-padding font-medium whitespace-nowrap transition-all outline-none select-none focus-visible:border-ring focus-visible:ring-3 focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-3 aria-invalid:ring-destructive/20 dark:aria-invalid:border-destructive/50 dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 border-border bg-card hover:bg-muted hover:text-foreground aria-expanded:bg-muted aria-expanded:text-foreground dark:border-input dark:bg-input/30 dark:hover:bg-input/50 h-7 gap-1 rounded-[min(var(--radius-md),12px)] text-[0.8rem] in-data-[slot=button-group]:rounded-lg has-data-[icon=inline-end]:pr-1.5 has-data-[icon=inline-start]:pl-1.5 [&amp;_svg:not([class*='size-'])]:size-3.5 p-0 border-0 ml-1" aria-expanded="false"><svg class="icon inline-flex shrink-0 size-6"><use href="#more"></use></svg></button></div><div class="flex items-center gap-1.5 text-[12px] ml-auto text-muted-foreground"><span>Read 311</span><span aria-hidden="true" class="select-none">|</span><span><b>Ch. #320</b></span></div></div></div></div><div class="mb-3 wrap-break-word text-base"><div class="tiptap-content text-base "><p>I hate the catch all human emperor/ sovereign/ king/ whatever that is somehow the undisputed best body cultivation technique ever but can never outperform the qi cultivation, always gets nerfed by any non human (beast, zombie, demon, etc), and can only work when the mc hands are empty cause why would you wield a sword with strength when you can can wield it it finesse. That the mc found easy as heck but nobody else will practice it but will still be able to match until mc kicks it into overgear(asspull).</p></div><button type="button" tabindex="0" data-slot="button" class="group/button cursor-pointer shrink-0 justify-center border-transparent bg-clip-padding font-medium whitespace-nowrap transition-all outline-none select-none focus-visible:border-ring focus-visible:ring-3 focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-3 aria-invalid:ring-destructive/20 dark:aria-invalid:border-destructive/50 dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 underline-offset-4 hover:underline h-7 rounded-[min(var(--radius-md),12px)] text-[0.8rem] in-data-[slot=button-group]:rounded-lg has-data-[icon=inline-end]:pr-1.5 has-data-[icon=inline-start]:pl-1.5 [&amp;_svg:not([class*='size-'])]:size-3.5 mt-1 flex items-center gap-1 border-0 p-0 text-muted-foreground"><svg class="icon inline-flex shrink-0 size-6"><use href="#g_translate"></use></svg><span>Translate to British English</span></button></div><div class="flex items-center [&amp;_.btn-sm]:min-h-[26px] [&amp;_.btn-sm]:min-w-[26px] [&amp;_svg]:size-5"><button type="button" tabindex="0" data-slot="button" class="group/button inline-flex cursor-pointer shrink-0 items-center justify-center border-transparent bg-clip-padding whitespace-nowrap transition-all outline-none select-none focus-visible:border-ring focus-visible:ring-3 focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-3 aria-invalid:ring-destructive/20 dark:aria-invalid:border-destructive/50 dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 underline-offset-4 hover:underline h-7 gap-1 rounded-[min(var(--radius-md),12px)] text-[0.8rem] in-data-[slot=button-group]:rounded-lg has-data-[icon=inline-end]:pr-1.5 has-data-[icon=inline-start]:pl-1.5 [&amp;_svg:not([class*='size-'])]:size-3.5 text-primary p-0 border-0 font-semibold uppercase"><svg class="icon inline-flex shrink-0 size-6 mr-1"><use href="#review"></use></svg><span>View 2 Replies</span></button><div class="flex items-center gap-2 ml-auto"><button type="button" tabindex="0" data-slot="button" class="group/button cursor-pointer shrink-0 justify-center border-transparent bg-clip-padding font-medium whitespace-nowrap transition-all outline-none select-none focus-visible:border-ring focus-visible:ring-3 focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-3 aria-invalid:ring-destructive/20 dark:aria-invalid:border-destructive/50 dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 h-7 gap-1 rounded-[min(var(--radius-md),12px)] text-[0.8rem] in-data-[slot=button-group]:rounded-lg has-data-[icon=inline-end]:pr-1.5 has-data-[icon=inline-start]:pl-1.5 [&amp;_svg:not([class*='size-'])]:size-3.5 text-muted-foreground hover:text-primary p-0 flex items-center"><svg class="icon inline-flex shrink-0 size-6"><use href="#premium"></use></svg></button><span class="like-button flex items-center cursor-pointer text-muted-foreground hover-primary-text" title="Like" style="cursor: pointer;"><svg class="icon inline-flex shrink-0 size-6 like"><use href="#like"></use></svg><span class="ml-1">1</span></span><button type="button" tabindex="0" data-slot="button" class="group/button cursor-pointer shrink-0 items-center justify-center border-transparent bg-clip-padding font-medium whitespace-nowrap transition-all outline-none select-none focus-visible:border-ring focus-visible:ring-3 focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-3 aria-invalid:ring-destructive/20 dark:aria-invalid:border-destructive/50 dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 flex h-7 gap-1 rounded-[min(var(--radius-md),12px)] text-[0.8rem] in-data-[slot=button-group]:rounded-lg has-data-[icon=inline-end]:pr-1.5 has-data-[icon=inline-start]:pl-1.5 [&amp;_svg:not([class*='size-'])]:size-3.5 text-muted-foreground hover:text-primary p-0"><svg class="icon inline-flex shrink-0 size-6"><use href="#reply"></use></svg></button></div></div></div><div class="transition-all duration-200 border rounded-md p-2 border-transparent border-b-border last:border-b-transparent hover:border-secondary hover:border-b-secondary  "><div class="mb-2"><div class="flex justify-between items-start"><div class="flex items-center"><div class="mr-1"><div class="font-bold flex items-center"><a target="_blank" rel="noopener noreferrer" class="user-name relative inline-flex items-center gap-1" href="/en/profile/51abbc69-bb1b-489b-8326-5d3acffd9d46"><span>Nano_Stat</span><span title="Lv. 3" class="w_lv -mr-1" style="background-position: 0px -72px;"></span></a></div><div class="flex items-center"><ul class="rc-rate mr-2 rc-rate-disabled" tabindex="-1"><li class="rc-rate-star rc-rate-star-full"><div role="radio" aria-checked="true" aria-posinset="1" aria-setsize="5" tabindex="-1"><div class="rc-rate-star-first">★</div><div class="rc-rate-star-second">★</div></div></li><li class="rc-rate-star rc-rate-star-full"><div role="radio" aria-checked="true" aria-posinset="2" aria-setsize="5" tabindex="-1"><div class="rc-rate-star-first">★</div><div class="rc-rate-star-second">★</div></div></li><li class="rc-rate-star rc-rate-star-full"><div role="radio" aria-checked="true" aria-posinset="3" aria-setsize="5" tabindex="-1"><div class="rc-rate-star-first">★</div><div class="rc-rate-star-second">★</div></div></li><li class="rc-rate-star rc-rate-star-full"><div role="radio" aria-checked="true" aria-posinset="4" aria-setsize="5" tabindex="-1"><div class="rc-rate-star-first">★</div><div class="rc-rate-star-second">★</div></div></li><li class="rc-rate-star rc-rate-star-full"><div role="radio" aria-checked="true" aria-posinset="5" aria-setsize="5" tabindex="-1"><div class="rc-rate-star-first">★</div><div class="rc-rate-star-second">★</div></div></li></ul></div></div></div><div class="flex flex-col text-right gap-1"><div class="text-muted-foreground text-sm flex items-center ml-auto"><span>Apr 14, 2026</span><button type="button" tabindex="0" data-slot="dropdown-menu-trigger" aria-haspopup="menu" id="base-ui-_r_1m6_" class="group/button inline-flex cursor-pointer shrink-0 items-center justify-center bg-clip-padding font-medium whitespace-nowrap transition-all outline-none select-none focus-visible:border-ring focus-visible:ring-3 focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-3 aria-invalid:ring-destructive/20 dark:aria-invalid:border-destructive/50 dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 border-border bg-card hover:bg-muted hover:text-foreground aria-expanded:bg-muted aria-expanded:text-foreground dark:border-input dark:bg-input/30 dark:hover:bg-input/50 h-7 gap-1 rounded-[min(var(--radius-md),12px)] text-[0.8rem] in-data-[slot=button-group]:rounded-lg has-data-[icon=inline-end]:pr-1.5 has-data-[icon=inline-start]:pl-1.5 [&amp;_svg:not([class*='size-'])]:size-3.5 p-0 border-0 ml-1" aria-expanded="false"><svg class="icon inline-flex shrink-0 size-6"><use href="#more"></use></svg></button></div><div class="flex items-center gap-1.5 text-[12px] ml-auto text-muted-foreground"><span>Read 3827</span><span aria-hidden="true" class="select-none">|</span><span><b>Ch. #3782</b></span></div></div></div></div><div class="mb-3 wrap-break-word text-base"><div class="tiptap-content text-base "><p>If you're looking for the same old Chinese cliche cultivation fantasy about human demon war, but with a fresh story, unique power system and NO plotholes. </p><p>THIS IS FOR YOU...</p></div><button type="button" tabindex="0" data-slot="button" class="group/button cursor-pointer shrink-0 justify-center border-transparent bg-clip-padding font-medium whitespace-nowrap transition-all outline-none select-none focus-visible:border-ring focus-visible:ring-3 focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-3 aria-invalid:ring-destructive/20 dark:aria-invalid:border-destructive/50 dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 underline-offset-4 hover:underline h-7 rounded-[min(var(--radius-md),12px)] text-[0.8rem] in-data-[slot=button-group]:rounded-lg has-data-[icon=inline-end]:pr-1.5 has-data-[icon=inline-start]:pl-1.5 [&amp;_svg:not([class*='size-'])]:size-3.5 mt-1 flex items-center gap-1 border-0 p-0 text-muted-foreground"><svg class="icon inline-flex shrink-0 size-6"><use href="#g_translate"></use></svg><span>Translate to British English</span></button></div><div class="flex items-center [&amp;_.btn-sm]:min-h-[26px] [&amp;_.btn-sm]:min-w-[26px] [&amp;_svg]:size-5"><button type="button" tabindex="0" data-slot="button" class="group/button inline-flex cursor-pointer shrink-0 items-center justify-center border-transparent bg-clip-padding whitespace-nowrap transition-all outline-none select-none focus-visible:border-ring focus-visible:ring-3 focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-3 aria-invalid:ring-destructive/20 dark:aria-invalid:border-destructive/50 dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 underline-offset-4 hover:underline h-7 gap-1 rounded-[min(var(--radius-md),12px)] text-[0.8rem] in-data-[slot=button-group]:rounded-lg has-data-[icon=inline-end]:pr-1.5 has-data-[icon=inline-start]:pl-1.5 [&amp;_svg:not([class*='size-'])]:size-3.5 text-primary p-0 border-0 font-semibold uppercase"><svg class="icon inline-flex shrink-0 size-6 mr-1"><use href="#review"></use></svg><span>View 3 Replies</span></button><div class="flex items-center gap-2 ml-auto"><button type="button" tabindex="0" data-slot="button" class="group/button cursor-pointer shrink-0 justify-center border-transparent bg-clip-padding font-medium whitespace-nowrap transition-all outline-none select-none focus-visible:border-ring focus-visible:ring-3 focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-3 aria-invalid:ring-destructive/20 dark:aria-invalid:border-destructive/50 dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 h-7 gap-1 rounded-[min(var(--radius-md),12px)] text-[0.8rem] in-data-[slot=button-group]:rounded-lg has-data-[icon=inline-end]:pr-1.5 has-data-[icon=inline-start]:pl-1.5 [&amp;_svg:not([class*='size-'])]:size-3.5 text-muted-foreground hover:text-primary p-0 flex items-center"><svg class="icon inline-flex shrink-0 size-6"><use href="#premium"></use></svg></button><span class="like-button flex items-center cursor-pointer text-muted-foreground hover-primary-text" title="Like" style="cursor: pointer;"><svg class="icon inline-flex shrink-0 size-6 like"><use href="#like"></use></svg></span><button type="button" tabindex="0" data-slot="button" class="group/button cursor-pointer shrink-0 items-center justify-center border-transparent bg-clip-padding font-medium whitespace-nowrap transition-all outline-none select-none focus-visible:border-ring focus-visible:ring-3 focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-3 aria-invalid:ring-destructive/20 dark:aria-invalid:border-destructive/50 dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 flex h-7 gap-1 rounded-[min(var(--radius-md),12px)] text-[0.8rem] in-data-[slot=button-group]:rounded-lg has-data-[icon=inline-end]:pr-1.5 has-data-[icon=inline-start]:pl-1.5 [&amp;_svg:not([class*='size-'])]:size-3.5 text-muted-foreground hover:text-primary p-0"><svg class="icon inline-flex shrink-0 size-6"><use href="#reply"></use></svg></button></div></div></div><div class="transition-all duration-200 border rounded-md p-2 border-transparent border-b-border last:border-b-transparent hover:border-secondary hover:border-b-secondary  "><div class="mb-2"><div class="flex justify-between items-start"><div class="flex items-center"><div class="mr-1"><div class="font-bold flex items-center"><a target="_blank" rel="noopener noreferrer" class="user-name relative inline-flex items-center gap-1" href="/en/profile/196513ca-7bff-4516-99c3-e2e5cc289d22"><span>Alto</span><span title="Lv. 2" class="w_lv -mr-1" style="background-position: 0px -48px;"></span></a></div><div class="flex items-center"><ul class="rc-rate mr-2 rc-rate-disabled" tabindex="-1"><li class="rc-rate-star rc-rate-star-full"><div role="radio" aria-checked="true" aria-posinset="1" aria-setsize="5" tabindex="-1"><div class="rc-rate-star-first">★</div><div class="rc-rate-star-second">★</div></div></li><li class="rc-rate-star rc-rate-star-full"><div role="radio" aria-checked="true" aria-posinset="2" aria-setsize="5" tabindex="-1"><div class="rc-rate-star-first">★</div><div class="rc-rate-star-second">★</div></div></li><li class="rc-rate-star rc-rate-star-full"><div role="radio" aria-checked="true" aria-posinset="3" aria-setsize="5" tabindex="-1"><div class="rc-rate-star-first">★</div><div class="rc-rate-star-second">★</div></div></li><li class="rc-rate-star rc-rate-star-full"><div role="radio" aria-checked="true" aria-posinset="4" aria-setsize="5" tabindex="-1"><div class="rc-rate-star-first">★</div><div class="rc-rate-star-second">★</div></div></li><li class="rc-rate-star rc-rate-star-zero"><div role="radio" aria-checked="false" aria-posinset="5" aria-setsize="5" tabindex="-1"><div class="rc-rate-star-first">★</div><div class="rc-rate-star-second">★</div></div></li></ul></div></div></div><div class="flex flex-col text-right gap-1"><div class="text-muted-foreground text-sm flex items-center ml-auto"><span>Feb 22, 2026</span><button type="button" tabindex="0" data-slot="dropdown-menu-trigger" aria-haspopup="menu" id="base-ui-_r_1mc_" class="group/button inline-flex cursor-pointer shrink-0 items-center justify-center bg-clip-padding font-medium whitespace-nowrap transition-all outline-none select-none focus-visible:border-ring focus-visible:ring-3 focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-3 aria-invalid:ring-destructive/20 dark:aria-invalid:border-destructive/50 dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 border-border bg-card hover:bg-muted hover:text-foreground aria-expanded:bg-muted aria-expanded:text-foreground dark:border-input dark:bg-input/30 dark:hover:bg-input/50 h-7 gap-1 rounded-[min(var(--radius-md),12px)] text-[0.8rem] in-data-[slot=button-group]:rounded-lg has-data-[icon=inline-end]:pr-1.5 has-data-[icon=inline-start]:pl-1.5 [&amp;_svg:not([class*='size-'])]:size-3.5 p-0 border-0 ml-1" aria-expanded="false"><svg class="icon inline-flex shrink-0 size-6"><use href="#more"></use></svg></button></div><div class="flex items-center gap-1.5 text-[12px] ml-auto text-muted-foreground"><span>Read 1042</span><span aria-hidden="true" class="select-none">|</span><span><b>Ch. #3270</b></span></div></div></div></div><div class="mb-3 wrap-break-word text-base"></div><div class="flex items-center [&amp;_.btn-sm]:min-h-[26px] [&amp;_.btn-sm]:min-w-[26px] [&amp;_svg]:size-5"><button type="button" tabindex="0" data-slot="button" class="group/button inline-flex cursor-pointer shrink-0 items-center justify-center border-transparent bg-clip-padding whitespace-nowrap transition-all outline-none select-none focus-visible:border-ring focus-visible:ring-3 focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-3 aria-invalid:ring-destructive/20 dark:aria-invalid:border-destructive/50 dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 underline-offset-4 hover:underline h-7 gap-1 rounded-[min(var(--radius-md),12px)] text-[0.8rem] in-data-[slot=button-group]:rounded-lg has-data-[icon=inline-end]:pr-1.5 has-data-[icon=inline-start]:pl-1.5 [&amp;_svg:not([class*='size-'])]:size-3.5 text-primary p-0 border-0 font-semibold uppercase"><svg class="icon inline-flex shrink-0 size-6 mr-1"><use href="#review"></use></svg><span>View 0 Replies</span></button><div class="flex items-center gap-2 ml-auto"><button type="button" tabindex="0" data-slot="button" class="group/button cursor-pointer shrink-0 justify-center border-transparent bg-clip-padding font-medium whitespace-nowrap transition-all outline-none select-none focus-visible:border-ring focus-visible:ring-3 focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-3 aria-invalid:ring-destructive/20 dark:aria-invalid:border-destructive/50 dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 h-7 gap-1 rounded-[min(var(--radius-md),12px)] text-[0.8rem] in-data-[slot=button-group]:rounded-lg has-data-[icon=inline-end]:pr-1.5 has-data-[icon=inline-start]:pl-1.5 [&amp;_svg:not([class*='size-'])]:size-3.5 text-muted-foreground hover:text-primary p-0 flex items-center"><svg class="icon inline-flex shrink-0 size-6"><use href="#premium"></use></svg></button><span class="like-button flex items-center cursor-pointer text-muted-foreground hover-primary-text" title="Like" style="cursor: pointer;"><svg class="icon inline-flex shrink-0 size-6 like"><use href="#like"></use></svg></span><button type="button" tabindex="0" data-slot="button" class="group/button cursor-pointer shrink-0 items-center justify-center border-transparent bg-clip-padding font-medium whitespace-nowrap transition-all outline-none select-none focus-visible:border-ring focus-visible:ring-3 focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-3 aria-invalid:ring-destructive/20 dark:aria-invalid:border-destructive/50 dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 flex h-7 gap-1 rounded-[min(var(--radius-md),12px)] text-[0.8rem] in-data-[slot=button-group]:rounded-lg has-data-[icon=inline-end]:pr-1.5 has-data-[icon=inline-start]:pl-1.5 [&amp;_svg:not([class*='size-'])]:size-3.5 text-muted-foreground hover:text-primary p-0"><svg class="icon inline-flex shrink-0 size-6"><use href="#reply"></use></svg></button></div></div></div><div class="transition-all duration-200 border rounded-md p-2 border-transparent border-b-border last:border-b-transparent hover:border-secondary hover:border-b-secondary  "><div class="mb-2"><div class="flex justify-between items-start"><div class="flex items-center"><div class="mr-1"><div class="font-bold flex items-center"><a target="_blank" rel="noopener noreferrer" class="user-name relative inline-flex items-center gap-1" href="/en/profile/663c59b5-7242-4a92-8818-80b4b86a86c2"><span>Zaar</span><span title="Lv. 2" class="w_lv -mr-1" style="background-position: 0px -48px;"></span></a></div><div class="flex items-center"><ul class="rc-rate mr-2 rc-rate-disabled" tabindex="-1"><li class="rc-rate-star rc-rate-star-full"><div role="radio" aria-checked="true" aria-posinset="1" aria-setsize="5" tabindex="-1"><div class="rc-rate-star-first">★</div><div class="rc-rate-star-second">★</div></div></li><li class="rc-rate-star rc-rate-star-full"><div role="radio" aria-checked="true" aria-posinset="2" aria-setsize="5" tabindex="-1"><div class="rc-rate-star-first">★</div><div class="rc-rate-star-second">★</div></div></li><li class="rc-rate-star rc-rate-star-full"><div role="radio" aria-checked="true" aria-posinset="3" aria-setsize="5" tabindex="-1"><div class="rc-rate-star-first">★</div><div class="rc-rate-star-second">★</div></div></li><li class="rc-rate-star rc-rate-star-full"><div role="radio" aria-checked="true" aria-posinset="4" aria-setsize="5" tabindex="-1"><div class="rc-rate-star-first">★</div><div class="rc-rate-star-second">★</div></div></li><li class="rc-rate-star rc-rate-star-full"><div role="radio" aria-checked="true" aria-posinset="5" aria-setsize="5" tabindex="-1"><div class="rc-rate-star-first">★</div><div class="rc-rate-star-second">★</div></div></li></ul></div></div></div><div class="flex flex-col text-right gap-1"><div class="text-muted-foreground text-sm flex items-center ml-auto"><span>Dec 18, 2025</span><button type="button" tabindex="0" data-slot="dropdown-menu-trigger" aria-haspopup="menu" id="base-ui-_r_1mi_" class="group/button inline-flex cursor-pointer shrink-0 items-center justify-center bg-clip-padding font-medium whitespace-nowrap transition-all outline-none select-none focus-visible:border-ring focus-visible:ring-3 focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-3 aria-invalid:ring-destructive/20 dark:aria-invalid:border-destructive/50 dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 border-border bg-card hover:bg-muted hover:text-foreground aria-expanded:bg-muted aria-expanded:text-foreground dark:border-input dark:bg-input/30 dark:hover:bg-input/50 h-7 gap-1 rounded-[min(var(--radius-md),12px)] text-[0.8rem] in-data-[slot=button-group]:rounded-lg has-data-[icon=inline-end]:pr-1.5 has-data-[icon=inline-start]:pl-1.5 [&amp;_svg:not([class*='size-'])]:size-3.5 p-0 border-0 ml-1" aria-expanded="false"><svg class="icon inline-flex shrink-0 size-6"><use href="#more"></use></svg></button></div><div class="flex items-center gap-1.5 text-[12px] ml-auto text-muted-foreground"><span>Read 6016</span><span aria-hidden="true" class="select-none">|</span><span><b>Ch. #3900</b></span></div></div></div></div><div class="mb-3 wrap-break-word text-base"></div><div class="flex items-center [&amp;_.btn-sm]:min-h-[26px] [&amp;_.btn-sm]:min-w-[26px] [&amp;_svg]:size-5"><button type="button" tabindex="0" data-slot="button" class="group/button inline-flex cursor-pointer shrink-0 items-center justify-center border-transparent bg-clip-padding whitespace-nowrap transition-all outline-none select-none focus-visible:border-ring focus-visible:ring-3 focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-3 aria-invalid:ring-destructive/20 dark:aria-invalid:border-destructive/50 dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 underline-offset-4 hover:underline h-7 gap-1 rounded-[min(var(--radius-md),12px)] text-[0.8rem] in-data-[slot=button-group]:rounded-lg has-data-[icon=inline-end]:pr-1.5 has-data-[icon=inline-start]:pl-1.5 [&amp;_svg:not([class*='size-'])]:size-3.5 text-primary p-0 border-0 font-semibold uppercase"><svg class="icon inline-flex shrink-0 size-6 mr-1"><use href="#review"></use></svg><span>View 1 Reply</span></button><div class="flex items-center gap-2 ml-auto"><button type="button" tabindex="0" data-slot="button" class="group/button cursor-pointer shrink-0 justify-center border-transparent bg-clip-padding font-medium whitespace-nowrap transition-all outline-none select-none focus-visible:border-ring focus-visible:ring-3 focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-3 aria-invalid:ring-destructive/20 dark:aria-invalid:border-destructive/50 dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 h-7 gap-1 rounded-[min(var(--radius-md),12px)] text-[0.8rem] in-data-[slot=button-group]:rounded-lg has-data-[icon=inline-end]:pr-1.5 has-data-[icon=inline-start]:pl-1.5 [&amp;_svg:not([class*='size-'])]:size-3.5 text-muted-foreground hover:text-primary p-0 flex items-center"><svg class="icon inline-flex shrink-0 size-6"><use href="#premium"></use></svg></button><span class="like-button flex items-center cursor-pointer text-muted-foreground hover-primary-text" title="Like" style="cursor: pointer;"><svg class="icon inline-flex shrink-0 size-6 like"><use href="#like"></use></svg></span><button type="button" tabindex="0" data-slot="button" class="group/button cursor-pointer shrink-0 items-center justify-center border-transparent bg-clip-padding font-medium whitespace-nowrap transition-all outline-none select-none focus-visible:border-ring focus-visible:ring-3 focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-3 aria-invalid:ring-destructive/20 dark:aria-invalid:border-destructive/50 dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 flex h-7 gap-1 rounded-[min(var(--radius-md),12px)] text-[0.8rem] in-data-[slot=button-group]:rounded-lg has-data-[icon=inline-end]:pr-1.5 has-data-[icon=inline-start]:pl-1.5 [&amp;_svg:not([class*='size-'])]:size-3.5 text-muted-foreground hover:text-primary p-0"><svg class="icon inline-flex shrink-0 size-6"><use href="#reply"></use></svg></button></div></div></div><div class="transition-all duration-200 border rounded-md p-2 border-transparent border-b-border last:border-b-transparent hover:border-secondary hover:border-b-secondary  "><div class="mb-2"><div class="flex justify-between items-start"><div class="flex items-center"><div class="mr-1"><div class="font-bold flex items-center"><a target="_blank" rel="noopener noreferrer" class="user-name relative inline-flex items-center gap-1" href="/en/profile/bf7a2829-4cc9-490d-ab50-6778aa17272f"><span>LongevityHater</span><span title="Lv. 2" class="w_lv -mr-1" style="background-position: 0px -48px;"></span></a></div><div class="flex items-center"><ul class="rc-rate mr-2 rc-rate-disabled" tabindex="-1"><li class="rc-rate-star rc-rate-star-full"><div role="radio" aria-checked="true" aria-posinset="1" aria-setsize="5" tabindex="-1"><div class="rc-rate-star-first">★</div><div class="rc-rate-star-second">★</div></div></li><li class="rc-rate-star rc-rate-star-full"><div role="radio" aria-checked="true" aria-posinset="2" aria-setsize="5" tabindex="-1"><div class="rc-rate-star-first">★</div><div class="rc-rate-star-second">★</div></div></li><li class="rc-rate-star rc-rate-star-full"><div role="radio" aria-checked="true" aria-posinset="3" aria-setsize="5" tabindex="-1"><div class="rc-rate-star-first">★</div><div class="rc-rate-star-second">★</div></div></li><li class="rc-rate-star rc-rate-star-full"><div role="radio" aria-checked="true" aria-posinset="4" aria-setsize="5" tabindex="-1"><div class="rc-rate-star-first">★</div><div class="rc-rate-star-second">★</div></div></li><li class="rc-rate-star rc-rate-star-full"><div role="radio" aria-checked="true" aria-posinset="5" aria-setsize="5" tabindex="-1"><div class="rc-rate-star-first">★</div><div class="rc-rate-star-second">★</div></div></li></ul></div></div></div><div class="flex flex-col text-right gap-1"><div class="text-muted-foreground text-sm flex items-center ml-auto"><span>Oct 15, 2024</span><button type="button" tabindex="0" data-slot="dropdown-menu-trigger" aria-haspopup="menu" id="base-ui-_r_1mo_" class="group/button inline-flex cursor-pointer shrink-0 items-center justify-center bg-clip-padding font-medium whitespace-nowrap transition-all outline-none select-none focus-visible:border-ring focus-visible:ring-3 focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-3 aria-invalid:ring-destructive/20 dark:aria-invalid:border-destructive/50 dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 border-border bg-card hover:bg-muted hover:text-foreground aria-expanded:bg-muted aria-expanded:text-foreground dark:border-input dark:bg-input/30 dark:hover:bg-input/50 h-7 gap-1 rounded-[min(var(--radius-md),12px)] text-[0.8rem] in-data-[slot=button-group]:rounded-lg has-data-[icon=inline-end]:pr-1.5 has-data-[icon=inline-start]:pl-1.5 [&amp;_svg:not([class*='size-'])]:size-3.5 p-0 border-0 ml-1" aria-expanded="false"><svg class="icon inline-flex shrink-0 size-6"><use href="#more"></use></svg></button></div><div class="flex items-center gap-1.5 text-[12px] ml-auto text-muted-foreground"><span>Read 5</span><span aria-hidden="true" class="select-none">|</span><span><b>Ch. #3</b></span></div></div></div></div><div class="mb-3 wrap-break-word text-base"></div><div class="flex items-center [&amp;_.btn-sm]:min-h-[26px] [&amp;_.btn-sm]:min-w-[26px] [&amp;_svg]:size-5"><button type="button" tabindex="0" data-slot="button" class="group/button inline-flex cursor-pointer shrink-0 items-center justify-center border-transparent bg-clip-padding whitespace-nowrap transition-all outline-none select-none focus-visible:border-ring focus-visible:ring-3 focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-3 aria-invalid:ring-destructive/20 dark:aria-invalid:border-destructive/50 dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 underline-offset-4 hover:underline h-7 gap-1 rounded-[min(var(--radius-md),12px)] text-[0.8rem] in-data-[slot=button-group]:rounded-lg has-data-[icon=inline-end]:pr-1.5 has-data-[icon=inline-start]:pl-1.5 [&amp;_svg:not([class*='size-'])]:size-3.5 text-primary p-0 border-0 font-semibold uppercase"><svg class="icon inline-flex shrink-0 size-6 mr-1"><use href="#review"></use></svg><span>View 1 Reply</span></button><div class="flex items-center gap-2 ml-auto"><button type="button" tabindex="0" data-slot="button" class="group/button cursor-pointer shrink-0 justify-center border-transparent bg-clip-padding font-medium whitespace-nowrap transition-all outline-none select-none focus-visible:border-ring focus-visible:ring-3 focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-3 aria-invalid:ring-destructive/20 dark:aria-invalid:border-destructive/50 dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 h-7 gap-1 rounded-[min(var(--radius-md),12px)] text-[0.8rem] in-data-[slot=button-group]:rounded-lg has-data-[icon=inline-end]:pr-1.5 has-data-[icon=inline-start]:pl-1.5 [&amp;_svg:not([class*='size-'])]:size-3.5 text-muted-foreground hover:text-primary p-0 flex items-center"><svg class="icon inline-flex shrink-0 size-6"><use href="#premium"></use></svg></button><span class="like-button flex items-center cursor-pointer text-muted-foreground hover-primary-text" title="Like" style="cursor: pointer;"><svg class="icon inline-flex shrink-0 size-6 like"><use href="#like"></use></svg></span><button type="button" tabindex="0" data-slot="button" class="group/button cursor-pointer shrink-0 items-center justify-center border-transparent bg-clip-padding font-medium whitespace-nowrap transition-all outline-none select-none focus-visible:border-ring focus-visible:ring-3 focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-3 aria-invalid:ring-destructive/20 dark:aria-invalid:border-destructive/50 dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 flex h-7 gap-1 rounded-[min(var(--radius-md),12px)] text-[0.8rem] in-data-[slot=button-group]:rounded-lg has-data-[icon=inline-end]:pr-1.5 has-data-[icon=inline-start]:pl-1.5 [&amp;_svg:not([class*='size-'])]:size-3.5 text-muted-foreground hover:text-primary p-0"><svg class="icon inline-flex shrink-0 size-6"><use href="#reply"></use></svg></button></div></div></div><div class="transition-all duration-200 border rounded-md p-2 border-transparent border-b-border last:border-b-transparent hover:border-secondary hover:border-b-secondary  "><div class="mb-2"><div class="flex justify-between items-start"><div class="flex items-center"><div class="mr-1"><div class="font-bold flex items-center"><a target="_blank" rel="noopener noreferrer" class="user-name relative inline-flex items-center gap-1" href="/en/profile/be70142c-bcc5-4687-a43c-c4f31c1e44de"><span>Momonomo</span><span title="Lv. 3" class="w_lv -mr-1" style="background-position: 0px -72px;"></span></a></div><div class="flex items-center"><ul class="rc-rate mr-2 rc-rate-disabled" tabindex="-1"><li class="rc-rate-star rc-rate-star-full"><div role="radio" aria-checked="true" aria-posinset="1" aria-setsize="5" tabindex="-1"><div class="rc-rate-star-first">★</div><div class="rc-rate-star-second">★</div></div></li><li class="rc-rate-star rc-rate-star-full"><div role="radio" aria-checked="true" aria-posinset="2" aria-setsize="5" tabindex="-1"><div class="rc-rate-star-first">★</div><div class="rc-rate-star-second">★</div></div></li><li class="rc-rate-star rc-rate-star-full"><div role="radio" aria-checked="true" aria-posinset="3" aria-setsize="5" tabindex="-1"><div class="rc-rate-star-first">★</div><div class="rc-rate-star-second">★</div></div></li><li class="rc-rate-star rc-rate-star-full"><div role="radio" aria-checked="true" aria-posinset="4" aria-setsize="5" tabindex="-1"><div class="rc-rate-star-first">★</div><div class="rc-rate-star-second">★</div></div></li><li class="rc-rate-star rc-rate-star-full"><div role="radio" aria-checked="true" aria-posinset="5" aria-setsize="5" tabindex="-1"><div class="rc-rate-star-first">★</div><div class="rc-rate-star-second">★</div></div></li></ul></div></div></div><div class="flex flex-col text-right gap-1"><div class="text-muted-foreground text-sm flex items-center ml-auto"><span>Jun 6, 2024</span><button type="button" tabindex="0" data-slot="dropdown-menu-trigger" aria-haspopup="menu" id="base-ui-_r_1mu_" class="group/button inline-flex cursor-pointer shrink-0 items-center justify-center bg-clip-padding font-medium whitespace-nowrap transition-all outline-none select-none focus-visible:border-ring focus-visible:ring-3 focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-3 aria-invalid:ring-destructive/20 dark:aria-invalid:border-destructive/50 dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 border-border bg-card hover:bg-muted hover:text-foreground aria-expanded:bg-muted aria-expanded:text-foreground dark:border-input dark:bg-input/30 dark:hover:bg-input/50 h-7 gap-1 rounded-[min(var(--radius-md),12px)] text-[0.8rem] in-data-[slot=button-group]:rounded-lg has-data-[icon=inline-end]:pr-1.5 has-data-[icon=inline-start]:pl-1.5 [&amp;_svg:not([class*='size-'])]:size-3.5 p-0 border-0 ml-1" aria-expanded="false"><svg class="icon inline-flex shrink-0 size-6"><use href="#more"></use></svg></button></div><div class="flex items-center gap-1.5 text-[12px] ml-auto text-muted-foreground"><span>Read 11</span><span aria-hidden="true" class="select-none">|</span><span><b>Ch. #512</b></span></div></div></div></div><div class="mb-3 wrap-break-word text-base"></div><div class="flex items-center [&amp;_.btn-sm]:min-h-[26px] [&amp;_.btn-sm]:min-w-[26px] [&amp;_svg]:size-5"><button type="button" tabindex="0" data-slot="button" class="group/button inline-flex cursor-pointer shrink-0 items-center justify-center border-transparent bg-clip-padding whitespace-nowrap transition-all outline-none select-none focus-visible:border-ring focus-visible:ring-3 focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-3 aria-invalid:ring-destructive/20 dark:aria-invalid:border-destructive/50 dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 underline-offset-4 hover:underline h-7 gap-1 rounded-[min(var(--radius-md),12px)] text-[0.8rem] in-data-[slot=button-group]:rounded-lg has-data-[icon=inline-end]:pr-1.5 has-data-[icon=inline-start]:pl-1.5 [&amp;_svg:not([class*='size-'])]:size-3.5 text-primary p-0 border-0 font-semibold uppercase"><svg class="icon inline-flex shrink-0 size-6 mr-1"><use href="#review"></use></svg><span>View 0 Replies</span></button><div class="flex items-center gap-2 ml-auto"><button type="button" tabindex="0" data-slot="button" class="group/button cursor-pointer shrink-0 justify-center border-transparent bg-clip-padding font-medium whitespace-nowrap transition-all outline-none select-none focus-visible:border-ring focus-visible:ring-3 focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-3 aria-invalid:ring-destructive/20 dark:aria-invalid:border-destructive/50 dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 h-7 gap-1 rounded-[min(var(--radius-md),12px)] text-[0.8rem] in-data-[slot=button-group]:rounded-lg has-data-[icon=inline-end]:pr-1.5 has-data-[icon=inline-start]:pl-1.5 [&amp;_svg:not([class*='size-'])]:size-3.5 text-muted-foreground hover:text-primary p-0 flex items-center"><svg class="icon inline-flex shrink-0 size-6"><use href="#premium"></use></svg></button><span class="like-button flex items-center cursor-pointer text-muted-foreground hover-primary-text" title="Like" style="cursor: pointer;"><svg class="icon inline-flex shrink-0 size-6 like"><use href="#like"></use></svg></span><button type="button" tabindex="0" data-slot="button" class="group/button cursor-pointer shrink-0 items-center justify-center border-transparent bg-clip-padding font-medium whitespace-nowrap transition-all outline-none select-none focus-visible:border-ring focus-visible:ring-3 focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-3 aria-invalid:ring-destructive/20 dark:aria-invalid:border-destructive/50 dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 flex h-7 gap-1 rounded-[min(var(--radius-md),12px)] text-[0.8rem] in-data-[slot=button-group]:rounded-lg has-data-[icon=inline-end]:pr-1.5 has-data-[icon=inline-start]:pl-1.5 [&amp;_svg:not([class*='size-'])]:size-3.5 text-muted-foreground hover:text-primary p-0"><svg class="icon inline-flex shrink-0 size-6"><use href="#reply"></use></svg></button></div></div></div><div class="transition-all duration-200 border rounded-md p-2 border-transparent border-b-border last:border-b-transparent hover:border-secondary hover:border-b-secondary  "><div class="mb-2"><div class="flex justify-between items-start"><div class="flex items-center"><div class="mr-1"><div class="font-bold flex items-center"><a target="_blank" rel="noopener noreferrer" class="user-name relative inline-flex items-center gap-1" href="/en/profile/7e594b50-f444-4a3f-81f6-e2743b4e4d99"><span>Alcibar19</span><span title="Lv. 5" class="w_lv -mr-1" style="background-position: 0px -120px;"></span></a></div><div class="flex items-center"><ul class="rc-rate mr-2 rc-rate-disabled" tabindex="-1"><li class="rc-rate-star rc-rate-star-full"><div role="radio" aria-checked="true" aria-posinset="1" aria-setsize="5" tabindex="-1"><div class="rc-rate-star-first">★</div><div class="rc-rate-star-second">★</div></div></li><li class="rc-rate-star rc-rate-star-full"><div role="radio" aria-checked="true" aria-posinset="2" aria-setsize="5" tabindex="-1"><div class="rc-rate-star-first">★</div><div class="rc-rate-star-second">★</div></div></li><li class="rc-rate-star rc-rate-star-full"><div role="radio" aria-checked="true" aria-posinset="3" aria-setsize="5" tabindex="-1"><div class="rc-rate-star-first">★</div><div class="rc-rate-star-second">★</div></div></li><li class="rc-rate-star rc-rate-star-zero"><div role="radio" aria-checked="false" aria-posinset="4" aria-setsize="5" tabindex="-1"><div class="rc-rate-star-first">★</div><div class="rc-rate-star-second">★</div></div></li><li class="rc-rate-star rc-rate-star-zero"><div role="radio" aria-checked="false" aria-posinset="5" aria-setsize="5" tabindex="-1"><div class="rc-rate-star-first">★</div><div class="rc-rate-star-second">★</div></div></li></ul></div></div></div><div class="flex flex-col text-right gap-1"><div class="text-muted-foreground text-sm flex items-center ml-auto"><span>Jun 6, 2024</span><button type="button" tabindex="0" data-slot="dropdown-menu-trigger" aria-haspopup="menu" id="base-ui-_r_1n4_" class="group/button inline-flex cursor-pointer shrink-0 items-center justify-center bg-clip-padding font-medium whitespace-nowrap transition-all outline-none select-none focus-visible:border-ring focus-visible:ring-3 focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-3 aria-invalid:ring-destructive/20 dark:aria-invalid:border-destructive/50 dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 border-border bg-card hover:bg-muted hover:text-foreground aria-expanded:bg-muted aria-expanded:text-foreground dark:border-input dark:bg-input/30 dark:hover:bg-input/50 h-7 gap-1 rounded-[min(var(--radius-md),12px)] text-[0.8rem] in-data-[slot=button-group]:rounded-lg has-data-[icon=inline-end]:pr-1.5 has-data-[icon=inline-start]:pl-1.5 [&amp;_svg:not([class*='size-'])]:size-3.5 p-0 border-0 ml-1" aria-expanded="false"><svg class="icon inline-flex shrink-0 size-6"><use href="#more"></use></svg></button></div><div class="flex items-center gap-1.5 text-[12px] ml-auto text-muted-foreground"><span>Read 9</span><span aria-hidden="true" class="select-none">|</span><span><b>Ch. #9</b></span></div></div></div></div><div class="mb-3 wrap-break-word text-base"></div><div class="flex items-center [&amp;_.btn-sm]:min-h-[26px] [&amp;_.btn-sm]:min-w-[26px] [&amp;_svg]:size-5"><button type="button" tabindex="0" data-slot="button" class="group/button inline-flex cursor-pointer shrink-0 items-center justify-center border-transparent bg-clip-padding whitespace-nowrap transition-all outline-none select-none focus-visible:border-ring focus-visible:ring-3 focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-3 aria-invalid:ring-destructive/20 dark:aria-invalid:border-destructive/50 dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 underline-offset-4 hover:underline h-7 gap-1 rounded-[min(var(--radius-md),12px)] text-[0.8rem] in-data-[slot=button-group]:rounded-lg has-data-[icon=inline-end]:pr-1.5 has-data-[icon=inline-start]:pl-1.5 [&amp;_svg:not([class*='size-'])]:size-3.5 text-primary p-0 border-0 font-semibold uppercase"><svg class="icon inline-flex shrink-0 size-6 mr-1"><use href="#review"></use></svg><span>View 0 Replies</span></button><div class="flex items-center gap-2 ml-auto"><button type="button" tabindex="0" data-slot="button" class="group/button cursor-pointer shrink-0 justify-center border-transparent bg-clip-padding font-medium whitespace-nowrap transition-all outline-none select-none focus-visible:border-ring focus-visible:ring-3 focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-3 aria-invalid:ring-destructive/20 dark:aria-invalid:border-destructive/50 dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 h-7 gap-1 rounded-[min(var(--radius-md),12px)] text-[0.8rem] in-data-[slot=button-group]:rounded-lg has-data-[icon=inline-end]:pr-1.5 has-data-[icon=inline-start]:pl-1.5 [&amp;_svg:not([class*='size-'])]:size-3.5 text-muted-foreground hover:text-primary p-0 flex items-center"><svg class="icon inline-flex shrink-0 size-6"><use href="#premium"></use></svg></button><span class="like-button flex items-center cursor-pointer text-muted-foreground hover-primary-text" title="Like" style="cursor: pointer;"><svg class="icon inline-flex shrink-0 size-6 like"><use href="#like"></use></svg></span><button type="button" tabindex="0" data-slot="button" class="group/button cursor-pointer shrink-0 items-center justify-center border-transparent bg-clip-padding font-medium whitespace-nowrap transition-all outline-none select-none focus-visible:border-ring focus-visible:ring-3 focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-3 aria-invalid:ring-destructive/20 dark:aria-invalid:border-destructive/50 dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 flex h-7 gap-1 rounded-[min(var(--radius-md),12px)] text-[0.8rem] in-data-[slot=button-group]:rounded-lg has-data-[icon=inline-end]:pr-1.5 has-data-[icon=inline-start]:pl-1.5 [&amp;_svg:not([class*='size-'])]:size-3.5 text-muted-foreground hover:text-primary p-0"><svg class="icon inline-flex shrink-0 size-6"><use href="#reply"></use></svg></button></div></div></div></div></div><div data-slot="card" data-size="default" data-variant="default" class="group/card flex flex-col gap-0 bg-card text-sm text-card-foreground has-[&gt;img:first-child]:pt-0 *:[img:first-child]:rounded-t-xl *:[img:last-child]:rounded-b-xl shadow rounded-md ring-1 ring-foreground/20 mt-2"><div data-slot="card-content" class="p-4"><h3 class="text-lg font-semibold mb-2"><span>Leave a Review</span></h3><ul class="rc-rate" tabindex="0"><li class="rc-rate-star rc-rate-star-zero"><div role="radio" aria-checked="false" aria-posinset="1" aria-setsize="5" tabindex="0"><div class="rc-rate-star-first">★</div><div class="rc-rate-star-second">★</div></div></li><li class="rc-rate-star rc-rate-star-zero"><div role="radio" aria-checked="false" aria-posinset="2" aria-setsize="5" tabindex="0"><div class="rc-rate-star-first">★</div><div class="rc-rate-star-second">★</div></div></li><li class="rc-rate-star rc-rate-star-zero"><div role="radio" aria-checked="false" aria-posinset="3" aria-setsize="5" tabindex="0"><div class="rc-rate-star-first">★</div><div class="rc-rate-star-second">★</div></div></li><li class="rc-rate-star rc-rate-star-zero"><div role="radio" aria-checked="false" aria-posinset="4" aria-setsize="5" tabindex="0"><div class="rc-rate-star-first">★</div><div class="rc-rate-star-second">★</div></div></li><li class="rc-rate-star rc-rate-star-zero"><div role="radio" aria-checked="false" aria-posinset="5" aria-setsize="5" tabindex="0"><div class="rc-rate-star-first">★</div><div class="rc-rate-star-second">★</div></div></li></ul><div class="mt-3"><label class="block text-sm font-medium mb-1">Write your review: (Optional)</label><div class="tiptap-wrapper "><div><div contenteditable="true" placeholder="This novel is..." translate="no" class="tiptap ProseMirror tiptap-editor" tabindex="0"><p><br class="ProseMirror-trailingBreak"></p></div></div><div class="bubble-menu bubble-menu-static rounded-b-md "><button type="button" title="Bold" class="btn-decoration"><strong>B</strong></button><button type="button" title="Italic" class="btn-decoration"><em>I</em></button><button type="button" title="Strikethrough" class="btn-decoration"><s>S</s></button><button type="button" title="Spoiler" class="btn-decoration">Spoiler</button><div class="separator"></div><button type="button" title="Mention User (Requires level 3+ or membership)" disabled="" class="btn-mention btn-disabled">@User</button><button type="button" title="Mention Novel" class="btn-mention">#Novel</button><div class="character-count ">0 / 2048</div></div></div></div><div class="flex mt-2"><button type="button" tabindex="0" data-slot="button" class="group/button inline-flex cursor-pointer shrink-0 items-center justify-center rounded-lg border-transparent bg-clip-padding text-sm font-medium whitespace-nowrap transition-all outline-none select-none focus-visible:border-ring focus-visible:ring-3 focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-3 aria-invalid:ring-destructive/20 dark:aria-invalid:border-destructive/50 dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 [&amp;_svg:not([class*='size-'])]:size-4 border bg-secondary text-secondary-foreground hover:bg-secondary/80 aria-expanded:bg-secondary aria-expanded:text-secondary-foreground h-8 gap-1.5 px-2.5 has-data-[icon=inline-end]:pr-2 has-data-[icon=inline-start]:pl-2 ml-auto">Post Review</button></div></div></div></div></div></div></div></div><div data-slot="card" data-size="default" data-variant="default" class="group/card flex flex-col gap-0 bg-card text-sm text-card-foreground has-[&gt;img:first-child]:pt-0 *:[img:first-child]:rounded-t-xl *:[img:last-child]:rounded-b-xl shadow rounded-md ring-1 ring-foreground/20 mt-2 fix-size fix-edge"><div data-slot="card-content" class="p-2"><div class="flex items-center justify-between gap-2 mb-2 pl-2.5 border-l-2 border-l-primary mt-0"><span class="flex-1 text-base font-semibold text-card-foreground">Similar Novels</span></div><div class="relative flex h-full items-stretch overflow-hidden"><div class="flex h-full gap-2 overflow-x-auto pb-1.5 wtr-scroll"><div class="flex shrink-0 flex-col rounded-md bg-card text-sm text-card-foreground shadow border border-wtr-border-muted w-44 min-w-44"><a target="_blank" class="block" href="/en/novel/9347/becoming-an-immortal-against-the-will-of-heaven"><div class="image-wrap relative rounded-md aspect-3/4 w-full rounded-b-none zoom"><img alt="Becoming an immortal against the will of heaven" loading="lazy" src="/api/v2/img?src=s3%3A%2F%2Fwtrimg%2Fseries%2Fg_1TWwXrrPlpPyBvkgc2uVcTJ3BCp1Z2CgeRedGPEec.png&amp;w=344"><span class="status corner st-0"></span><span class="status top-right"><span>1.0</span> <span class="star">★</span></span></div></a><a target="_blank" class="line-clamp-2 px-2 pt-1 mb-1 font-bold text-sm leading-snug" title="Becoming an immortal against the will of heaven" href="/en/novel/9347/becoming-an-immortal-against-the-will-of-heaven">Becoming an immortal against the will of heaven</a><div class="mt-auto flex flex-col gap-1 px-2 pb-2 text-foreground [&amp;&gt;*:first-child]:text-xs"><div class="flex items-center gap-1"><span class="inline-flex items-center font-medium border rounded px-1.5 py-0.5 whitespace-nowrap transition-colors border-border/70 text-card-foreground text-xs capitalize">action</span><span class="ms-auto text-xs">1737 Ch</span></div></div></div><div class="flex shrink-0 flex-col rounded-md bg-card text-sm text-card-foreground shadow border border-wtr-border-muted w-44 min-w-44"><a target="_blank" class="block" href="/en/novel/10457/cannon-fodder-cultivates-immortality"><div class="image-wrap relative rounded-md aspect-3/4 w-full rounded-b-none zoom"><img alt="Cannon fodder cultivates immortality " loading="lazy" src="/api/v2/img?src=s3%3A%2F%2Fwtrimg%2Fseries%2F7DsaYmOQmjdT7BhBeBOLQ7ymJ8zPiKqLslNN1BTeOaQ.png&amp;w=344"><span class="status corner st-0"></span><span class="status top-right"><span>2.3</span> <span class="star">★</span></span></div></a><a target="_blank" class="line-clamp-2 px-2 pt-1 mb-1 font-bold text-sm leading-snug" title="Cannon fodder cultivates immortality " href="/en/novel/10457/cannon-fodder-cultivates-immortality">Cannon fodder cultivates immortality </a><div class="mt-auto flex flex-col gap-1 px-2 pb-2 text-foreground [&amp;&gt;*:first-child]:text-xs"><div class="flex items-center gap-1"><span class="inline-flex items-center font-medium border rounded px-1.5 py-0.5 whitespace-nowrap transition-colors border-border/70 text-card-foreground text-xs capitalize">action</span><span class="ms-auto text-xs">498 Ch</span></div></div></div><div class="flex shrink-0 flex-col rounded-md bg-card text-sm text-card-foreground shadow border border-wtr-border-muted w-44 min-w-44"><a target="_blank" class="block" href="/en/novel/17985/stepping-on-ominous-things-i-can-prove-my-immortality-in-troubled-times"><div class="image-wrap relative rounded-md aspect-3/4 w-full rounded-b-none zoom"><img alt=" Stepping on Ominous Things, I Can Prove My Immortality in Troubled Times" loading="lazy" src="/api/v2/img?src=s3%3A%2F%2Fwtrimg%2Fseries%2FJGTVhoMOh8HPsjqdfJjME9Zash3_08Pldv7KDd9E3h8.jpg&amp;w=344"><span class="status corner st-0"></span><span class="status top-right"><span>2.4</span> <span class="star">★</span></span></div></a><a target="_blank" class="line-clamp-2 px-2 pt-1 mb-1 font-bold text-sm leading-snug" title=" Stepping on Ominous Things, I Can Prove My Immortality in Troubled Times" href="/en/novel/17985/stepping-on-ominous-things-i-can-prove-my-immortality-in-troubled-times"> Stepping on Ominous Things, I Can Prove My Immortality in Troubled Times</a><div class="mt-auto flex flex-col gap-1 px-2 pb-2 text-foreground [&amp;&gt;*:first-child]:text-xs"><div class="flex items-center gap-1"><span class="inline-flex items-center font-medium border rounded px-1.5 py-0.5 whitespace-nowrap transition-colors border-border/70 text-card-foreground text-xs capitalize">action</span><span class="ms-auto text-xs">1610 Ch</span></div></div></div><div class="flex shrink-0 flex-col rounded-md bg-card text-sm text-card-foreground shadow border border-wtr-border-muted w-44 min-w-44"><a target="_blank" class="block" href="/en/novel/8461/an-ordinary-path-to-immortality"><div class="image-wrap relative rounded-md aspect-3/4 w-full rounded-b-none zoom"><img alt="An Ordinary Path to Immortality" loading="lazy" src="/api/v2/img?src=s3%3A%2F%2Fwtrimg%2Fseries%2FBnn-BbA1YKxWIR6laihixA.jpeg&amp;w=344"><span class="status corner st-0"></span><span class="status top-right"><span>3.6</span> <span class="star">★</span></span></div></a><a target="_blank" class="line-clamp-2 px-2 pt-1 mb-1 font-bold text-sm leading-snug" title="An Ordinary Path to Immortality" href="/en/novel/8461/an-ordinary-path-to-immortality">An Ordinary Path to Immortality</a><div class="mt-auto flex flex-col gap-1 px-2 pb-2 text-foreground [&amp;&gt;*:first-child]:text-xs"><div class="flex items-center gap-1"><span class="inline-flex items-center font-medium border rounded px-1.5 py-0.5 whitespace-nowrap transition-colors border-border/70 text-card-foreground text-xs capitalize">action</span><span class="ms-auto text-xs">1857 Ch</span></div></div></div><div class="flex shrink-0 flex-col rounded-md bg-card text-sm text-card-foreground shadow border border-wtr-border-muted w-44 min-w-44"><a target="_blank" class="block" href="/en/novel/9218/lingxu-sword-coffin-blind-swordsman"><div class="image-wrap relative rounded-md aspect-3/4 w-full rounded-b-none zoom"><img alt="Lingxu, Sword Coffin, Blind Swordsman" loading="lazy" src="/api/v2/img?src=s3%3A%2F%2Fwtrimg%2Fseries%2Fe5GkRn_LAJbJykE-kjtVg9LaZq5E9FK4Mu9XKD4wbOk.png&amp;w=344"><span class="status corner st-0"></span><span class="status top-right"><span>3.4</span> <span class="star">★</span></span></div></a><a target="_blank" class="line-clamp-2 px-2 pt-1 mb-1 font-bold text-sm leading-snug" title="Lingxu, Sword Coffin, Blind Swordsman" href="/en/novel/9218/lingxu-sword-coffin-blind-swordsman">Lingxu, Sword Coffin, Blind Swordsman</a><div class="mt-auto flex flex-col gap-1 px-2 pb-2 text-foreground [&amp;&gt;*:first-child]:text-xs"><div class="flex items-center gap-1"><span class="inline-flex items-center font-medium border rounded px-1.5 py-0.5 whitespace-nowrap transition-colors border-border/70 text-card-foreground text-xs capitalize">action</span><span class="ms-auto text-xs">3716 Ch</span></div></div></div><div class="flex shrink-0 flex-col rounded-md bg-card text-sm text-card-foreground shadow border border-wtr-border-muted w-44 min-w-44"><a target="_blank" class="block" href="/en/novel/7378/a-mortal-asks-the-heavens-i-want-to-become-an-immortal"><div class="image-wrap relative rounded-md aspect-3/4 w-full rounded-b-none zoom"><img alt="A mortal asks the heavens: I want to become an immortal" loading="lazy" src="/api/v2/img?src=s3%3A%2F%2Fwtrimg%2Fseries%2FuJBbbJgIDM27E1sBfoZB2w.jpg&amp;w=344"><span class="status corner st-1"></span><span class="status top-right"><span>2.8</span> <span class="star">★</span></span></div></a><a target="_blank" class="line-clamp-2 px-2 pt-1 mb-1 font-bold text-sm leading-snug" title="A mortal asks the heavens: I want to become an immortal" href="/en/novel/7378/a-mortal-asks-the-heavens-i-want-to-become-an-immortal">A mortal asks the heavens: I want to become an immortal</a><div class="mt-auto flex flex-col gap-1 px-2 pb-2 text-foreground [&amp;&gt;*:first-child]:text-xs"><div class="flex items-center gap-1"><span class="inline-flex items-center font-medium border rounded px-1.5 py-0.5 whitespace-nowrap transition-colors border-border/70 text-card-foreground text-xs capitalize">action</span><span class="ms-auto text-xs">1710 Ch</span></div></div></div><div class="flex shrink-0 flex-col rounded-md bg-card text-sm text-card-foreground shadow border border-wtr-border-muted w-44 min-w-44"><a target="_blank" class="block" href="/en/novel/8471/mortals-aspirations-to-cultivate-immortality"><div class="image-wrap relative rounded-md aspect-3/4 w-full rounded-b-none zoom"><img alt="Mortals' aspirations to cultivate immortality" loading="lazy" src="/api/v2/img?src=s3%3A%2F%2Fwtrimg%2Fseries%2FLw_dU_hBQvwRm2tSTzd9Mw.jpg&amp;w=344"><span class="status corner st-3"></span><span class="status top-right"><span>4.0</span> <span class="star">★</span></span></div></a><a target="_blank" class="line-clamp-2 px-2 pt-1 mb-1 font-bold text-sm leading-snug" title="Mortals' aspirations to cultivate immortality" href="/en/novel/8471/mortals-aspirations-to-cultivate-immortality">Mortals' aspirations to cultivate immortality</a><div class="mt-auto flex flex-col gap-1 px-2 pb-2 text-foreground [&amp;&gt;*:first-child]:text-xs"><div class="flex items-center gap-1"><span class="inline-flex items-center font-medium border rounded px-1.5 py-0.5 whitespace-nowrap transition-colors border-border/70 text-card-foreground text-xs capitalize">action</span><span class="ms-auto text-xs">611 Ch</span></div></div></div><div class="flex shrink-0 flex-col rounded-md bg-card text-sm text-card-foreground shadow border border-wtr-border-muted w-44 min-w-44"><a target="_blank" class="block" href="/en/novel/13175/the-cultivation-tale-of-yimu"><div class="image-wrap relative rounded-md aspect-3/4 w-full rounded-b-none zoom"><img alt="The Cultivation Tale of Yimu" loading="lazy" src="/api/v2/img?src=s3%3A%2F%2Fwtrimg%2Fseries%2F5KMCxI6a3qjOJcLyJxGtgkqjJJ0zgStiN3yanwVTfxA.png&amp;w=344"><span class="status corner st-0"></span><span class="status top-right"><span>2.3</span> <span class="star">★</span></span></div></a><a target="_blank" class="line-clamp-2 px-2 pt-1 mb-1 font-bold text-sm leading-snug" title="The Cultivation Tale of Yimu" href="/en/novel/13175/the-cultivation-tale-of-yimu">The Cultivation Tale of Yimu</a><div class="mt-auto flex flex-col gap-1 px-2 pb-2 text-foreground [&amp;&gt;*:first-child]:text-xs"><div class="flex items-center gap-1"><span class="inline-flex items-center font-medium border rounded px-1.5 py-0.5 whitespace-nowrap transition-colors border-border/70 text-card-foreground text-xs capitalize">action</span><span class="ms-auto text-xs">1099 Ch</span></div></div></div><div class="flex shrink-0 flex-col rounded-md bg-card text-sm text-card-foreground shadow border border-wtr-border-muted w-44 min-w-44"><a target="_blank" class="block" href="/en/novel/10825/cultivation-starting-from-slavery"><div class="image-wrap relative rounded-md aspect-3/4 w-full rounded-b-none zoom"><img alt="Cultivation: Starting from Slavery" loading="lazy" src="/api/v2/img?src=s3%3A%2F%2Fwtrimg%2Fseries%2FvmhBl2YpBItim_dTQNSbGMHrlYt65mmIi_G1Ih4qDo0.png&amp;w=344"><span class="status corner st-0"></span><span class="status top-right"><span>2.9</span> <span class="star">★</span></span></div></a><a target="_blank" class="line-clamp-2 px-2 pt-1 mb-1 font-bold text-sm leading-snug" title="Cultivation: Starting from Slavery" href="/en/novel/10825/cultivation-starting-from-slavery">Cultivation: Starting from Slavery</a><div class="mt-auto flex flex-col gap-1 px-2 pb-2 text-foreground [&amp;&gt;*:first-child]:text-xs"><div class="flex items-center gap-1"><span class="inline-flex items-center font-medium border rounded px-1.5 py-0.5 whitespace-nowrap transition-colors border-border/70 text-card-foreground text-xs capitalize">action</span><span class="ms-auto text-xs">1854 Ch</span></div></div></div><div class="flex shrink-0 flex-col rounded-md bg-card text-sm text-card-foreground shadow border border-wtr-border-muted w-44 min-w-44"><a target="_blank" class="block" href="/en/novel/29154/ninefold-immortal-map"><div class="image-wrap relative rounded-md aspect-3/4 w-full rounded-b-none zoom"><img alt="Ninefold Immortal Map" loading="lazy" src="/api/v2/img?src=s3%3A%2F%2Fwtrimg%2Fseries%2F2pLXPiSGZboc3nVhf_q41NH1lRfzfgWpFdSqFM8Rch0.jpeg&amp;w=344"><span class="status corner st-1"></span><span class="status top-right"><span>1.0</span> <span class="star">★</span></span></div></a><a target="_blank" class="line-clamp-2 px-2 pt-1 mb-1 font-bold text-sm leading-snug" title="Ninefold Immortal Map" href="/en/novel/29154/ninefold-immortal-map">Ninefold Immortal Map</a><div class="mt-auto flex flex-col gap-1 px-2 pb-2 text-foreground [&amp;&gt;*:first-child]:text-xs"><div class="flex items-center gap-1"><span class="inline-flex items-center font-medium border rounded px-1.5 py-0.5 whitespace-nowrap transition-colors border-border/70 text-card-foreground text-xs capitalize">adventure</span><span class="ms-auto text-xs">1431 Ch</span></div></div></div><div class="flex shrink-0 flex-col rounded-md bg-card text-sm text-card-foreground shadow border border-wtr-border-muted w-44 min-w-44"><a target="_blank" class="block" href="/en/novel/36516/king-of-body"><div class="image-wrap relative rounded-md aspect-3/4 w-full rounded-b-none zoom"><img alt="King of Body" loading="lazy" src="/api/v2/img?src=s3%3A%2F%2Fwtrimg%2Fseries%2FMmYYXuhN4RsjUOnw34TtexqHyRi7JG4NCeI8w-PzEZE.jpg&amp;w=344"><span class="status corner st-0"></span><span class="status top-right"><span>2.5</span> <span class="star">★</span></span></div></a><a target="_blank" class="line-clamp-2 px-2 pt-1 mb-1 font-bold text-sm leading-snug" title="King of Body" href="/en/novel/36516/king-of-body">King of Body</a><div class="mt-auto flex flex-col gap-1 px-2 pb-2 text-foreground [&amp;&gt;*:first-child]:text-xs"><div class="flex items-center gap-1"><span class="inline-flex items-center font-medium border rounded px-1.5 py-0.5 whitespace-nowrap transition-colors border-border/70 text-card-foreground text-xs capitalize">action</span><span class="ms-auto text-xs">1218 Ch</span></div></div></div><div class="flex shrink-0 flex-col rounded-md bg-card text-sm text-card-foreground shadow border border-wtr-border-muted w-44 min-w-44"><a target="_blank" class="block" href="/en/novel/24048/xuanhua-xianyuan"><div class="image-wrap relative rounded-md aspect-3/4 w-full rounded-b-none zoom"><img alt="Xuanhua Xianyuan" loading="lazy" src="/api/v2/img?src=s3%3A%2F%2Fwtrimg%2Fseries%2FI5CP_wYO485xuRgP-knpqiyTKsUqG1NaQAIvK8RkmxU.jpg&amp;w=344"><span class="status corner st-0"></span><span class="status top-right"><span>1.8</span> <span class="star">★</span></span></div></a><a target="_blank" class="line-clamp-2 px-2 pt-1 mb-1 font-bold text-sm leading-snug" title="Xuanhua Xianyuan" href="/en/novel/24048/xuanhua-xianyuan">Xuanhua Xianyuan</a><div class="mt-auto flex flex-col gap-1 px-2 pb-2 text-foreground [&amp;&gt;*:first-child]:text-xs"><div class="flex items-center gap-1"><span class="inline-flex items-center font-medium border rounded px-1.5 py-0.5 whitespace-nowrap transition-colors border-border/70 text-card-foreground text-xs capitalize">action</span><span class="ms-auto text-xs">1090 Ch</span></div></div></div><div class="flex shrink-0 flex-col rounded-md text-sm text-card-foreground shadow border border-wtr-border-muted w-44 min-w-44 cursor-pointer items-center justify-center bg-primary/5 transition-colors hover:bg-primary/10" role="button" tabindex="0"><div class="mb-2 flex text-primary"><svg class="icon inline-flex shrink-0 size-5 rotate-270"><use href="#chevron_down"></use></svg><svg class="icon inline-flex shrink-0 size-5 rotate-270"><use href="#chevron_down"></use></svg><svg class="icon inline-flex shrink-0 size-5 rotate-270"><use href="#chevron_down"></use></svg></div><span class="font-semibold text-primary">Show More</span><span class="text-sm text-muted-foreground">(48 left)</span></div></div><button type="button" tabindex="0" data-slot="button" class="group/button inline-flex cursor-pointer shrink-0 items-center justify-center rounded-lg bg-clip-padding text-sm font-medium whitespace-nowrap transition-all outline-none select-none focus-visible:border-ring focus-visible:ring-3 focus-visible:ring-ring/50 disabled:pointer-events-none disabled:opacity-50 aria-invalid:border-destructive aria-invalid:ring-3 aria-invalid:ring-destructive/20 dark:aria-invalid:border-destructive/50 dark:aria-invalid:ring-destructive/40 [&amp;_svg]:pointer-events-none [&amp;_svg]:shrink-0 [&amp;_svg:not([class*='size-'])]:size-4 border border-current/50 bg-card text-dark hover:bg-card-foreground hover:text-card gap-1.5 has-data-[icon=inline-end]:pr-2 has-data-[icon=inline-start]:pl-2 absolute top-1/2 -translate-y-1/2! z-15 right-2 w-8 h-12 p-0"><svg class="icon inline-flex shrink-0 size-6 rotate-270"><use href="#chevron_down"></use></svg></button></div></div></div></div><section aria-label="Notifications alt+T" tabindex="-1" aria-live="polite" aria-relevant="additions text" aria-atomic="false"></section></div><footer class="wtr-footer mt-2 border-t border-border bg-card text-center text-xs duration-200 ease-in-out font-sans antialiased" style=""><div class="max-w-7xl mx-auto px-4 py-1"><div class="flex flex-wrap items-center justify-center gap-x-1 py-2"><a class="px-1.5 py-0.5 no-underline" href="/">Intro</a><a class="px-1.5 py-0.5 no-underline" href="/en/about-us">About Us</a><a class="px-1.5 py-0.5 no-underline" href="/en/contact-us">Contact Us</a><a class="px-1.5 py-0.5 no-underline" href="/en/news?type=changelog">Changelog</a><a class="px-1.5 py-0.5 no-underline" href="/en/dmca">DMCA</a><a class="px-1.5 py-0.5 no-underline" href="/en/cookie-policy">Cookie Policy</a><a class="px-1.5 py-0.5 no-underline" href="/en/privacy-policy">Privacy Policy</a><a class="px-1.5 py-0.5 no-underline" href="/en/terms-of-use">Terms of Use</a><a class="px-1.5 py-0.5 no-underline" href="/en/public-stats">Stats</a></div><div data-orientation="horizontal" role="separator" aria-orientation="horizontal" data-slot="separator" class="shrink-0 bg-border data-horizontal:h-px data-horizontal:w-full data-vertical:w-px data-vertical:self-stretch my-1"></div><div class="py-2"><span>Copyright © 2022 - wtr-lab.com</span><a class="ms-1" href="/en/news?type=changelog"><b>v1.13.4</b></a></div></div></footer></div><script id="__NEXT_DATA__" type="application/json">{"props":{"pageProps":{"serie":{"serie_data":{"id":7251,"slug":"mortal-bones","data":{"title":"Mortal Bones","author":"Yi Geng Da Shi","description":"There are four grades of spirit bones in the world. The first grade is the Heavenly Spirit Bone. The second grade is the Golden Spirit Bone. The third grade is the Profound Spirit Bone. The fourth grade is the White Spirit Bone. The rest are all Mortal Bones, with no chance of cultivation. Xu Taiping, a mere Mortal Bone, vows to prove to this cultivation world that Mortal Bones can also slay demons, Mortal Bones can also exorcise evil, Mortal Bones can also ascend to immortality!","raw":{"title":"凡骨","author":"壹更大师","description":"世间灵骨，共分四品。\\n一品，天灵骨。二品，金灵骨。三品，玄灵骨。四品，白灵骨。\\n余者，皆为凡骨，无缘修行。\\n一介凡骨许太平，誓要向这修行界证明，凡骨亦能斩妖，凡骨亦能除魔，凡骨亦能登仙！[壹更大师] 主角：许太平"},"image":"https://img.wtr-lab.com/cdn/series/P0DFglC6FSy273hoZ3e48A.jpg"},"status":0,"raw_id":7336,"user_status":0,"is_default":true,"chapter_count":3992,"ai_enabled":true,"raw_status":0,"raw_verified":true,"genres":[1,3,5,9,37],"tags":[93,115,125,169,181,184,193,197,257,265,357,417,539,590,680,692,715,731,750],"is_adult":false},"default_service":"ai","chapter":{"id":9971151,"serie_id":7251,"raw_id":7336,"status":1,"slug":"mortal-bones","name":"取令牌，心神不宁的独孤青霄","order":1747,"is_update":false,"created_at":"2025-03-04 17:16:59.856+00","updated_at":"2025-03-05 00:39:03.991+00","title":"Taking the token, the uneasy Dugu Qingxiao","code":"c119","char_count":1803,"locked":false,"released":true}},"disable_ads":false,"active_service":{"id":"webplus","label":"Web+"}},"__N_SSP":true},"page":"/[locale]/novel/[raw_id]/[serie_slug]/[chapter_no]","query":{"service":"webplus","locale":"en","raw_id":"7336","serie_slug":"mortal-bones","chapter_no":"chapter-1747"},"buildId":"0rJUaTB6TLmnU7nTm-twr","isFallback":false,"isExperimentalCompile":false,"gssp":true,"scriptLoader":[]}</script><script defer="" src="https://static.cloudflareinsights.com/beacon.min.js/v4513226cdae34746b4dedf0b4dfa099e1781791509496" integrity="sha512-ZE9pZaUXND66v380QUtch/5sE9tPFh2zg45pR2PB0CVkCtOREv2AJKkSidISWkysEuQ0EH8faUU5du78bx87UQ==" data-cf-beacon="{&quot;version&quot;:&quot;2024.11.0&quot;,&quot;token&quot;:&quot;22b0af416698496bb1e289b16c774dca&quot;,&quot;r&quot;:1,&quot;server_timing&quot;:{&quot;name&quot;:{&quot;cfCacheStatus&quot;:true,&quot;cfEdge&quot;:true,&quot;cfExtPri&quot;:true,&quot;cfL4&quot;:true,&quot;cfOrigin&quot;:true,&quot;cfSpeedBrain&quot;:true},&quot;location_startswith&quot;:null}}" crossorigin="anonymous"></script>
<script>(function(){function c(){var b=a.contentDocument||a.contentWindow.document;if(b){var d=b.createElement('script');d.innerHTML="window.__CF$cv$params={r:'a14d9764be279475',t:'MTc4Mjk5NDY4OA=='};var a=document.createElement('script');a.src='/cdn-cgi/challenge-platform/scripts/jsd/main.js';document.getElementsByTagName('head')[0].appendChild(a);";b.getElementsByTagName('head')[0].appendChild(d)}}if(document.body){var a=document.createElement('iframe');a.height=1;a.width=1;a.style.position='absolute';a.style.top=0;a.style.left=0;a.style.border='none';a.style.visibility='hidden';document.body.appendChild(a);if('loading'!==document.readyState)c();else if(window.addEventListener)document.addEventListener('DOMContentLoaded',c);else{var e=document.onreadystatechange||function(){};document.onreadystatechange=function(b){e(b);'loading'!==document.readyState&&(document.onreadystatechange=e,c())}}}})();</script><iframe height="1" width="1" style="position: absolute; top: 0px; left: 0px; border: none; visibility: hidden;"></iframe><script id="_next-ga-init" data-nscript="afterInteractive">
          window['dataLayer'] = window['dataLayer'] || [];
          function gtag(){window['dataLayer'].push(arguments);}
          gtag('js', new Date());

          gtag('config', 'G-N72PBEFJ1N' );</script><script src="https://www.googletagmanager.com/gtag/js?id=G-N72PBEFJ1N" id="_next-ga" data-nscript="afterInteractive"></script><next-route-announcer><p aria-live="assertive" id="__next-route-announcer__" role="alert" style="border: 0px; clip: rect(0px, 0px, 0px, 0px); height: 1px; margin: -1px; overflow: hidden; padding: 0px; position: absolute; top: 0px; width: 1px; white-space: nowrap; overflow-wrap: normal;">Read Mortal Bones RAW English Translation - WTR-LAB</p></next-route-announcer><script src="/_next/./static/chunks/abb6af767f9755d4.js"></script><script src="/_next/./static/chunks/5120554a604a243d.js"></script><script src="/_next/./static/chunks/388a37aa601a5b50.js"></script><script src="/_next/./static/chunks/dfa4383987f4d407.js"></script></body></html>"""




In [43]:
tree = LexborHTMLParser(info_html)

parsed = urlparse(test_web_novel)
base_url = f"{parsed.scheme}://{parsed.netloc}"

base_url

'https://wtr-lab.com'

In [34]:

titlebox = tree.css_first(".p-2[data-slot='card-content']").child
statusbox = tree.css_first(".p-2[data-slot='card-content']").child.next
tagbox = tree.css_first(".p-2[data-slot='card-content']").last_child
detailbox = tree.css_first(".p-2[data-slot='card-content'].chapter-details")

In [35]:
title = titlebox.css_first("h1").text()
other_title = titlebox.css_first("p").text()

(title, other_title)

('Lord: God-tier Attribute, Recruits Fallen Angels of Original Sin',
 '领主：神级词条，招募原罪堕天使')

In [36]:
cover_image_url = statusbox.css_first("img").attributes['src']
total_chapters = statusbox.css_first("span[translate='no']:lexbor-contains('chapters'i)").next.text()

(cover_image_url, total_chapters)

('/api/v2/img?src=s3%3A%2F%2Fwtrimg%2Fseries%2FujGWOOGMtucuE5A8XnGYu1cBWtQClJi3k8BzzzVOh_4.jpeg&w=344',
 '664')

In [37]:
tags = [tag.text().lower() for tag in tagbox.css("span")]

tags

['male',
 'kingdom building',
 'reincarnation',
 'system',
 'game elements',
 'class awakening',
 'action',
 'fantasy',
 'game',
 'harem',
 'urban-life']

In [38]:
summary = detailbox.css_first(".desc-wrap > span.description").text()
summary

"(The rating just came out and will rise later. It's definitely a satisfying read, not toxic or frustrating, and it has a bit of logic!)[Lord] + [Troop Type] + [Conquest] + [Multiple Women] + [Construction] + [Hoarding Rat]Su Ye transmigrated to a world where everyone is a lord, and being a lord is everything, as well as the path to godhood.Starting Awakening Talent: Supreme Ruler, First Trait: Echo of Strong Luck.Echoes of Fortune: Upgrading Lord's Heart guarantees a gold or higher-level affix.Obtained terms: Super God Evolution (Red), Crystal Palace (Gold)...Super God Evolution: Can evolve any barracks into a god-level barracks, with unit loyalty remaining constant at 100 and never decreasing.Crystal Palace: This territory can only recruit female units. Female units have a 100% increase in combat power and a potential level of +1!...........Years later, fallen angels chanted the praises of the God-Emperor, illuminating all worlds. Primordial dragons roamed the world, amidst countless

In [39]:
detailgrid = detailbox.css_first("[data-slot='tabs-content']").css_first("span:lexbor-contains('details'i)").parent.next
status = detailgrid.css_first(":lexbor-contains('status'i)").next.text()
author = detailgrid.css_first(":lexbor-contains('author'i)").next.child.text()
other_author_name = detailgrid.css_first(":lexbor-contains('author'i)").next.last_child.text()

print(f"Status: {status.lower()}\n", f"Author: {author}, {other_author_name}", sep="")

Status: ongoing
Author: 浮生老五, Fu Sheng Lao Wu


In [67]:
# Clean up the raw chapter links

links = [f"{base_url}/{x}" for x in raw_chapter_links]
links

['https://wtr-lab.com//en/novel/53992/lord-god-tier-attribute-recruits-fallen-angels-of-original-sin/chapter-1',
 'https://wtr-lab.com//en/novel/53992/lord-god-tier-attribute-recruits-fallen-angels-of-original-sin/chapter-2',
 'https://wtr-lab.com//en/novel/53992/lord-god-tier-attribute-recruits-fallen-angels-of-original-sin/chapter-3',
 'https://wtr-lab.com//en/novel/53992/lord-god-tier-attribute-recruits-fallen-angels-of-original-sin/chapter-4',
 'https://wtr-lab.com//en/novel/53992/lord-god-tier-attribute-recruits-fallen-angels-of-original-sin/chapter-5',
 'https://wtr-lab.com//en/novel/53992/lord-god-tier-attribute-recruits-fallen-angels-of-original-sin/chapter-6',
 'https://wtr-lab.com//en/novel/53992/lord-god-tier-attribute-recruits-fallen-angels-of-original-sin/chapter-7',
 'https://wtr-lab.com//en/novel/53992/lord-god-tier-attribute-recruits-fallen-angels-of-original-sin/chapter-8',
 'https://wtr-lab.com//en/novel/53992/lord-god-tier-attribute-recruits-fallen-angels-of-original